In [ ]:
from dotenv import load_dotenv
load_dotenv()

%env HF_XET_HIGH_PERFORMANCE=1

# NCM-MoE Omni: Next Concept Modelling, 6 modality experts, text / image / audio

A from-scratch notebook (nothing imported from a repo). One transformer predicts **every code of a
level in a single forward**, coarse to fine: `K` forwards generate a sample instead of one per
token. Text, image and audio share one **level ladder** (level *k* has the same compression ratio
`R^(K-1-k)` in every modality), so "level 2" means the same altitude everywhere.

| piece | what it is |
|---|---|
| trunk | d=1536, L=16, GQA 16/4, QK-norm, block-causal-over-levels attention |
| experts | 3 modality groups x 2 SwiGLU experts = **6 experts, 1.66B total, ~360M active** |
| routing | group forced by the token's modality; inside the group, **expert-choice** (each expert takes its top-capacity tokens per level), so shapes are static, nothing is dropped from the graph, and a token gets 0, 1 or 2 experts (avg 1) |
| conditioning | a text prefix: text->text (prompt), text->image (caption), text->speech (transcript = TTS) |
| data | streamed straight from the Hub (HF cache lives in RAM). Every encoded row is also **recorded as `uint8` code shards on local disk** (`SHARD_DIR`, ~1 byte per code) so a later run can read them instead of streaming |
| default run | **T2 = 20B live codes** (see the tier table in the budget cell) |

Verification knobs: every knob can be overridden by an `NCM_<NAME>` environment variable, which is
how the same notebook is executed headlessly on CPU and on a small GPU (`PRESET="tiny"`).

## 0. Knobs

In [ ]:
import os, sys, time, math, json, io, gc, threading, queue, random, warnings, shutil, subprocess
T_START = time.time()
from dataclasses import dataclass

# where are we running? Colab has a ~12 h session limit and /content; RunPod has neither (persistent volume at /workspace)
ON_COLAB = "COLAB_RELEASE_TAG" in os.environ or "COLAB_GPU" in os.environ
ON_RUNPOD = "RUNPOD_POD_ID" in os.environ or (os.path.isdir("/workspace") and not ON_COLAB)
WORK_DIR = "/workspace" if ON_RUNPOD else ("/content" if os.path.isdir("/content") else os.getcwd())

os.environ.setdefault("HF_HUB_DOWNLOAD_TIMEOUT", "60")      # flaky links (RunPod): the 10 s default times out on 100 MB parquet files
os.environ.setdefault("HF_HUB_ETAG_TIMEOUT", "30")

def _knob(name, default):
    """Env override `NCM_<name>` for headless runs; the literal in this cell is the default."""
    v = os.environ.get("NCM_" + name)
    if v is None:
        return default
    if isinstance(default, bool):
        return v.lower() in ("1", "true", "yes")
    return type(default)(v)

# ---- model + run size ----
PRESET   = _knob("PRESET", "full")     # "full" 1.66B total / 360M active | "tiny" same code at d=512 (any 8 GB GPU / CPU)
TIER     = _knob("TIER", "T1")         # T0 0.02B | T1 2B | T2 20B | T3 50B live codes (see budget cell). T0 = smoke test; set "T1" for the pilot
TOKENS   = _knob("TOKENS", 0.3)        # billions of live codes; 0 = take it from TIER
STEPS    = _knob("STEPS", 0)           # >0 = hard cap on optimizer steps (smoke / CI)
SEED     = _knob("SEED", 0)

# ---- speed ----
DEVICE   = _knob("DEVICE", "auto")     # "auto" | "cuda" | "cpu"
COMPILE  = _knob("COMPILE", True)      # torch.compile the step (per pack shape, static shapes)
COMPILE_MODE = _knob("COMPILE_MODE", "default")   # "max-autotune-no-cudagraphs" for the long run
ATTN     = _knob("ATTN", "levels")     # "levels" (default: one unmasked SDPA call per level, the mask is block-causal over levels; +17% codes/s with flat dispatch on an RTX PRO 6000) | "auto" | "flex" | "sdpa"
SWIGLU   = _knob("SWIGLU", "auto")     # "auto" | "triton" | "torch"
GPU_JPEG = _knob("GPU_JPEG", True)     # decode JPEG on the GPU (nvJPEG) instead of in the workers
MOE_CF   = _knob("MOE_CF", 0.5)        # expert capacity as a fraction of the level's tokens (2 experts x 0.5 = 1 active/token)
MOE_DISPATCH = _knob("MOE_DISPATCH", "flat")     # "flat" (default: index_select / index_add_ on flattened rows, same maths, fewer copies) | "gather" (reference)
RESID    = _knob("RESID", "fp32")      # residual-stream dtype under autocast: "fp32" (default) | "bf16" (measured on an RTX PRO 6000: no speed gain, same loss)
PROFILE  = _knob("PROFILE", False)     # run the profile + A/B cell after the correctness checks (prints which speed knobs to set)
MICRO_TOKENS = _knob("MICRO_TOKENS", 0)   # packed positions per micro-batch; 0 = preset default
BATCH_CODES  = _knob("BATCH_CODES", 0)    # live codes per optimizer step; 0 = preset default

# ---- data: streamed from the Hub; only the encoded codes may be kept (shards) ----
SHARD_DIR  = _knob("SHARD_DIR", os.path.join(WORK_DIR, "ncm_shards"))  # "" = no shards
SHARD_MODE = _knob("SHARD_MODE", "off")      # "off" = stream live (default; the learned tokenizers do not support shards) | "encode" | "record" | "read"
META_ONLY  = _knob("META_ONLY", False) or SHARD_MODE == "encode"   # encode-only runs never build, benchmark or train the model
EVAL_ONLY  = _knob("EVAL_ONLY", False)       # the eval notebook: restore CKPT_DIR/ckpt.pt and run only evaluation (no training, no benchmark)
ACT_CKPT = _knob("ACT_CKPT", -1)     # activation checkpointing: -1 = preset default, 0 = off (faster, needs ~2x activation memory; fine on 96 GB), 1 = on
STREAM_PREFETCH = _knob("STREAM_PREFETCH", 1)   # 1 = Parquet read-ahead + 128 MiB requests when the source allows it
WORKERS  = _knob("WORKERS", 0)         # processes streaming from the Hub; 0 = preset default
PREFETCH = _knob("PREFETCH", 6)        # ready micro-batches held in RAM
RAM_ONLY = _knob("RAM_ONLY", True)     # HF cache dir on /dev/shm, removed at exit
FIT_IMAGES = _knob("FIT_IMAGES", 0)    # images used to fit the frozen codebooks; 0 = preset default
FIT_AUDIO  = _knob("FIT_AUDIO", 0)
VAL_ROWS   = _knob("VAL_ROWS", 0)      # held-out rows per row type; 0 = preset default

# ---- tokenizers, audio decode, caption guidance ----
TOKENIZER  = _knob("TOKENIZER", "feat")  # "feat" = frozen DINOv2 + DAC features, tree-quantised (section 3b; no image/pixel decoder) | "vq" = learned multi-scale residual VQ-VAEs, trained at startup | "kmeans" = the old frozen k-means codecs
TOK_STEPS  = _knob("TOK_STEPS", 0)     # tokenizer training steps; 0 = by tier (T0 600 | T1 4000 | larger 8000)
TOK_IMAGES = _knob("TOK_IMAGES", 0)    # images held in RAM to train the image tokenizer; 0 = by tier (T0 1024 | T1+ 8192)
TOK_AUDIO  = _knob("TOK_AUDIO", 0)     # clips held in RAM to train the audio tokenizer; 0 = by tier (T0 512 | T1+ 2048)
TOK_GATE_DB = _knob("TOK_GATE_DB", 0.5)  # stop before training the model if learned image PSNR is not this many dB above k-means (-99 = never stop)
AUDIO_INV  = _knob("AUDIO_INV", "nnls")  # mel -> spectrum before Griffin-Lim: "nnls" (positive, multiplicative updates) | "pinv" (old, goes negative)
AUDIO_NNLS_ITERS = _knob("AUDIO_NNLS_ITERS", 60)
LEVEL_W    = _knob("LEVEL_W", "slots")  # loss weight per level: "slots" = every position counts the same (a level's weight = its slot count: the old behaviour) | "sqrt" = weight sqrt(slots) | "equal" = every level counts the same
CAP_DROP   = _knob("CAP_DROP", 0.1)    # fraction of image/audio training rows whose caption/transcript is blanked (makes classifier-free guidance possible)
CFG_SCALE  = _knob("CFG_SCALE", 3.0)   # default guidance scale in the generate cells (1 = off; only meaningful if CAP_DROP > 0)

# ---- optimisation ----
LR       = _knob("LR", 0.0)            # 0 = preset default
WARMUP   = _knob("WARMUP", 0)
WD       = _knob("WD", 0.1)
CLIP     = _knob("CLIP", 1.0)

# ---- long-run handling ----
CKPT_DIR = _knob("CKPT_DIR", "")       # e.g. /content/drive/MyDrive/ncm_moe ; "" = no checkpoints
CKPT_MIN = _knob("CKPT_MIN", 30)       # minutes between checkpoints
SESSION_HOURS = _knob("SESSION_HOURS", 11.5 if ON_COLAB else 0.0)   # watchdog: save and stop cleanly before the session limit (0 = off: RunPod has none)
# A Colab disk dies with the session, and `CKPT_DIR=""` means nothing is ever written. The T1 run
# kept ONE rolling checkpoint on ephemeral disk and lost its best audio model (held-out 1.52 at
# 0.73B codes) to a late overfit overwrite. So on Colab, mount Drive and put checkpoints on it.
if ON_COLAB and not CKPT_DIR:
    try:
        from google.colab import drive
        drive.mount("/content/drive", force_remount=False)
        CKPT_DIR = "/content/drive/MyDrive/ncm_moe"
        os.makedirs(CKPT_DIR, exist_ok=True)
        print(f"checkpoints: Google Drive mounted -> {CKPT_DIR} (+ <name>_best.pt held-out)")
    except Exception as _e:                                    # noqa: BLE001
        print(f"!! Drive mount failed ({type(_e).__name__}: {_e}). Set NCM_CKPT_DIR=<persistent path>"
              " or this run writes no checkpoints at all.")
EVAL_EVERY_B  = _knob("EVAL_EVERY_B", 0.25)     # evaluate every N billion live codes
LOG_EVERY = _knob("LOG_EVERY", 10)
INSTALL  = _knob("INSTALL", True)

# ---- sources: set a name to 0 to disable it (weights inside a row type are renormalised) ----
SOURCE_WEIGHTS = {}                    # e.g. {"fineweb": 0.5, "libri": 0}; empty = defaults in the data cell

print(f"preset {PRESET} | tier {TIER} | compile {COMPILE} | attn {ATTN} | swiglu {SWIGLU} | cf {MOE_CF}")
print(f"environment: {'Colab' if ON_COLAB else 'RunPod' if ON_RUNPOD else 'other'} | work dir {WORK_DIR} | checkpoints: {CKPT_DIR or 'OFF'}"
      + (" | !! set CKPT_DIR (e.g. /workspace/ncm_moe_ckpt) for any run you want to keep or resume" if ON_RUNPOD and not CKPT_DIR and TIER != "T0" else ""))

## 0.1 Runtime: RAM-only HF cache, deps, determinism

In [ ]:
# Streaming does not write dataset shards, but the hub client keeps small metadata and lock files.
# Point them at RAM so nothing survives the process; this is asserted in the no-storage check.
_RAM_HF = None
if RAM_ONLY:
    _base = "/dev/shm" if os.path.isdir("/dev/shm") else "/tmp"
    _RAM_HF = os.path.join(_base, f"ncm_hf_{os.getpid()}")
    os.makedirs(_RAM_HF, exist_ok=True)
    for _v in ("HF_HOME", "HF_DATASETS_CACHE", "HF_HUB_CACHE", "XDG_CACHE_HOME"):
        os.environ[_v] = _RAM_HF
    import atexit
    atexit.register(lambda: shutil.rmtree(_RAM_HF, ignore_errors=True))
os.environ.setdefault("HF_HUB_DISABLE_PROGRESS_BARS", "1")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True")
warnings.filterwarnings("ignore")

if INSTALL:
    _need = []
    for _m, _p in (("datasets", "datasets"), ("soundfile", "soundfile"), ("PIL", "pillow"), ("tqdm", "tqdm")):
        try:
            __import__(_m)
        except ImportError:
            _need.append(_p)
    if _need:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", *_need], check=True)

def _secret(name):
    v = os.environ.get(name, "")
    if v:
        return v
    try:
        from google.colab import userdata
        return userdata.get(name) or ""
    except Exception:
        return ""

if _secret("HF_TOKEN"):
    os.environ["HF_TOKEN"] = _secret("HF_TOKEN")      # raises rate limits; unlocks gated sources

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("torch", torch.__version__, "| HF_TOKEN", bool(os.environ.get("HF_TOKEN")), "| HF cache ->", _RAM_HF)

## 1. Hardware and kernel probe

Every fast path is probed here and printed, so a silent fallback to a slow kernel shows up as a
line of text instead of a lower tokens/s three hours later.

In [ ]:
if DEVICE == "auto":
    DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
IS_CUDA = DEVICE == "cuda"
dev = torch.device(DEVICE)

# Approximate dense bf16 peaks (TFLOPS) used only to print MFU; unknown cards print TFLOPS only.
_PEAK = {"H100 PCIe": 756, "H100": 989, "H200": 989, "A100": 312, "L4": 121, "4090": 165, "3090": 71, "3070": 40, "T4": 65, "V100": 125,
         "RTX PRO 6000": 500,                                              # Blackwell workstation, dense bf16, approximate (from memory: check the datasheet)
         "MI300X": 1307, "MI325X": 1307, "MI250": 362}                     # AMD numbers are the dense bf16 spec peaks, from memory
IS_ROCM = bool(getattr(torch.version, "hip", None))                         # AMD build of PyTorch: devices still appear as "cuda"
GPU_NAME, VRAM_GB, PEAK_TFLOPS = "cpu", 0.0, 0.0
if IS_CUDA:
    _p = torch.cuda.get_device_properties(0)
    GPU_NAME, VRAM_GB = _p.name, _p.total_memory / 2 ** 30
    PEAK_TFLOPS = next((v for k, v in _PEAK.items() if k in GPU_NAME), 0.0)
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    torch.set_float32_matmul_precision("high")
BF16 = IS_CUDA and torch.cuda.is_bf16_supported()
AMP_DTYPE = torch.bfloat16 if BF16 else (torch.float16 if IS_CUDA else torch.bfloat16)

def _probe_triton():
    try:
        import triton  # noqa: F401
        return IS_CUDA
    except Exception:
        return False

def _probe_flex():
    if not IS_CUDA or IS_ROCM:                                              # flex-attention is not trusted on ROCm: use ATTN="sdpa" or "levels"
        return False
    try:
        from torch.nn.attention.flex_attention import flex_attention, create_block_mask  # noqa: F401
        return True
    except Exception:
        return False

def _probe_nvjpeg():
    if not IS_CUDA or IS_ROCM:                                              # nvJPEG is NVIDIA-only: JPEGs are then decoded on the CPU workers
        return False
    try:
        import torchvision
        from torchvision.io import decode_jpeg  # noqa: F401
        return True
    except Exception:
        return False

def _usable_cpus():
    """CPUs this process may really use: os.cpu_count() reports the HOST (256 on a RunPod H100 pod limited to 24 vCPU)."""
    n = len(os.sched_getaffinity(0)) if hasattr(os, "sched_getaffinity") else (os.cpu_count() or 8)
    try:                                                                    # cgroup v2 quota: "max 100000" or "2400000 100000"
        q, p = open("/sys/fs/cgroup/cpu.max").read().split()[:2]
        if q != "max":
            n = min(n, max(1, int(q) // int(p)))
    except Exception:
        pass
    try:                                                                    # cgroup v1 quota
        q, p = int(open("/sys/fs/cgroup/cpu/cpu.cfs_quota_us").read()), int(open("/sys/fs/cgroup/cpu/cpu.cfs_period_us").read())
        if q > 0:
            n = min(n, max(1, q // p))
    except Exception:
        pass
    return n

N_CPUS = _usable_cpus()
HAS_TRITON, HAS_FLEX, HAS_NVJPEG = _probe_triton(), _probe_flex(), _probe_nvjpeg()
print(f"device {DEVICE} | {GPU_NAME} | {VRAM_GB:.0f} GiB | bf16 {BF16} | peak ~{PEAK_TFLOPS} TFLOPS | {'ROCm/HIP ' + str(torch.version.hip) if IS_ROCM else 'CUDA'}")
print(f"triton {HAS_TRITON} | flex_attention {HAS_FLEX} | nvJPEG {HAS_NVJPEG} | grouped_mm {hasattr(torch, '_grouped_mm')} "
      f"| cpus {N_CPUS} usable (host reports {os.cpu_count()})")
try:                                                                        # the streaming workers hand batches over through shared memory
    _shm_gb = shutil.disk_usage("/dev/shm").total / 2 ** 30
    print(f"/dev/shm {_shm_gb:.1f} GiB" + ("  !! small: raise the pod's shared memory (RunPod: container settings) or the workers will crash with 'bus error'" if _shm_gb < 4 else ""))
except Exception:
    pass

## 2. Presets, the level ladder and the pack layout

**The ladder.** `K` levels, ratio `R`. Level *k* of a modality with finest length `n` has
`n / R^(K-1-k)` slots, so level 0 is the whole sample in a handful of codes and level `K-1` is the
finest detail. Images use a 2D grid (`R=4` means 2x2 pooling per level), text and audio are 1D.

**The pack.** A row is `[ text prefix | level 0 | level 1 | ... | level K-1 ]`. The prefix is the
condition (prompt / caption / transcript, bytes, always fully known). Attention is *block-causal
over levels*: a slot sees the prefix and every slot of levels <= its own, bidirectionally inside a
level. The input at level *k* is the codes of level *k-1* upsampled to length *l_k*, so a level
never sees its own targets (the leak that made an earlier version of this idea score below the
constant baseline; the correctness cell asserts it is gone).

**Codes** are 1..256 and 0 is NULL, so vocab = 257 and a real code can never be confused with padding.

In [ ]:
from typing import Union


@dataclass
class Spec:
    K: int
    text: int            # finest text length (bytes)
    image: int           # finest image length (patches; a perfect square)
    audio: int           # finest audio length (mel frames)
    cond: int            # prefix length (bytes of prompt / caption / transcript)
    patch: int           # pixels per image patch side
    R: Union[int, dict] = 4   # one ratio for every modality, or {m: ratio}. A shared ratio is
                              # what gave text ONE level-0 code for 256 bytes (K=5, R=4).
    V: int = 257
    hop: int = 256
    sr: int = 16000
    n_mels: int = 80
    image_extra: int = 0 # extra residual levels at the finest image resolution (a 2nd codebook per position: +8 bits per slot of the finest grid)
    audio_extra: int = 0 # the same for audio: residual levels at the finest frame rate (8 codebooks in total = 7 extra)

    def __post_init__(self):
        # Resolved once; `self.R` keeps whatever was given so a scalar spec serialises as before.
        mods = ("text", "image", "audio")
        if isinstance(self.R, dict):
            missing = [m for m in mods if m not in self.R]
            if missing:
                raise KeyError(f"R is a per-modality dict but has no ratio for {missing}")
            self.Rm = {m: int(self.R[m]) for m in mods}
        else:
            self.Rm = {m: int(self.R) for m in mods}

    def R_of(self, m):
        """This modality's compression ratio. Every pooling site uses THIS, never `spec.R`."""
        return self.Rm[m]

    def lens(self, m):
        base = getattr(self, m)
        out = [base // self.R_of(m) ** (self.K - 1 - k) for k in range(self.K)]
        return out + [base] * {"image": self.image_extra, "audio": self.audio_extra}.get(m, 0)

    def ladder_report(self, min_level0=8):
        """-> (rows, warnings, fatal). A level-0 code that stands for a whole sample carries
        almost nothing and nothing downstream can learn it, so this is printed (and can block a
        run) BEFORE the data is encoded rather than diagnosed from a finished run.

        Measured, K=5/R=4 text (1/4/16/64/256 slots): level 0 was ONE code for 256 bytes and the
        probe gain at the next level was -0.31 bits, i.e. worse than a shuffled control. Images
        tolerate one slot (31.2% L0 accuracy measured - they are spatially redundant); a
        sequential modality does not.
        """
        rows, warn, fatal = [], [], []
        for m in ("text", "image", "audio"):
            base, R = getattr(self, m), self.R_of(m)
            lens = self.lens(m)[:self.K]
            span = base // max(1, lens[0])
            rows.append(f"{m:6s} R={R} base={base:5d} levels={lens} (L0: 1 code per {span})")
            if lens[0] < min_level0:
                how = "NCM_TEXT_R=2" if m == "text" else f"spec R[{m!r}]"
                msg = (f"{m}: level 0 has {lens[0]} slot(s) for {base} units (1 code per {span}) "
                       f"- below {min_level0}. Lower the ratio for {m} ({how}) or raise its base "
                       f"length to {min_level0 * R ** (self.K - 1)}+.")
                warn.append(msg)
                if m == "text":
                    fatal.append(msg)
        return rows, warn, fatal

    @property
    def side(self):                       # image side in pixels
        return math.isqrt(self.image) * self.patch

    @property
    def wave_len(self):                   # samples covered by `audio` frames
        return self.audio * self.hop

    def check(self):
        for m in ("text", "image", "audio"):
            assert getattr(self, m) % self.R_of(m) ** (self.K - 1) == 0, \
                f"{m} base {getattr(self, m)} not divisible by R^(K-1) = {self.R_of(m) ** (self.K - 1)}"
        assert math.isqrt(self.image) ** 2 == self.image
        assert math.isqrt(self.R_of("image")) ** 2 == self.R_of("image"), \
            "the image ratio must be a perfect square for 2D pooling"
        assert all(math.isqrt(n) ** 2 == n for n in self.lens("image"))
        return self


# Image bit budget (finest level, 8 bits per code): "v1" = 16x16 grid at 256 px = 2,048 bits/image (the original setting);
# "v2" = 32x32 grid at 256 px with two residual codebooks = 2 x 1,024 x 8 = 16,384 bits/image. v2 needs the learned image VQ
# (the DINO feature codec has a fixed 16x16 grid) and makes an image row ~7x longer (2,388 live codes against 341).
IMAGE_RES = _knob("IMAGE_RES", "v1")
_IMG = {"v1": dict(image=256, patch=16, image_extra=0), "v2": dict(image=1024, patch=8, image_extra=1)}[IMAGE_RES]
# Audio bit budget: "v1" = 512 frames x 8 bits = 4,096 bits/clip (~500 bit/s, about one DAC codebook); "v2" = 8 residual levels at the finest
# frame rate = 32,768 bits/clip (~4 kbps = 8 DAC codebooks, the setting that passed the listening test). v2 needs TOKENIZER="feat" (a residual
# quantiser on the DAC latents) and makes an audio row ~6x longer (about 4,266 live codes against 680).
AUDIO_RES = _knob("AUDIO_RES", "v1")
_AUDX = {"v1": 0, "v2": 7}[AUDIO_RES]

# Text's ladder ratio. 4 puts the whole 256-byte block in ONE level-0 code (nothing learnable
# there); 2 gives 16/32/64/128/256 and a level 0 that carries real structure. Costs a ~30%
# longer text row (624 packed slots against 469), which the token budget absorbs.
TEXT_R = _knob("TEXT_R", 2)
# Text context length in bytes (the FINEST level; the sample is that many bytes). Raising it
# raises the whole ladder proportionally at fixed R, so L0 keeps its 16-byte reach - but it
# also grows the finest level, and that level is predicted in ONE parallel forward with no
# byte-to-byte dependence, so more bytes per sample means more independently-wrong positions.
# 1024 is the sensible ceiling until the finest level gets a causal tail; 2048 mostly buys
# attention cost (T 4096) and cost the mix whole sources before TEXT_PACK.
TEXT_LEN = _knob("TEXT_LEN", 256)
# Audio window in mel frames (must be divisible by R**(K-1) = 256 at R=4, K=5; 1024 frames = 16.4 s
# at hop 256 / 16 kHz). This is not cosmetic: the largest open transcript corpus (People's Speech)
# segments at ~14.8 s (measured: p10 14.3, p90 15.0), and a window shorter than the segment drops
# the ENTIRE clip - at 512 frames only 1% of that corpus survived, which is why audio overfit while
# carrying weight 0.85. 1024 frames covers it; 2048 buys nothing but attention.
AUDIO_LEN = _knob("AUDIO_LEN", 1024)

PRESETS = {
    # same code path as "full", ~150M total / ~35M active: any 8 GB GPU, and CPU for CI
    "tiny": dict(d=512, layers=8, heads=8, kv_heads=2, ffn=1904, act_ckpt=False,
                 spec=dict(K=4, text=128, image=64, audio=512, cond=128, patch=8),
                 micro_tokens=8192, batch_codes=32768, workers=4, lr=1.5e-3, warmup=50,
                 fit_images=256, fit_audio=128, val_rows=16),
    # d=1536, L=16, expert h=3584 -> 264M per expert x 6 = 1.58B, trunk ~100M: ~1.68B total, ~360M active
    "full": dict(d=1536, layers=16, heads=16, kv_heads=4, ffn=3584, act_ckpt=True,
                 spec=dict(K=5, text=TEXT_LEN, audio=AUDIO_LEN, cond=128, audio_extra=_AUDX, **_IMG,
                           R=dict(text=TEXT_R, image=4, audio=4)),
                 micro_tokens=32768, batch_codes=1_000_000, workers=min(12, N_CPUS), lr=6e-4, warmup=1000,
                 fit_images=4096, fit_audio=1024, val_rows=64),
}
P = dict(PRESETS[PRESET])
SPEC = Spec(**P["spec"]).check()
# The ladder gate. Printed always; a degenerate TEXT level 0 blocks the run, because it is the
# one defect that no amount of training, capacity or data can reach past.
LADDER_GATE = _knob("LADDER_GATE", PRESET == "full")
_lrows, _lwarn, _lfatal = SPEC.ladder_report()
print("ladder:")
for _r in _lrows:
    print("  " + _r)
if _lwarn:
    print("  !! coarse level 0 (1 code per sample-scale span):")
    for _w in _lwarn:
        print("     " + _w)
if _lfatal and LADDER_GATE:
    raise SystemExit("DEGENERATE TEXT LADDER - refusing to encode a corpus nothing can learn from.\n"
                     "  " + _lfatal[0] + "\n"
                     "  Fix: NCM_TEXT_R=2 (default for preset 'full'), or NCM_LADDER_GATE=0 to override.")
print(f"  audio: {SPEC.audio} frames = {SPEC.wave_len / SPEC.sr:.1f} s per clip, window = spec.wave_len "
      f"(a clip longer than this is DROPPED, not cropped)")
_tl = SPEC.lens("text")
print(f"  text: {SPEC.text} bytes/sample; the finest level is {_tl[-1]} bytes in ONE parallel "
      f"forward (its parent supplies 1 code per {_tl[-1] // max(1, _tl[-2])} bytes, neighbours nothing)")
if SPEC.text >= 1024 and not _knob("TEXT_PACK", 1):
    print("  !! TEXT_PACK=0 with a long text sample: every document shorter than "
          f"{SPEC.text // 2} bytes is dropped and a document yields at most 4 samples")
if SPEC.text >= 1024:
    print(f"  note: the longest text row is {SPEC.text + SPEC.cond} packed slots; if the first "
          f"step OOMs, halve NCM_MICRO_TOKENS (32768 -> 16384) and expect attention to take a "
          f"larger share of the step than the 9% measured at text=256")
N_GROUP_EXPERTS = 2
_BIG_GPU = PRESET == "full" and VRAM_GB >= 90        # RTX PRO 6000 (95 GB), MI300X (192 GB): 69 GiB peak measured with micro 49152 and no activation checkpointing
# A long text or audio sample is the same memory problem as a v2 residual image row: more packed
# slots per row, so the v1 micro-batch measurements (50-69 GiB) no longer bound the peak. Grouping
# them here keeps the big-GPU shortcuts (micro 49152, no activation checkpointing) OFF, which is
# what stops a step-1 OOM on a 40-95 GiB card.
_LONG_ROWS = IMAGE_RES != "v1" or AUDIO_RES != "v1" or TEXT_LEN > 256 or AUDIO_LEN > 512   # v2 image / audio rows are 4-6x longer with up to 12 levels: the v1 memory measurements below do not apply (an audio micro-batch ran out of memory on 95 GB)
if PRESET == "full" and VRAM_GB >= 70 and ACT_CKPT < 0 and not _LONG_ROWS:
    ACT_CKPT = 0                                      # no activation checkpointing: faster. 50 GiB peak measured at micro 32768, so it fits an 80 GB H100/A100 too
MICRO_TOKENS = MICRO_TOKENS or (49152 if _BIG_GPU and not _LONG_ROWS else P["micro_tokens"])   # +2% codes/s over 32768 on the RTX PRO 6000 (fuller GEMMs)
BATCH_CODES = BATCH_CODES or P["batch_codes"]
WORKERS = WORKERS or P["workers"]
FIT_IMAGES = FIT_IMAGES or P["fit_images"]
FIT_AUDIO = FIT_AUDIO or P["fit_audio"]
VAL_ROWS = VAL_ROWS or P["val_rows"]
LR = LR or P["lr"]
WARMUP = WARMUP or P["warmup"]

ROW_TYPES = ("text0", "text", "image", "audio")      # text0 = plain text, text = prompt -> response
ROW_MOD = {"text0": "text", "text": "text", "image": "image", "audio": "audio"}
ROW_COND = {"text0": 0, "text": SPEC.cond, "image": SPEC.cond, "audio": SPEC.cond}


class Layout:
    """Static geometry of one row type: where every slot lives and which group routes it."""

    def __init__(self, spec, rtype):
        self.rtype, self.mod, self.cond = rtype, ROW_MOD[rtype], ROW_COND[rtype]
        self.lens = spec.lens(self.mod)
        self.K = len(self.lens)                              # = spec.K, +image_extra for images
        self.lvl, o = [], self.cond
        for n in self.lens:
            self.lvl.append((o, o + n)); o += n
        self.T, self.Tt = o, o - self.cond                   # packed length, target length
        bid = [0] * self.cond + [k + 1 for k, n in enumerate(self.lens) for _ in range(n)]
        self.block_id = torch.tensor(bid, dtype=torch.int32)
        self.mask = self.block_id[None, :] <= self.block_id[:, None]    # [q, kv] True = may attend
        # which MoE group routes which slots (expert-choice runs per segment = per level)
        self.segs = {}
        if self.cond:
            self.segs.setdefault("text", []).append((0, self.cond))
        self.segs.setdefault(self.mod, []).extend(self.lvl)

    def to(self, device):
        self.block_id, self.mask = self.block_id.to(device), self.mask.to(device)
        return self


LAYOUTS = {r: Layout(SPEC, r) for r in ROW_TYPES}
for _r, _l in LAYOUTS.items():
    print(f"{_r:6s} T={_l.T:5d} target={_l.Tt:5d} live codes/row={_l.Tt} levels={_l.lens}")
_gi = math.isqrt(SPEC.image)
print(f"BIT BUDGET (finest resolution, 8 bits per code): "
      f"image {SPEC.image * (1 + SPEC.image_extra) * 8:,} bits/image = {_gi}x{_gi} grid x {1 + SPEC.image_extra} codebook(s) x 8 ({SPEC.side}px, IMAGE_RES={IMAGE_RES}) | "
      f"audio {SPEC.audio * (1 + SPEC.audio_extra) * 8:,} bits/clip = {SPEC.audio * (1 + SPEC.audio_extra) * 8 / (SPEC.wave_len / SPEC.sr):.0f} bit/s ({SPEC.wave_len / SPEC.sr:.1f} s, {1 + SPEC.audio_extra} level(s) at the finest rate, AUDIO_RES={AUDIO_RES}) | text {SPEC.text * 8:,} bits/row (lossless bytes)")

## 3. Frozen codecs (all on the GPU)

No learned encoder: every modality is turned into the **same ladder** of 256-way codes by a fixed
procedure, so "level k" is comparable across text, image and audio.

| modality | finest level (K-1) | coarser levels | decode |
|---|---|---|---|
| text | the byte itself (exact) | span-pooled random byte projection, sign bits = 8-bit locality-sensitive code | bytes -> UTF-8 |
| image | VQ of a `patch x patch x 3` pixel block | 2x2 average-pooled grid (a blurry image), VQ per level | codeword tiles |
| audio | VQ of an 80-bin log-mel frame (16 ms hop) | rate pyramid (average-pool in time), VQ per level | log-mel -> pseudo-inverse -> Griffin-Lim |

Codebooks are **fit once at startup** by k-means on a streamed sample that is held in RAM and then
discarded; they are tiny and live in the checkpoint, so a resumed run uses the same codes.
Quality of image/audio generations is bounded by these codecs: they are coarse by construction.

In [ ]:
class TextCodec:
    def __init__(self, spec, device, seed=1234):
        self.spec, self.device = spec, device
        g = torch.Generator().manual_seed(seed)
        self.planes = torch.randn(256, 8, generator=g).to(device)              # byte -> 8 hash planes
        self.pw = (1 << torch.arange(8, device=device)).long()

    @torch.no_grad()
    def encode(self, b):
        """b [B, L] uint8 -> K long tensors [B, l_k] with codes in 1..256."""
        K, R = self.spec.K, self.spec.R_of("text")
        b = b.long()
        codes = [None] * K
        codes[K - 1] = b + 1
        proj = self.planes[b].transpose(1, 2)                                   # [B, 8, L]
        for j in range(K - 1):
            pooled = F.avg_pool1d(proj, R ** (K - 1 - j)).transpose(1, 2)       # [B, l_j, 8]
            codes[j] = 1 + ((pooled > 0).long() * self.pw).sum(-1)
        return codes

    @staticmethod
    def decode(finest):
        return [bytes((c - 1).clamp(0, 255).to(torch.uint8).cpu().tolist()).rstrip(b"\x00").decode("utf-8", "replace")
                for c in finest]


def kmeans(x, k=256, iters=25, seed=0):
    """x [N, D] float -> centroids [k, D]. Lloyd on the GPU with dead-centroid re-seeding."""
    g = torch.Generator(device=x.device).manual_seed(seed)
    if x.shape[0] > 200_000:
        x = x[torch.randperm(x.shape[0], generator=g, device=x.device)[:200_000]]
    N = x.shape[0]
    if N < k:
        x = torch.cat([x, x[torch.randint(N, (k - N,), generator=g, device=x.device)]
                       + 1e-3 * torch.randn(k - N, x.shape[1], device=x.device, generator=g)])
        N = k
    cent = x[torch.randperm(N, generator=g, device=x.device)[:k]].clone()
    x2 = (x * x).sum(1, keepdim=True)
    for _ in range(iters):
        a = (x2 - 2 * x @ cent.T + (cent * cent).sum(1)).argmin(1)
        sums = torch.zeros_like(cent).index_add_(0, a, x)
        cnt = torch.bincount(a, minlength=k)
        new = sums / cnt.clamp(min=1)[:, None]
        dead = cnt == 0
        if dead.any():
            new[dead] = x[torch.randint(N, (int(dead.sum()),), generator=g, device=x.device)]
        cent = new
    return cent


def sample_points(feats, cap):
    """Keep at most `cap` random rows of each level's features, so fitting never holds a whole corpus."""
    out = []
    for f in feats:
        f = f.reshape(-1, f.shape[-1])
        out.append(f if f.shape[0] <= cap else f[torch.randperm(f.shape[0], device=f.device)[:cap]])
    return out


def vq(x, cb):
    """x [..., D], cb [k, D] -> 1-based codes [...]. argmin ||x-c||^2 via one matmul."""
    sh = x.shape[:-1]
    xf = x.reshape(-1, x.shape[-1])
    d = (cb * cb).sum(1)[None, :] - 2 * xf @ cb.T
    return (d.argmin(1) + 1).view(sh)


class ImageCodec:
    def __init__(self, spec, device):
        self.spec, self.device = spec, device
        self.G, self.p = math.isqrt(spec.image), spec.patch
        self.D = 3 * self.p * self.p
        self.cb = None                                                           # list K x [256, D]

    def features(self, img):
        """img uint8 [B, 3, S, S] -> list K of float [B, n_k, D]; level j is the grid pooled R^(K-1-j)x."""
        K, f = self.spec.K, math.isqrt(self.spec.R_of("image"))
        x = img.float() / 255.0 - 0.5
        B = x.shape[0]
        cur = F.unfold(x, self.p, stride=self.p).view(B, self.D, self.G, self.G)
        feats = [None] * K
        for j in reversed(range(K)):
            feats[j] = cur.flatten(2).transpose(1, 2).contiguous()
            if j > 0:
                cur = F.avg_pool2d(cur, f)
        return feats

    @torch.no_grad()
    def fit(self, feats_list):
        """feats_list: per batch, a list over levels of [m, D] points (see `sample_points`); one codebook per level."""
        self.cb = [kmeans(torch.cat([f[j].reshape(-1, self.D) for f in feats_list]), 256, seed=j)
                   for j in range(self.spec.K)]

    @torch.no_grad()
    def encode(self, img):
        return [vq(f, self.cb[j]) for j, f in enumerate(self.features(img))]

    @torch.no_grad()
    def decode(self, codes, level=None):
        """codes [B, n_j] at `level` (default finest), or the whole code list (then `codes[level]` is used) -> uint8 [B, 3, S, S]."""
        level = self.spec.K - 1 if level is None else level
        if isinstance(codes, (list, tuple)):
            codes = codes[level]
        cells = self.cb[level][(codes - 1).clamp(min=0)]                         # [B, n, D]
        g = math.isqrt(codes.shape[1])
        img = F.fold(cells.transpose(1, 2), (g * self.p, g * self.p), self.p, stride=self.p)
        S = self.spec.side
        if img.shape[-1] != S:
            img = F.interpolate(img, size=(S, S), mode="nearest")
        return ((img + 0.5).clamp(0, 1) * 255).round().to(torch.uint8)

    def state(self):
        return None if self.cb is None else [c.cpu() for c in self.cb]

    def load(self, st):
        self.cb = [c.to(self.device) for c in st]


def decode_jpeg_gpu(blobs, side, device):
    """list of JPEG byte strings -> uint8 [B, 3, side, side] (short side resized, centre-cropped), nvJPEG if available."""
    from torchvision.io import decode_jpeg, ImageReadMode
    ts = [torch.frombuffer(bytearray(b), dtype=torch.uint8) for b in blobs]
    try:
        imgs = decode_jpeg(ts, device=device, mode=ImageReadMode.RGB)
    except Exception:
        imgs = [decode_jpeg(t, mode=ImageReadMode.RGB).to(device) for t in ts]
    out = []
    for im in imgs:
        _, H, W = im.shape
        s = side / min(H, W)
        nh, nw = max(side, round(H * s)), max(side, round(W * s))
        im = F.interpolate(im[None].float(), size=(nh, nw), mode="bilinear", antialias=True, align_corners=False)[0]
        t, l = (nh - side) // 2, (nw - side) // 2
        out.append(im[:, t:t + side, l:l + side].round().clamp(0, 255).to(torch.uint8))
    return torch.stack(out)


def _mel_fb(n_freqs, n_mels, sr):
    """Slaney-style mel filterbank [n_freqs, n_mels] (no torchaudio dependency)."""
    def hz2mel(f):
        f = torch.as_tensor(f, dtype=torch.float64)
        lin = f / (200.0 / 3)
        log = 15.0 + torch.log(f.clamp(min=1000.0) / 1000.0) / (math.log(6.4) / 27.0)
        return torch.where(f >= 1000.0, log, lin)
    def mel2hz(m):
        m = torch.as_tensor(m, dtype=torch.float64)
        lin = (200.0 / 3) * m
        log = 1000.0 * torch.exp((math.log(6.4) / 27.0) * (m - 15.0))
        return torch.where(m >= 15.0, log, lin)
    all_f = torch.linspace(0, sr / 2, n_freqs, dtype=torch.float64)
    pts = mel2hz(torch.linspace(float(hz2mel(0.0)), float(hz2mel(sr / 2)), n_mels + 2, dtype=torch.float64))
    diff = pts[1:] - pts[:-1]
    slopes = pts[None, :] - all_f[:, None]
    down = -slopes[:, :-2] / diff[:-1]
    up = slopes[:, 2:] / diff[1:]
    fb = torch.clamp(torch.minimum(down, up), min=0.0)
    fb = fb * (2.0 / (pts[2:n_mels + 2] - pts[:n_mels]))[None, :]
    return fb.float()


class AudioCodec:
    N_FFT = 1024

    def __init__(self, spec, device):
        self.spec, self.device = spec, device
        self.fb = _mel_fb(self.N_FFT // 2 + 1, spec.n_mels, spec.sr).to(device)    # [513, 80]
        self.pinv = torch.linalg.pinv(self.fb.cpu()).to(device)                    # [80, 513]
        self.win = torch.hann_window(self.N_FFT, device=device)
        self.mean, self.std = -5.0, 2.5
        self.cb = None

    def logmel(self, wave):
        """wave float [B, n] -> log-mel [B, audio, 80] (n = audio * hop)."""
        s = torch.stft(wave, self.N_FFT, self.spec.hop, window=self.win, center=True, return_complex=True)
        mel = (s.abs().pow(2).transpose(1, 2) @ self.fb)[:, :self.spec.audio]
        return torch.log(mel.clamp(min=1e-5))

    def features(self, wave):
        K, R = self.spec.K, self.spec.R_of("audio")
        x = ((self.logmel(wave) - self.mean) / self.std).transpose(1, 2)          # [B, 80, F]
        return [F.avg_pool1d(x, R ** (K - 1 - j)).transpose(1, 2).contiguous() if j < K - 1
                else x.transpose(1, 2).contiguous() for j in range(K)]

    @torch.no_grad()
    def set_stats(self, waves):
        lm = torch.cat([self.logmel(w) for w in waves]).flatten()
        self.mean, self.std = float(lm.mean()), float(lm.std().clamp(min=1e-3))

    @torch.no_grad()
    def fit(self, feats_list):
        self.cb = [kmeans(torch.cat([f[j].reshape(-1, self.spec.n_mels) for f in feats_list]), 256, seed=j)
                   for j in range(self.spec.K)]

    @torch.no_grad()
    def encode(self, wave):
        return [vq(f, self.cb[j]) for j, f in enumerate(self.features(wave))]

    @torch.no_grad()
    def decode_mel(self, codes, level=None):
        """codes [B, n_level] (or the whole code list) -> natural-log mel [B, audio, 80] (codeword lookup, repeated up to the finest rate)."""
        level = self.spec.K - 1 if level is None else level
        if isinstance(codes, (list, tuple)):
            codes = codes[level]
        z = self.cb[level][(codes - 1).clamp(min=0)]                              # [B, n, 80]
        if z.shape[1] != self.spec.audio:
            z = z.repeat_interleave(self.spec.audio // z.shape[1], 1)
        return z * self.std + self.mean

    @torch.no_grad()
    def mel_to_mag(self, mel, iters=None):
        """Power mel [B, F, 80] -> magnitude spectrum [B, 513, F].

        `pinv` (the old path) goes negative, the clamp then leaves spectral holes and Griffin-Lim sounds metallic. `nnls` solves
        S >= 0 with S @ fb ~ mel by multiplicative KL updates (the NMF rule), so the spectrum stays positive and fills the holes."""
        if AUDIO_INV == "pinv":
            return (mel @ self.pinv).clamp(min=1e-8).sqrt().transpose(1, 2)
        W = self.fb                                                               # [513, 80]
        S = ((mel / W.sum(0).clamp(min=1e-8)) @ W.T).clamp(min=1e-8)              # positive back-projection as the start
        den = W.sum(1).clamp(min=1e-8)                                            # [513]
        for _ in range(iters or AUDIO_NNLS_ITERS):
            S = S * (((mel / (S @ W).clamp(min=1e-8)) @ W.T) / den)
        return S.clamp(min=1e-8).sqrt().transpose(1, 2)

    @torch.no_grad()
    def decode(self, codes, level=None, gl_iters=64):
        """codes [B, F] -> waveform float [B, n]: codeword log-mel -> power spec (NNLS) -> Griffin-Lim."""
        mag = self.mel_to_mag(torch.exp(self.decode_mel(codes, level)))
        return griffin_lim(mag, self.win, self.spec.hop, self.N_FFT, gl_iters)

    def state(self):
        return None if self.cb is None else dict(cb=[c.cpu() for c in self.cb], mean=self.mean, std=self.std)

    def load(self, st):
        self.cb, self.mean, self.std = [c.to(self.device) for c in st["cb"]], st["mean"], st["std"]


@torch.no_grad()
def griffin_lim(mag, win, hop, n_fft, iters=64, momentum=0.99):
    """Fast Griffin-Lim. mag [B, n_freq, F] -> waveform [B, (F-1)*hop]."""
    g = torch.Generator(device=mag.device).manual_seed(0)
    phase = torch.exp(2j * math.pi * torch.rand(mag.shape, device=mag.device, generator=g))
    spec, prev = mag * phase, None
    for _ in range(iters):
        wav = torch.istft(spec, n_fft, hop, window=win, center=True)
        est = torch.stft(wav, n_fft, hop, window=win, center=True, return_complex=True)[..., :mag.shape[-1]]
        if prev is not None:
            est = est + momentum * (est - prev)
        prev = est
        spec = mag * est / est.abs().clamp(min=1e-8)
    return torch.istft(spec, n_fft, hop, window=win, center=True)


class Codecs:
    def __init__(self, spec, device):
        if TOKENIZER == "feat":                     # frozen DINOv2 / DAC features + a data-fit text ladder (section 3b)
            self.text = make_text_codec(spec, device)
            self.image = VQImageCodec(spec, device) if IMAGE_CODEC == "vq" else DinoImageCodec(spec, device, FEAT_QUANT)    # IMAGE_CODEC="vq": learned pixel codec (needed for IMAGE_RES="v2")
            self.audio = (DacResidualAudioCodec if AUDIO_RES == "v2" else DacAudioFeatureCodec)(spec, device, FEAT_QUANT)
        elif TOKENIZER == "vq":                     # learned multi-scale residual VQ-VAEs (defined in the next section, trained at startup)
            self.text, self.image, self.audio = TextCodec(spec, device), VQImageCodec(spec, device), VQAudioCodec(spec, device)
        else:
            self.text, self.image, self.audio = TextCodec(spec, device), ImageCodec(spec, device), AudioCodec(spec, device)

    def ready(self):
        return self.image.cb is not None and self.audio.cb is not None

    def state(self):
        d = dict(image=self.image.state(), audio=self.audio.state())
        if hasattr(self.text, "state") and self.text.state() is not None:       # only the data-fit text codecs have a state
            d["text"] = self.text.state()
        return d

    def load(self, st):
        self.image.load(st["image"]); self.audio.load(st["audio"])
        if st.get("text") is not None and hasattr(self.text, "load"):
            self.text.load(st["text"])

## 4. Data: streamed from the Hub

```
Hub (streaming=True) -> W worker processes (decode, cheap) -> micro-batch builder
     -> GPU codecs on a side CUDA stream -> RAM ring (PREFETCH batches) -> trainer
```

* **No raw data and no dataset cache are written.** The only buffer is the in-RAM ring; the encoded codes can be
  recorded as shards (next section).
* Rows are **modality-homogeneous** (text-only, prompt->response, caption->image, transcript->speech),
  so there are exactly 4 static pack shapes and no wasted NULL slots.
* The stream is split 3 ways by shard (or by row index when a dataset has few shards):
  part 0 = validation, part 1 = codebook fit, parts 2.. = training workers. Splits never overlap.
* A source that errors is retried with backoff and dropped after repeated failures (gated or renamed
  datasets cost their own weight, not the run).

Sources are what open pretraining recipes use: FineWeb-Edu / Wikipedia / Cosmopedia / code / OpenWebMath
(text), NuminaMath + SmolTalk (instruction), CC12M / CC3M / COCO / DocVQA (image+caption),
LibriSpeech / LibriTTS-R / People's Speech (audio+transcript).

In [ ]:
import re as _re


@dataclass
class Source:
    name: str
    kind: str                      # text0 | text | image | audio
    path: str
    config: str = None
    split: str = "train"
    weight: float = 1.0
    raw: dict = None               # column -> "image" | "audio" (kept as undecoded bytes)
    buf: int = 512                 # shuffle buffer (rows, in RAM)
    fn: str = ""                   # extractor name


# ---- which sources ---------------------------------------------------------------------------
# "lean" (default): ONE corpus per modality, so a codec/ladder change has no source-mix confound and
#   a gain can only come from the thing you changed.
#     text  fineweb-edu sample-10BT       - the strongest open filtered web corpus
#     image ImageNet-1k resized 256        - caption = the class label, i.e. a 1000-way DETERMINISTIC
#                                            function of the image. That is a cleaner caption test than
#                                            CC12M alt-text, which is why image looked like it used
#                                            its caption while using only the caption's presence.
#     audio LibriSpeech 960 h (3 splits)   - clean read speech, sentence-level clips that fit the 8 s
#                                            window. 9.6x the distinct audio of train.100 alone, which
#                                            is the direct response to the T1 audio overfit.
# "rich": the original multi-source mix. NCM_SRC_SET=rich.
SRC_SET = _knob("SRC_SET", "lean")
KEEP_INSTRUCT = _knob("KEEP_INSTRUCT", False)   # fineweb-edu has no prompts: without this there are no
                                                # prompt->response text rows and the text prefix test
                                                # cannot run. Adds HuggingFaceTB/smoltalk (small).
AUDIO_SRC = _knob("AUDIO_SRC", "peoples")   # librispeech | libritts | peoples | gigaspeech | emilia

# GigaSpeech (10k h) and Emilia (101k h) are the modern large corpora, but both are GATED: the Hub
# refuses them (auto-gate) until the terms are accepted on the dataset page. Swap AUDIO_SRC once they are.
AUDIO_SETS = {
    # People's Speech is the largest OPEN transcript-aligned corpus on the Hub (~30k h across clean +
    # dirty). GigaSpeech (10k h) and Emilia (101k h) are gated; espnet/yodas and yodas2 are DEAD
    # ("dataset scripts are no longer supported"); MLS ships no `english` config any more.
    # MEASURED with this dataset's own duration_ms: median 14.8 s, p10 14.3, p90 15.0 - and only 1%
    # of clips fit the old 8.19 s window. That is why the source held weight 0.85 and still left audio
    # overfitting: 99% of it was thrown away at the door. AUDIO_LEN=1024 frames = 16.4 s covers it.
    "peoples": [
        Source("peoples_clean", "audio", "MLCommons/peoples_speech", "clean", weight=0.60,
               raw={"audio": "audio"}, buf=64, fn="a_text"),
        Source("peoples_dirty", "audio", "MLCommons/peoples_speech", "dirty", weight=0.40,
               raw={"audio": "audio"}, buf=64, fn="a_text"),
    ],
    "librispeech": [
        Source("libri100", "audio", "openslr/librispeech_asr", "all", "train.clean.100", weight=1 / 3,
               raw={"audio": "audio"}, buf=64, fn="a_text"),
        Source("libri360", "audio", "openslr/librispeech_asr", "all", "train.clean.360", weight=1 / 3,
               raw={"audio": "audio"}, buf=64, fn="a_text"),
        Source("libri500", "audio", "openslr/librispeech_asr", "all", "train.other.500", weight=1 / 3,
               raw={"audio": "audio"}, buf=64, fn="a_text"),
    ],
    "libritts": [Source("libritts", "audio", "mythicinfinity/libritts_r", "clean", "train.clean.100",
                        weight=1.0, raw={"audio": "audio"}, buf=64, fn="a_tts")],
    "gigaspeech": [Source("giga", "audio", "speechcolab/gigaspeech", "s", "train", weight=1.0,
                          raw={"audio": "audio"}, buf=64, fn="a_text")],
    "emilia": [Source("emilia", "audio", "amphion/Emilia-Dataset", weight=1.0,
                      raw={"audio": "audio"}, buf=64, fn="a_text")],
}
if AUDIO_SRC not in AUDIO_SETS:
    raise SystemExit(f"NCM_AUDIO_SRC={AUDIO_SRC!r} unknown; pick from {sorted(AUDIO_SETS)}")

if SRC_SET == "lean":
    # Largest sources that carry image BYTES. Measured against the Hub: DataComp-1B (1.28B) and
    # Recap-DataComp-1B are `url` + caption only, conceptual_captions is image_url + caption, LAION-2B
    # is gated. So CC12M + CC3M = 15.3M captioned images is the open ceiling that streams as bytes,
    # and it is 12x ImageNet-1k with real captions instead of a class label.
    SOURCES = [
        Source("fineweb", "text0", "HuggingFaceFW/fineweb-edu", "sample-10BT", weight=1.0, fn="t_text"),
        Source("cc12m", "image", "pixparse/cc12m-wds", weight=12 / 15.3, raw={"jpg": "image"},
               buf=128, fn="i_wds"),
        Source("cc3m", "image", "pixparse/cc3m-wds", weight=3.3 / 15.3, raw={"jpg": "image"},
               buf=128, fn="i_wds"),
    ] + AUDIO_SETS[AUDIO_SRC]
    if KEEP_INSTRUCT:
        SOURCES.append(Source("smoltalk", "text", "HuggingFaceTB/smoltalk", "all", weight=1.0,
                              fn="p_smoltalk"))
else:
    SOURCES = [
        # --- text, plain pretraining (text0)
        Source("fineweb", "text0", "HuggingFaceFW/fineweb-edu", "sample-10BT", weight=0.45, fn="t_text"),
        Source("wiki", "text0", "wikimedia/wikipedia", "20231101.en", weight=0.15, fn="t_text"),
        Source("cosmo", "text0", "HuggingFaceTB/smollm-corpus", "cosmopedia-v2", weight=0.15, fn="t_text"),
        Source("code", "text0", "codeparrot/codeparrot-clean", weight=0.10, fn="t_content"),
        Source("owm", "text0", "open-web-math/open-web-math", weight=0.15, fn="t_text"),
        # --- text, prompt -> response
        Source("numina", "text", "AI-MO/NuminaMath-CoT", weight=0.5, fn="p_numina"),
        Source("smoltalk", "text", "HuggingFaceTB/smoltalk", "all", weight=0.5, fn="p_smoltalk"),
        # --- image + caption
        # weights follow how much DISTINCT data a source has: a small source at a high weight is repeated tens of times in a short run and gets memorised
        Source("cc12m", "image", "pixparse/cc12m-wds", weight=0.60, raw={"jpg": "image"}, buf=128, fn="i_wds"),
        Source("cc3m", "image", "pixparse/cc3m-wds", weight=0.33, raw={"jpg": "image"}, buf=128, fn="i_wds"),
        Source("docvqa", "image", "HuggingFaceM4/DocumentVQA", weight=0.05, raw={"image": "image"}, buf=64, fn="i_docvqa"),
        Source("coco", "image", "lmms-lab/COCO-Caption2017", split="val", weight=0.02, raw={"image": "image"}, buf=64, fn="i_coco"),
        # --- audio + transcript (clips longer than the 8 s window are dropped, so the Libri sets keep only a fraction of their rows)
        Source("libri", "audio", "openslr/librispeech_asr", "clean", "train.100", weight=0.05, raw={"audio": "audio"}, buf=64, fn="a_text"),
        Source("libritts", "audio", "mythicinfinity/libritts_r", "clean", "train.clean.100", weight=0.10, raw={"audio": "audio"}, buf=64, fn="a_tts"),
        Source("peoples", "audio", "MLCommons/peoples_speech", "clean", weight=0.85, raw={"audio": "audio"}, buf=64, fn="a_text"),
    ]
for _s in SOURCES:
    if _s.name in SOURCE_WEIGHTS:
        _s.weight = float(SOURCE_WEIGHTS[_s.name])
SOURCES = [s for s in SOURCES if s.weight > 0]
_kinds = sorted({s.kind for s in SOURCES})
print(f"source set {SRC_SET}: {len(SOURCES)} sources " + (f"| audio {AUDIO_SRC}" if SRC_SET == "lean" else ""))
for _s in SOURCES:
    print(f"  {_s.kind:6s} {_s.name:10s} {_s.path}"
          + (f" | config {_s.config}" if _s.config else "") + f" | split {_s.split}")
_missing = [r for r in ROW_TYPES if r not in _kinds]
if _missing:
    print(f"  !! no source produces row type(s) {_missing}. Those rows never appear, their share of "
          f"the code budget is redistributed, and any check that needs them cannot run."
          + (" Set NCM_KEEP_INSTRUCT=1 to add a prompt->response text source." if "text" in _missing else ""))


def _bytes(s, n):
    b = (s or "").encode("utf-8", "ignore")[:n]
    a = np.zeros(n, np.uint8)
    a[:len(b)] = np.frombuffer(b, np.uint8)
    return a, len(b)


def _clean(s):
    return _re.sub(r"\s+", " ", s or "").strip()


# ---- extractors: HF example -> list of row dicts (numpy / bytes only, so they pickle cheaply) ----
TEXT_PACK = _knob("TEXT_PACK", 1)      # 1 = pool consecutive documents into exactly-`text`-byte
                                       # samples. 0 = the old single-document window, which
                                       # DROPS every document shorter than text/2 and caps a
                                       # document at 4 samples - at text=1024/2048 that deletes
                                       # whole sources from the mix without saying so.
_TXT_BUF = {}                          # per-worker remainder, keyed by sample length


def _x_t_text(ex, spec, rng, key="text"):
    raw = (ex.get(key) or "").encode("utf-8", "ignore")
    L = spec.text
    out = []
    if not TEXT_PACK:
        if len(raw) < L // 2:
            return []
        n = min(4, max(1, len(raw) // L))
        for _ in range(n):
            st = rng.randint(0, max(0, len(raw) - L))
            a = np.zeros(L, np.uint8); c = raw[st:st + L]; a[:len(c)] = np.frombuffer(c, np.uint8)
            out.append(dict(t="text0", tgt=a))
        return out
    buf = _TXT_BUF.pop(L, b"") + raw + b"\n"
    while len(buf) >= L:
        out.append(dict(t="text0", tgt=np.frombuffer(buf[:L], np.uint8).copy()))
        buf = buf[L:]
    _TXT_BUF[L] = buf                     # shorter than L: carried to the next document
    return out

def x_t_text(ex, spec, rng): return _x_t_text(ex, spec, rng, "text")
def x_t_content(ex, spec, rng): return _x_t_text(ex, spec, rng, "content")


def _prompt_row(p, r, spec):
    p, r = (p or "").strip(), (r or "").strip()
    if len(p) < 4 or len(r) < 8:
        return []
    return [dict(t="text", cond=_bytes(p, spec.cond)[0], tgt=_bytes(r, spec.text)[0])]

def x_p_numina(ex, spec, rng): return _prompt_row(ex.get("problem"), ex.get("solution"), spec)

def x_p_smoltalk(ex, spec, rng):
    m = ex.get("messages") or []
    u = next((x["content"] for x in m if x.get("role") == "user"), None)
    a = next((x["content"] for x in m if x.get("role") == "assistant"), None)
    return _prompt_row(u, a, spec)


def _img_row(img, cap, spec):
    """img: {'bytes':..} from an undecoded Image column. JPEGs stay as bytes for nvJPEG; others are decoded here."""
    cap = _clean(cap)
    if not cap or not img or not img.get("bytes"):
        return []
    raw = img["bytes"]
    row = dict(t="image", cond=_bytes(cap, spec.cond)[0])
    if GPU_JPEG and HAS_NVJPEG and raw[:2] == b"\xff\xd8":
        row["jpeg"] = raw
        return [row]
    from PIL import Image
    im = Image.open(io.BytesIO(raw))
    S = spec.side
    im.draft("RGB", (S * 2, S * 2))
    im = im.convert("RGB")
    w, h = im.size
    s = S / min(w, h)
    im = im.resize((max(S, round(w * s)), max(S, round(h * s))), Image.BICUBIC)
    w, h = im.size
    l, t = (w - S) // 2, (h - S) // 2
    row["img"] = np.asarray(im.crop((l, t, l + S, t + S)), dtype=np.uint8).transpose(2, 0, 1).copy()
    return [row]

def x_i_wds(ex, spec, rng): return _img_row(ex.get("jpg"), ex.get("txt"), spec)
def x_i_coco(ex, spec, rng):
    a = ex.get("answer")
    return _img_row(ex.get("image"), a[0] if isinstance(a, list) and a else a, spec)
def x_i_docvqa(ex, spec, rng):
    a = ex.get("answers") or [""]
    return _img_row(ex.get("image"), f"{ex.get('question', '')} {a[0] if a else ''}", spec)


def _wave(aud, spec):
    import soundfile as sf
    data, sr = sf.read(io.BytesIO(aud["bytes"]), dtype="float32", always_2d=False)
    if data.ndim > 1:
        data = data.mean(1)
    if sr != spec.sr:
        x = torch.from_numpy(data)[None, None]
        data = F.interpolate(x, size=int(len(data) * spec.sr / sr), mode="linear", align_corners=False)[0, 0].numpy()
    return data


_AUD = {"seen": 0, "kept": 0}      # audio clips offered vs kept, per worker process


def _aud_row(aud, text, spec):
    text = _clean(text).lower()
    if not text or not aud or not aud.get("bytes"):
        return []
    w = _wave(aud, spec)
    n = spec.wave_len
    _AUD["seen"] += 1
    if len(w) > n or len(w) < spec.sr // 2:           # a crop would no longer match the transcript
        if _AUD["seen"] % 500 == 1:
            print(f"  [audio] {_AUD['seen']} clips offered, {_AUD['kept']} kept "
                  f"({100 * _AUD['kept'] / max(1, _AUD['seen']):.0f}%) at a {n / spec.sr:.1f} s window "
                  f"- raise NCM_AUDIO_LEN if this is low", file=sys.stderr, flush=True)
        return []
    _AUD["kept"] += 1
    pcm = np.zeros(n, np.int16)
    pcm[:len(w)] = np.clip(w * 32767, -32768, 32767).astype(np.int16)
    return [dict(t="audio", cond=_bytes(text, spec.cond)[0], wav=pcm)]

def x_a_text(ex, spec, rng): return _aud_row(ex.get("audio"), ex.get("text"), spec)
def x_a_tts(ex, spec, rng): return _aud_row(ex.get("audio"), ex.get("text_normalized") or ex.get("text_original"), spec)

EXTRACT = {k[2:]: v for k, v in list(globals().items()) if k.startswith("x_")}


def open_stream(src, part, nparts, seed):
    from datasets import load_dataset, Audio, Image
    kw = dict(split=src.split, streaming=True)
    ds = None
    if STREAM_PREFETCH:                              # Parquet read-ahead + 128 MiB requests (HF "streaming datasets" blog)
        try:
            import pyarrow, pyarrow.dataset as pds
            fso = pds.ParquetFragmentScanOptions(cache_options=pyarrow.CacheOptions(prefetch_limit=1, range_size_limit=128 << 20))
            ds = (load_dataset(src.path, src.config, fragment_scan_options=fso, **kw) if src.config
                  else load_dataset(src.path, fragment_scan_options=fso, **kw))
        except Exception:
            ds = None                                # non-Parquet builder or older `datasets`: plain stream
    if ds is None:
        ds = load_dataset(src.path, src.config, **kw) if src.config else load_dataset(src.path, **kw)
    if part == 1 and nparts > 1:
        print(f"    [{src.name}] shards={ds.n_shards} (streams {nparts - 1})", flush=True)
    for col, kind in (src.raw or {}).items():
        ds = ds.cast_column(col, Audio(decode=False) if kind == "audio" else Image(decode=False))
    if nparts > 1:
        n_sh = ds.n_shards or 1
        if n_sh >= nparts:
            ds = ds.shard(num_shards=nparts, index=part)
        elif n_sh >= 2:                              # fewer shards than parts: part 0 (validation) owns shard 0, the training workers share shards 1.. (two workers may read one shard, each with its own shuffle)
            ds = ds.shard(num_shards=n_sh, index=0 if part == 0 else 1 + (part - 1) % (n_sh - 1))
        else:                                        # one shard: partition by row index (every part reads it all)
            ds = ds.filter(lambda ex, i, _p=part, _n=nparts: i % _n == _p, with_indices=True)
    return iter(ds.shuffle(seed=seed, buffer_size=src.buf))


def rows_of(src, spec, part, nparts, seed, rng, log=None):
    """Endless generator of rows from one source; reconnects with backoff, gives up after repeated failure."""
    fn, fails, epoch = EXTRACT[src.fn], 0, 0
    while True:
        try:
            for ex in open_stream(src, part, nparts, seed + 977 * epoch):
                for r in fn(ex, spec, rng):
                    r["src"] = src.name
                    fails = 0
                    yield r
            epoch += 1                                # stream exhausted: reshuffle and go again
        except Exception as e:
            fails += 1
            if log:
                import traceback
                fr = traceback.extract_tb(e.__traceback__)[-1]
                log(f"[{src.name}] {type(e).__name__}: {str(e)[:100]} @ {os.path.basename(fr.filename)}:{fr.lineno} (fail {fails})")
            if fails > 6:
                if log:
                    log(f"[{src.name}] dropped")
                return
            time.sleep(min(30, 2 ** fails) * (0.5 + random.random()))      # jitter: workers must not retry in lock-step


# ---- row-type mix: codes share per modality -> sampling probability per row type ----
CODE_SHARE = {"text": 0.40, "image": 0.35, "audio": 0.25}      # of live codes
TEXT_PROMPT_FRAC = 0.15                                          # of text codes that are prompt->response
def type_probs():
    share = {"text0": CODE_SHARE["text"] * (1 - TEXT_PROMPT_FRAC), "text": CODE_SHARE["text"] * TEXT_PROMPT_FRAC,
             "image": CODE_SHARE["image"], "audio": CODE_SHARE["audio"]}
    p = {r: share[r] / LAYOUTS[r].Tt for r in ROW_TYPES}                # codes -> rows
    z = sum(p.values())
    return {r: v / z for r, v in p.items()}
TYPE_P = type_probs()


class MixDataset:
    """Samples sources by (row-type probability x source weight) and yields raw rows for one worker."""

    def __init__(self, spec, kinds=None, part0=1, nparts=2, seed=0, even=False):
        self.spec, self.part0, self.nparts, self.seed = spec, part0, nparts, seed
        srcs = [s for s in SOURCES if kinds is None or s.kind in kinds]
        tw = {}
        for s in srcs:
            tw[s.kind] = tw.get(s.kind, 0) + s.weight
        self.srcs = srcs
        self.p = [(1.0 if even else TYPE_P[s.kind]) * s.weight / tw[s.kind] for s in srcs]   # even: equal share per row type

    def rows(self, wid=0):
        part = self.part0 + wid                      # worker w reads partition part0 + w
        rng = random.Random(self.seed * 7919 + part)
        log = lambda m: print(f"  [stream w{wid}] {m}", file=sys.stderr, flush=True)
        gens = {}
        alive = list(range(len(self.srcs)))
        while alive:
            i = rng.choices(alive, [self.p[j] for j in alive])[0]
            if i not in gens:
                gens[i] = rows_of(self.srcs[i], self.spec, part, self.nparts, self.seed * 31 + part, rng, log)
            try:
                yield next(gens[i])
            except StopIteration:
                alive.remove(i)


def _stream_worker(ds, wid, q, stop):
    torch.set_num_threads(1)                         # a forked child must not inherit the parent's OpenMP pool
    time.sleep(min(20.0, 0.6 * wid))                 # stagger the start: dozens of workers resolving the same Hub datasets at once get rate-limited
    try:
        for row in ds.rows(wid):
            while not stop.is_set():
                try:
                    q.put(row, timeout=1.0)
                    break
                except queue.Full:
                    pass
            if stop.is_set():
                return
    except Exception as e:
        print(f"  [stream w{wid}] worker died: {e!r}", file=sys.stderr, flush=True)


class RowStream:
    """Rows from `workers` forked processes into one queue, delivered in ARRIVAL order.

    (A torch DataLoader hands out worker batches strictly round-robin, so one worker that is busy
    opening a slow dataset stalls every other worker. Streaming needs the opposite.)
    """

    def __init__(self, ds, workers, depth=256):
        self.ds, self.workers = ds, workers
        if workers == 0:
            self.gen = ds.rows(0)
            return
        import multiprocessing as mp
        ctx = mp.get_context("fork")
        self.q, self.stop = ctx.Queue(maxsize=depth), ctx.Event()
        self.procs = [ctx.Process(target=_stream_worker, args=(ds, w, self.q, self.stop), daemon=True)
                      for w in range(workers)]
        for pr in self.procs:
            pr.start()

    def __iter__(self):
        return self

    def __next__(self):
        if self.workers == 0:
            return next(self.gen)
        while True:
            try:
                return self.q.get(timeout=2.0)
            except queue.Empty:
                if not any(pr.is_alive() for pr in self.procs):
                    raise RuntimeError("all stream workers exited")

    def close(self):
        if self.workers:
            self.stop.set()
            time.sleep(0.2)
            for pr in self.procs:
                if pr.is_alive():
                    pr.terminate()


# ---- rows -> GPU codes ----
def micro_batch_sizes():
    return {r: max(1, MICRO_TOKENS // LAYOUTS[r].T) for r in ROW_TYPES}
MICRO_B = micro_batch_sizes()


@torch.no_grad()
def encode_rows(rtype, rows, codecs, device=dev):
    """list of raw rows (one row type) -> dict(rtype, cond [B,Lc] long|None, codes list, n_live)."""
    B = len(rows)
    cond = None
    if LAYOUTS[rtype].cond:
        cond = torch.from_numpy(np.stack([r["cond"] for r in rows])).to(device).long() + 1
    if ROW_MOD[rtype] == "text":
        codes = codecs.text.encode(torch.from_numpy(np.stack([r["tgt"] for r in rows])).to(device))
    elif ROW_MOD[rtype] == "image":
        S = SPEC.side
        imgs = [None] * B
        jidx = [i for i, r in enumerate(rows) if "jpeg" in r]
        if jidx:
            dec = decode_jpeg_gpu([rows[i]["jpeg"] for i in jidx], S, device)
            for n, i in enumerate(jidx):
                imgs[i] = dec[n]
        for i, r in enumerate(rows):
            if imgs[i] is None:
                imgs[i] = torch.from_numpy(r["img"]).to(device)
        codes = codecs.image.encode(torch.stack(imgs))
    else:
        w = torch.from_numpy(np.stack([r["wav"] for r in rows])).to(device).float() / 32768.0
        codes = codecs.audio.encode(w)
    return dict(rtype=rtype, cond=cond, codes=codes, n_live=B * LAYOUTS[rtype].Tt,
                src=[r.get("src", "?") for r in rows])


class Prefetcher:
    """Background thread: loader rows -> per-type micro-batches -> GPU codecs (side stream) -> RAM ring."""

    def __init__(self, loader, codecs, depth=PREFETCH, writer=None):
        self.q = queue.Queue(maxsize=depth)
        self.writer = writer
        self.loader, self.codecs = loader, codecs   # loader: any iterator of raw rows (RowStream)
        self.stop = threading.Event()
        self.err = None
        self.rows_in = 0
        self.stream = torch.cuda.Stream() if IS_CUDA else None
        self.t = threading.Thread(target=self._run, daemon=True)
        self.t.start()

    def _run(self):
        try:
            bufs = {r: [] for r in ROW_TYPES}
            for row in self.loader:
                if self.stop.is_set():
                    return
                self.rows_in += 1
                b = bufs[row["t"]]
                b.append(row)
                if len(b) >= MICRO_B[row["t"]]:
                    rows, bufs[row["t"]] = b, []
                    if self.stream is not None:
                        with torch.cuda.stream(self.stream):
                            mb = encode_rows(row["t"], rows, self.codecs)
                            if self.writer is not None:
                                self.writer.add(mb)             # D2H copy on the side stream: never waits on training kernels
                        self.stream.synchronize()
                    else:
                        mb = encode_rows(row["t"], rows, self.codecs)
                        if self.writer is not None:
                            self.writer.add(mb)
                    while not self.stop.is_set():
                        try:
                            self.q.put(mb, timeout=1.0)
                            break
                        except queue.Full:
                            pass
        except Exception as e:                                  # surfaced to the trainer instead of a silent stall
            self.err = e

    def get(self, timeout=600):
        t0 = time.time()
        while True:
            try:
                mb = self.q.get(timeout=1.0)
                break
            except queue.Empty:
                if self.err is not None:
                    raise RuntimeError(f"data thread died: {self.err!r}")
                if time.time() - t0 > timeout:
                    raise TimeoutError(f"no batch for {timeout}s: streams are stalled or throttled (set HF_TOKEN)")
        if IS_CUDA:                                              # tensors were born on the side stream
            cur = torch.cuda.current_stream()
            for t in ([mb["cond"]] if mb["cond"] is not None else []) + mb["codes"]:
                t.record_stream(cur)
        return mb

    def close(self):
        self.stop.set()


def collect_rows(kinds, quota, part0, nparts, workers, timeout=1800, seed=0, even=False):
    """Pull `quota[row_type]` raw rows into RAM from partitions part0.. and stop. Nothing touches disk."""
    ds = MixDataset(SPEC, kinds=kinds, part0=part0, nparts=nparts, seed=seed, even=even)
    out, t0, last = [], time.time(), time.time()
    cnt = {k: 0 for k in quota}
    stream = RowStream(ds, workers)
    try:
        for row in stream:
            t = row["t"]
            if t in cnt and cnt[t] < quota[t]:
                out.append(row); cnt[t] += 1
            if all(cnt[k] >= quota[k] for k in cnt):
                break
            if time.time() - last > 20:
                print(f"    collecting {cnt} / {quota}  ({time.time() - t0:.0f}s)", flush=True); last = time.time()
            if time.time() - t0 > timeout:
                print(f"    !! timeout: got {cnt} of {quota}")
                break
    finally:
        stream.close()
    return out

## 4b. Learned tokenizers: multi-scale residual VQ-VAE for images and audio

The frozen k-means codecs cap image fidelity near 20 dB (a blocky mosaic) and give robotic audio. These replace them
with small **learned** tokenizers that fit the same ladder, so the model, the layouts and the vocabulary (257) do not change:

* a convolutional **encoder** maps the image (or the log-mel) to a latent grid at the finest ladder resolution (16x16, or 512 frames);
* **multi-scale residual quantisation** (VAR style): level 0 quantises the average of the latent, every next level quantises
  what the levels before it missed, one 256-entry codebook per level (EMA-updated, dead codes re-seeded);
* a convolutional **decoder** rebuilds the image / mel from the sum of the quantised levels, so decoding levels 0..k gives a
  progressively sharper result (trained on random prefixes as well as the full stack).

Because the codes are residual, decoding level *k* needs the codes of **all** levels <= *k*: `decode(codes_list, level=k)`.
They are trained once at startup on a streamed sample held in RAM (same idea as the old codebook fit) and the weights go into the
checkpoint. **Gate:** before any model compute is spent, each learned tokenizer is compared with a k-means codec fitted on the same
data; if the image tokenizer is not `TOK_GATE_DB` dB better (audio: 5% lower log-mel error) the notebook stops.
The bit budget is unchanged (8 bits per code), so expect a real gain from the decoder and the residual structure,
not a different league; much higher fidelity needs more codes per image.

In [ ]:
def _gn(c):
    return nn.GroupNorm(8, c)


class _Res2d(nn.Module):
    def __init__(self, c):
        super().__init__()
        self.n1, self.c1, self.n2, self.c2 = _gn(c), nn.Conv2d(c, c, 3, padding=1), _gn(c), nn.Conv2d(c, c, 3, padding=1)

    def forward(self, x):
        return x + self.c2(F.silu(self.n2(self.c1(F.silu(self.n1(x))))))


class _Res1d(nn.Module):
    def __init__(self, c, dil=1):
        super().__init__()
        self.n1, self.c1 = _gn(c), nn.Conv1d(c, c, 5, padding=2 * dil, dilation=dil)
        self.n2, self.c2 = _gn(c), nn.Conv1d(c, c, 5, padding=2)

    def forward(self, x):
        return x + self.c2(F.silu(self.n2(self.c1(F.silu(self.n1(x))))))


class EMAQuant(nn.Module):
    """One 256-entry codebook updated by exponential moving averages (no codebook gradient); dead codes are re-seeded from the batch."""

    def __init__(self, n=256, d=64, decay=0.99, eps=1e-5):
        super().__init__()
        self.n, self.d, self.decay, self.eps = n, d, decay, eps
        self.register_buffer("embed", torch.randn(n, d) * 0.5)
        self.register_buffer("size", torch.ones(n))
        self.register_buffer("avg", self.embed.clone())
        self.register_buffer("steps", torch.zeros(()))
        self.register_buffer("seeded", torch.zeros(()))

    def forward(self, x):
        """x [N, d] -> (quantised, straight-through [N, d]; index [N]; commitment mse)."""
        with torch.autocast("cuda", enabled=False):
            x = x.float()
            if self.training and float(self.seeded) == 0:                              # first batch: start the codebook on real data
                pick = torch.randint(x.shape[0], (self.n,), device=x.device)
                self.embed.copy_(x[pick].detach() + 0.01 * torch.randn(self.n, self.d, device=x.device))
                self.avg.copy_(self.embed); self.seeded.fill_(1)
            dist = (x * x).sum(1, keepdim=True) - 2 * x @ self.embed.T + (self.embed * self.embed).sum(1)
            idx = dist.argmin(1)
            q = self.embed[idx]
            if self.training:
                with torch.no_grad():
                    oh = F.one_hot(idx, self.n).float()
                    self.size.mul_(self.decay).add_(oh.sum(0), alpha=1 - self.decay)
                    self.avg.mul_(self.decay).add_(oh.T @ x.detach(), alpha=1 - self.decay)
                    tot = self.size.sum()
                    sz = (self.size + self.eps) / (tot + self.n * self.eps) * tot
                    self.embed.copy_(self.avg / sz[:, None])
                    self.steps += 1
                    if int(self.steps) % 25 == 0:                                      # re-seed codes used < 5% as often as an average code
                        dead = self.size < 0.05 * tot / self.n
                        nd = int(dead.sum())
                        if nd:
                            r = x.detach()[torch.randint(x.shape[0], (nd,), device=x.device)]
                            self.embed[dead] = r; self.avg[dead] = r; self.size[dead] = tot / self.n
            commit = F.mse_loss(x, q.detach())
            q = x + (q - x).detach()
        return q, idx, commit


class MSRVQ(nn.Module):
    """Multi-scale residual quantiser. `sizes` = grid side per level (2D) or length per level (1D); input latent [B, d, *finest]."""

    def __init__(self, sizes, d, nd):
        super().__init__()
        self.sizes, self.d, self.nd = list(sizes), d, nd
        self.q = nn.ModuleList(EMAQuant(256, d) for _ in self.sizes)

    def _pool(self, r, n):
        return F.adaptive_avg_pool2d(r, n) if self.nd == 2 else F.adaptive_avg_pool1d(r, n)

    def _up(self, q, size):
        return (F.interpolate(q, size=(size, size), mode="bilinear", align_corners=False) if self.nd == 2
                else F.interpolate(q, size=size, mode="linear", align_corners=False))

    def forward(self, z):
        """z [B, d, ...] float -> (summed quantised latent after each level [K], 1-based codes per level [K x [B, n^nd]], commitment loss)."""
        fin = z.shape[-1]
        f_hat, outs, codes, commit = torch.zeros_like(z), [], [], 0.0
        for n, qz in zip(self.sizes, self.q):
            rp = self._pool(z - f_hat, n)                                               # what is still missing, at this level's resolution
            flat = rp.permute(0, 2, 3, 1).reshape(-1, self.d) if self.nd == 2 else rp.permute(0, 2, 1).reshape(-1, self.d)
            qf, idx, c = qz(flat)
            qg = qf.view(rp.shape[0], *rp.shape[2:], self.d)
            qg = qg.permute(0, 3, 1, 2) if self.nd == 2 else qg.permute(0, 2, 1)
            f_hat = f_hat + self._up(qg, fin)
            outs.append(f_hat); codes.append(idx.view(rp.shape[0], -1) + 1); commit = commit + c
        return outs, codes, commit

    @torch.no_grad()
    def from_codes(self, codes, upto):
        """1-based codes (list with >= upto+1 entries) -> summed quantised latent after levels 0..upto, at the finest size."""
        fin, f_hat = self.sizes[-1], 0.0
        for k in range(upto + 1):
            n = self.sizes[k]
            e = self.q[k].embed[(codes[k] - 1).clamp(min=0)]                            # [B, n^nd, d]
            e = e.view(e.shape[0], n, n, self.d).permute(0, 3, 1, 2) if self.nd == 2 else e.permute(0, 2, 1)
            f_hat = f_hat + self._up(e, fin)
        return f_hat


class ImageVQ(nn.Module):
    def __init__(self, spec, d=64):
        super().__init__()
        nd = int(round(math.log2(spec.patch)))
        assert 2 ** nd == spec.patch, f"patch {spec.patch} must be a power of two"
        ch = [min(32 * 2 ** i, 256) for i in range(nd + 1)]
        enc = [nn.Conv2d(3, ch[0], 3, padding=1)]
        for i in range(nd):
            enc += [_Res2d(ch[i]), nn.Conv2d(ch[i], ch[i + 1], 4, stride=2, padding=1)]
        enc += [_Res2d(ch[nd]), _Res2d(ch[nd]), _gn(ch[nd]), nn.SiLU(), nn.Conv2d(ch[nd], d, 1)]
        dec = [nn.Conv2d(d, ch[nd], 3, padding=1), _Res2d(ch[nd]), _Res2d(ch[nd])]
        for i in reversed(range(nd)):
            dec += [nn.Upsample(scale_factor=2, mode="nearest"), nn.Conv2d(ch[i + 1], ch[i], 3, padding=1), _Res2d(ch[i])]
        dec += [_gn(ch[0]), nn.SiLU(), nn.Conv2d(ch[0], 3, 3, padding=1)]
        self.enc, self.dec = nn.Sequential(*enc), nn.Sequential(*dec)
        self.rvq = MSRVQ([math.isqrt(n) for n in spec.lens("image")], d, 2)


class AudioVQ(nn.Module):
    def __init__(self, spec, d=64, ch=256):
        super().__init__()
        m = spec.n_mels
        self.enc = nn.Sequential(nn.Conv1d(m, ch, 5, padding=2), _Res1d(ch, 1), _Res1d(ch, 2), _Res1d(ch, 4), _Res1d(ch, 8),
                                 _gn(ch), nn.SiLU(), nn.Conv1d(ch, d, 1))
        self.dec = nn.Sequential(nn.Conv1d(d, ch, 5, padding=2), _Res1d(ch, 1), _Res1d(ch, 2), _Res1d(ch, 4), _Res1d(ch, 8),
                                 _gn(ch), nn.SiLU(), nn.Conv1d(ch, m, 5, padding=2))
        self.rvq = MSRVQ(spec.lens("audio"), d, 1)


def _amp_bf16():
    return torch.autocast("cuda", dtype=torch.bfloat16, enabled=bool(IS_CUDA))


def _need_list(codes):
    assert isinstance(codes, (list, tuple)), "learned tokenizers decode from the code list of levels 0..k (residual codes): pass `codes`, not `codes[k]`"


class VQImageCodec:
    """Same interface as ImageCodec: encode, decode(codes_list, level), state, load. `cb` is the list of level codebooks once trained."""

    def __init__(self, spec, device):
        self.spec, self.device = spec, device
        self.net = ImageVQ(spec).to(device).eval()
        self.cb, self.trained = None, False

    def mark_trained(self):
        self.trained, self.cb = True, [q.embed for q in self.net.rvq.q]

    @torch.no_grad()
    def encode(self, img):
        with _amp_bf16():
            z = self.net.enc(img.float() / 127.5 - 1.0)
        return self.net.rvq(z.float())[1]

    @torch.no_grad()
    def decode(self, codes, level=None):
        _need_list(codes)
        level = self.spec.K - 1 if level is None else level
        f_hat = self.net.rvq.from_codes(codes, level)
        with _amp_bf16():
            y = self.net.dec(f_hat)
        return ((y.float() + 1.0) * 127.5).clamp(0, 255).round().to(torch.uint8)

    def state(self):
        return dict(kind="vq", trained=self.trained, sd={k: v.detach().cpu() for k, v in self.net.state_dict().items()})

    def load(self, st):
        assert st.get("kind") == "vq", "this checkpoint holds k-means codebooks but TOKENIZER='vq': use a new CKPT_DIR or TOKENIZER='kmeans'"
        self.net.load_state_dict(st["sd"])
        if st["trained"]:
            self.mark_trained()
        self.net.eval()


class VQAudioCodec(AudioCodec):
    """AudioCodec with a learned encoder/decoder over the normalised log-mel instead of k-means codebooks. The vocoder is unchanged."""

    def __init__(self, spec, device):
        super().__init__(spec, device)
        self.net = AudioVQ(spec).to(device).eval()
        self.trained = False

    def mark_trained(self):
        self.trained, self.cb = True, [q.embed for q in self.net.rvq.q]

    def norm_mel(self, wave):
        return ((self.logmel(wave) - self.mean) / self.std).transpose(1, 2)            # [B, 80, F]

    @torch.no_grad()
    def encode(self, wave):
        x = self.norm_mel(wave)                                                         # mel in fp32, outside autocast
        with _amp_bf16():
            z = self.net.enc(x)
        return self.net.rvq(z.float())[1]

    @torch.no_grad()
    def decode_mel(self, codes, level=None):
        _need_list(codes)
        level = self.spec.K - 1 if level is None else level
        f_hat = self.net.rvq.from_codes(codes, level)
        with _amp_bf16():
            y = self.net.dec(f_hat)
        return y.float().transpose(1, 2) * self.std + self.mean                         # natural-log mel [B, F, 80]

    def state(self):
        return dict(kind="vq", trained=self.trained, mean=self.mean, std=self.std,
                    sd={k: v.detach().cpu() for k, v in self.net.state_dict().items()})

    def load(self, st):
        assert st.get("kind") == "vq", "this checkpoint holds k-means codebooks but TOKENIZER='vq': use a new CKPT_DIR or TOKENIZER='kmeans'"
        self.net.load_state_dict(st["sd"]); self.mean, self.std = st["mean"], st["std"]
        if st["trained"]:
            self.mark_trained()
        self.net.eval()


# ---- training ----
def _cosine_lr(t, steps, lr, warm=100):
    return lr * min(1.0, (t + 1) / warm) * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * t / max(1, steps))))


def train_vq(net, batch_fn, loss_fn, steps, lr, tag):
    opt = torch.optim.AdamW(net.parameters(), lr=lr, betas=(0.9, 0.99), weight_decay=0.0)
    net.train(); t0, run = time.time(), None
    for t in range(steps):
        for g in opt.param_groups:
            g["lr"] = _cosine_lr(t, steps, lr)
        opt.zero_grad(set_to_none=True)
        b = batch_fn()
        with _amp_bf16():
            loss, st = loss_fn(b)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), 1.0)
        opt.step()
        run = st if run is None else {k: 0.9 * run[k] + 0.1 * st[k] for k in st}
        if t == 0 or (t + 1) % max(1, steps // 8) == 0:
            print(f"    [{tag}] step {t + 1}/{steps} " + " ".join(f"{k} {v:.4f}" for k, v in run.items()) + f" ({time.time() - t0:.0f}s)", flush=True)
    net.eval()


def _recon_loss(y, x):
    return F.mse_loss(y.float(), x) + 0.2 * F.l1_loss(y.float(), x)


def _vq_loss(net, K, target):
    """target(batch) -> (input to the encoder, reconstruction target). Full stack + one random prefix, plus commitment."""
    def f(batch):
        xin, xt = target(batch)
        z = net.enc(xin).float()
        outs, _, commit = net.rvq(z)
        y = net.dec(outs[-1])
        k = random.randrange(K - 1)
        loss = 2.0 * (_recon_loss(y, xt) + 0.5 * _recon_loss(net.dec(outs[k]), xt)) + 0.25 * commit
        return loss, dict(mse=float(F.mse_loss(y.float(), xt)), commit=float(commit))
    return f


def psnr_u8(a, b):
    mse = ((a.float() - b.float()) ** 2).flatten(1).mean(1).clamp(min=1e-8)
    return 10 * torch.log10(255.0 ** 2 / mse)


@torch.no_grad()
def eval_image_tok(codec, imgs):
    """mean PSNR (dB) vs the real image when decoding levels 0..K-1; works for the k-means and the learned codec."""
    per = None
    for i in range(0, len(imgs), 32):
        x = imgs[i:i + 32]
        codes = codec.encode(x)
        if per is None:
            per = [[] for _ in codes]                               # the k-means baseline has SPEC.K levels, the learned codec may have one more
        for k in range(len(per)):
            per[k].append(psnr_u8(codec.decode(codes, level=k), x))
    return [float(torch.cat(p).mean()) for p in per]


@torch.no_grad()
def eval_audio_tok(codec, wave_chunks):
    """log-mel RMSE (natural-log units) vs the real clip when decoding levels 0..K-1."""
    se, n = [0.0] * SPEC.K, 0
    for w in wave_chunks:
        target = codec.logmel(w)
        codes = codec.encode(w)
        for k in range(SPEC.K):
            se[k] += float(((codec.decode_mel(codes, level=k) - target) ** 2).mean()) * w.shape[0]
        n += w.shape[0]
    return [math.sqrt(s / n) for s in se]


@torch.no_grad()
def code_perplexity(codec, codes_fn):
    """Exp-entropy of the code histogram per level (256 = every code used equally)."""
    cnt = None
    for codes in codes_fn():
        if cnt is None:
            cnt = [torch.zeros(256, device=dev) for _ in codes]               # one histogram per level the codec returns (images may have one more than SPEC.K)
        for k, c in enumerate(codes):
            cnt[k] += torch.bincount(c.reshape(-1) - 1, minlength=256).float()
    out = []
    for c in cnt:
        p = c / c.sum().clamp(min=1)
        out.append(float(torch.exp(-(p[p > 0] * p[p > 0].log()).sum())))
    return out


def _img_tensor(rows):
    """raw image rows -> uint8 [N, 3, S, S] on the device (nvJPEG when the row is a JPEG)."""
    out, B = [], 64
    for i in range(0, len(rows), B):
        sub, tmp = rows[i:i + B], [None] * len(rows[i:i + B])
        jp = [j for j, r in enumerate(sub) if "jpeg" in r]
        if jp:
            d = decode_jpeg_gpu([sub[j]["jpeg"] for j in jp], SPEC.side, dev)
            for n, j in enumerate(jp):
                tmp[j] = d[n]
        for j, r in enumerate(sub):
            if tmp[j] is None:
                tmp[j] = torch.from_numpy(r["img"]).to(dev)
        out.append(torch.stack(tmp))
    return torch.cat(out)


def fit_tokenizers():
    """Stream images and clips into RAM, fit a k-means baseline and train the learned tokenizers on the same data, compare, gate."""
    t0, big = time.time(), TIER != "T0"
    n_img, n_aud = TOK_IMAGES or (8192 if big else 1024), TOK_AUDIO or (2048 if big else 512)
    steps = TOK_STEPS or {"T0": 600, "T1": 4000}.get(TIER, 8000)
    K = SPEC.K
    # cache: train once, reuse on every later run with the same spec and tokenizer settings (never reused if any of them changed)
    cache = _knob("TOK_CACHE", os.path.join(CKPT_DIR, "tokenizers.pt") if CKPT_DIR else "")
    key = repr((repr(SPEC), n_img, n_aud, steps))
    if cache and os.path.exists(cache):
        st = torch.load(cache, map_location="cpu", weights_only=False)
        if st.get("key") == key:
            codecs.load(st["codecs"])
            print(f"learned tokenizers restored from {cache} (trained earlier with the same spec and settings: no refit)")
            return
        print(f"!! {cache} was trained with other settings: retraining (and overwriting it)")
    # ---------------- images ----------------
    print(f"learned tokenizers: collecting {n_img} images and {n_aud} clips (RAM only), {steps} training steps each")
    imgs = _img_tensor(collect_rows(("image",), {"image": n_img}, 1, PARTS, WORKERS))
    n_hold = max(16, len(imgs) // 10)
    tr, hold = imgs[:-n_hold], imgs[-n_hold:]
    print(f"  images: {len(tr)} train / {len(hold)} held out ({time.time() - t0:.0f}s)")
    base = ImageCodec(SPEC, dev)                                                        # k-means on the same data = the old codec
    base.fit([sample_points(base.features(tr[i:i + 64]), 200_000 // (len(tr) // 64 + 1)) for i in range(0, len(tr), 64)])
    ps_base = eval_image_tok(base, hold); del base
    ci = codecs.image
    train_vq(ci.net, lambda: tr[torch.randint(len(tr), (32,), device=tr.device)], _vq_loss(ci.net, LAYOUTS["image"].K, _img_pair), steps, 5e-4, "image")
    ci.mark_trained()
    ps_new = eval_image_tok(ci, hold)
    ppl = code_perplexity(ci, lambda: (ci.encode(hold[i:i + 32]) for i in range(0, len(hold), 32)))
    print(f"  image PSNR by level (dB): k-means {[round(p, 2) for p in ps_base]} | learned {[round(p, 2) for p in ps_new]}")
    print(f"  learned image code perplexity per level (256 = all codes used): {[round(p) for p in ppl]}")
    del imgs, tr, hold
    # ---------------- audio ----------------
    rows = collect_rows(("audio",), {"audio": n_aud}, 1, PARTS, WORKERS)
    wav = torch.from_numpy(np.stack([r["wav"] for r in rows])).to(dev); del rows
    n_hold = max(16, len(wav) // 10)
    trw, holdw = wav[:-n_hold], wav[-n_hold:]
    chunks = lambda w: [w[i:i + 32].float() / 32768 for i in range(0, len(w), 32)]
    ca = codecs.audio
    ca.set_stats(chunks(trw))
    base = AudioCodec(SPEC, dev); base.set_stats(chunks(trw))
    base.fit([sample_points(base.features(w), 200_000 // (len(trw) // 32 + 1)) for w in chunks(trw)])
    rm_base = eval_audio_tok(base, chunks(holdw)); del base
    def a_batch():
        with torch.no_grad():
            return ca.norm_mel(trw[torch.randint(len(trw), (16,), device=trw.device)].float() / 32768)
    train_vq(ca.net, a_batch, _vq_loss(ca.net, K, lambda x: (x, x)), steps, 5e-4, "audio")
    ca.mark_trained()
    rm_new = eval_audio_tok(ca, chunks(holdw))
    print(f"  audio log-mel RMSE by level: k-means {[round(p, 3) for p in rm_base]} | learned {[round(p, 3) for p in rm_new]}")
    del wav, trw, holdw
    gc.collect(); torch.cuda.empty_cache() if IS_CUDA else None
    print(f"learned tokenizers ready ({time.time() - t0:.0f}s)")
    # ---------------- gate ----------------
    bad = []
    if ps_new[-1] < ps_base[-1] + TOK_GATE_DB:
        bad.append(f"image finest PSNR {ps_new[-1]:.2f} dB is not {TOK_GATE_DB} dB above k-means {ps_base[-1]:.2f} dB")
    if rm_new[-1] > 0.95 * rm_base[-1]:
        bad.append(f"audio finest log-mel RMSE {rm_new[-1]:.3f} is not 5% below k-means {rm_base[-1]:.3f}")
    if bad and TOK_GATE_DB > -90:
        raise RuntimeError("tokenizer gate failed: " + "; ".join(bad) + ". Train longer (TOK_STEPS / TOK_IMAGES) or set TOK_GATE_DB=-99 to continue anyway.")
    if bad:
        print("  !! gate would have failed (ignored): " + "; ".join(bad))
    if cache:                                                                           # only reached when the gate passed (or was switched off)
        os.makedirs(os.path.dirname(cache) or ".", exist_ok=True)
        torch.save(dict(key=key, codecs=codecs.state()), cache + ".tmp")
        os.replace(cache + ".tmp", cache)
        print(f"  tokenizers saved to {cache} ({os.path.getsize(cache) / 2**20:.0f} MB): later runs skip training")


def _img_pair(u8):
    x = u8.float() / 127.5 - 1.0
    if random.random() < 0.5:
        x = x.flip(-1)
    return x, x

## 3b. Feature codecs (`TOKENIZER="feat"`): frozen pretrained features, tree-quantised

The k-means and learned-VQ codecs reconstruct pixels / mel frames. These codecs instead take the codes from **frozen
pretrained features**, so the coarse rungs carry concept, not a blur (docs/codecs.md in `omni/`):

| modality | features | levels | decode |
|---|---|---|---|
| image | DINOv2 patch tokens (224 px -> 16x16 grid) | the grid, 2x2 average-pooled per level | prototype *features* only: **no pixel decoder**, so no image generation / PSNR |
| audio | DAC encoder latents (44.1 kHz), pooled to 512 frames | the frame rate pyramid | DAC decoder from prototype latents (a real waveform) |
| text | `TEXT_CODEC`: `km` data-fit byte-embedding ladder (default) / `entropy` word-aligned spans / `hash` | spans | finest level = the exact byte |

**Quantisation (`FEAT_QUANT`).** `tree`: one k-means codebook of 256 leaves on the finest features, a balanced binary tree built
post hoc over it (Tree-VQ style), and level *k* = the ancestor of the leaf at `d_min + (K-1-k)` merges up, so coarse levels
use fewer codes. `perlevel`: an independent 256-code k-means per level (the old style).
Both pool the *features* to each level's length first, then quantise, so a level is a coarsening of the same feature map.
Two fixes against the original `omni` implementation: depths are counted as merges up from the leaf everywhere (the
original mixed that with depth from the root and could not build a tree for >256 leaves), and features are **not**
centred per sample (that removed the content, and made a one-slot level constant).

In [ ]:
import hashlib

FEAT_QUANT = _knob("FEAT_QUANT", "tree")           # "tree" | "perlevel"
TEXT_CODEC = _knob("TEXT_CODEC", "km")             # "km" | "entropy" | "hash" | "vq"   (only with TOKENIZER="feat"; "vq" = learned multiscale residual VQ-VAE, the NCM paper's text tokenizer)
TEXT_VQ_STEPS = _knob("TEXT_VQ_STEPS", 3000)       # TEXT_CODEC="vq": training steps of the text residual quantiser
TEXT_VQ_ROWS = _knob("TEXT_VQ_ROWS", 40000)        # ... and the 256-byte text rows it trains on (10% held out)
DINO_MODEL = _knob("DINO_MODEL", "facebook/dinov2-base")
DAC_MODEL = _knob("DAC_MODEL", "16khz")            # 16 kHz: about 2.6x cheaper to encode than 44.1 kHz (measured); the T1 run used it
FEAT_IMAGES = _knob("FEAT_IMAGES", 6000)           # images / clips / text rows streamed once to fit the codebooks (RAM only)
FEAT_AUDIO = _knob("FEAT_AUDIO", 2000)
FEAT_TEXT = _knob("FEAT_TEXT", 8000)
FEAT_AUDIO_BATCH = _knob("FEAT_AUDIO_BATCH", 8)    # DAC runs at 44.1 kHz in fp32: a batch of 32 clips of 8 s needs many GB on a small GPU
IMAGE_CODEC = _knob("IMAGE_CODEC", "dino")         # "dino" = frozen DINOv2 features (no pixel decoder, 16x16 grid) | "vq" = the learned multi-scale residual VQ-VAE of section 4b (pixel decoder; any IMAGE_RES)
VQ_IMAGES = _knob("VQ_IMAGES", 8192)               # IMAGE_CODEC="vq": images held in RAM to train it (10% held out for its PSNR)
VQ_STEPS = _knob("VQ_STEPS", 3000)                 # ... and its training steps (3000 steps took ~5.5 min on an RTX PRO 6000 at the 32x32 grid; 4000 gave the same PSNR within 0.2 dB)
AUDIO_VQ_STEPS = _knob("AUDIO_VQ_STEPS", 4000)     # AUDIO_RES="v2": training steps of the residual quantiser on the DAC latents
assert AUDIO_RES == "v1" or TOKENIZER == "feat", "AUDIO_RES='v2' (8 residual levels on the DAC latents) is built into the feature codecs: set TOKENIZER='feat'"
assert IMAGE_RES == "v1" or (TOKENIZER == "vq" or (TOKENIZER == "feat" and IMAGE_CODEC == "vq")), \
    "IMAGE_RES='v2' (32x32 grid, two codebooks) needs a learned image codec: set TOKENIZER='feat' with IMAGE_CODEC='vq' (or TOKENIZER='vq')"

if TOKENIZER == "feat" and INSTALL:
    for _m, _p in (("transformers", "transformers"), ("dac", "descript-audio-codec")):
        try:
            __import__(_m)
        except ImportError:
            print(f"installing {_p} (feature codecs) ...", flush=True)
            subprocess.run([sys.executable, "-m", "pip", "install", "-q", _p], check=False)


class TreeHierarchy:
    """Balanced binary hierarchy over a flat codebook, built post hoc (Tree-VQ style, arXiv:2609.03641).

    Recursive median split along each node's first principal direction, so a coarse node is a prototype (the mean of the
    leaves under it) and not a bucket of unrelated vectors. "Climb c" = c merges up from the leaf: climb 0 is the leaf,
    climb `height` is the root."""

    def __init__(self, vectors):
        X = np.asarray(vectors, dtype=np.float32)
        if X.ndim != 2 or len(X) < 2 or not np.all(np.isfinite(X)):
            raise ValueError(f"tree needs a finite 2-D codebook of >= 2 vectors, got {X.shape}")
        self.X, self.V = X, len(X)
        self._vec, self._parent = [], []
        self._leaf = np.full(self.V, -1, dtype=np.int64)
        self._build(np.arange(self.V))
        self._parent = np.asarray(self._parent, dtype=np.int64)
        assert (self._leaf >= 0).all()
        self._anc = {}
        c = 0
        while len(np.unique(self.climbed(c))) > 1:
            c += 1
        self.height = c

    @staticmethod
    def _pc1(X, iters=8):
        Xc = X - X.mean(0, keepdims=True)
        v = np.ones(Xc.shape[1], dtype=np.float32)
        for _ in range(iters):
            nv = Xc.T @ (Xc @ v)
            n = float(np.linalg.norm(nv))
            if n < 1e-8:
                return np.linspace(1.0, 2.0, Xc.shape[1], dtype=np.float32)
            v = nv / n
        return v

    def _build(self, idx):
        node = len(self._vec)
        self._vec.append(self.X[idx].mean(0))
        self._parent.append(-1)
        if idx.size == 1:
            self._leaf[int(idx[0])] = node
            return node
        order = idx[np.argsort((self.X[idx] - self._vec[node]) @ self._pc1(self.X[idx]), kind="stable")]
        half = idx.size // 2
        l, r = self._build(order[:half]), self._build(order[half:])
        self._parent[l] = self._parent[r] = node
        return node

    def climbed(self, c):
        """[V] node id of every leaf's ancestor after c merges up (a leaf already at the root stays there)."""
        if c not in self._anc:
            out = self._leaf.copy()
            for _ in range(int(c)):
                par = self._parent[out]
                out = np.where(par >= 0, par, out)
            self._anc[c] = out
        return self._anc[c]

    def level_map(self, c):
        """[V] compact code 1..n of every leaf at climb c, and n."""
        anc = self.climbed(c)
        uniq = np.unique(anc)
        lut = np.zeros(len(self._vec), dtype=np.int32)
        lut[uniq] = np.arange(1, len(uniq) + 1, dtype=np.int32)
        return lut[anc], len(uniq)

    def plan(self, K, cap):
        """Climbs per level (coarse -> fine): the finest level is the smallest climb whose node count fits `cap`."""
        d_min = 0
        while len(np.unique(self.climbed(d_min))) > cap and d_min < self.height:
            d_min += 1
        climbs = [min(d_min + (K - 1 - k), self.height) for k in range(K)]
        note = f"tree: {self.V} leaves, height {self.height}, finest level at climb {d_min}"
        if d_min + K - 1 > self.height:
            note += f" | !! tree shallower than K={K}: coarse levels share a partition (climbs {climbs})"
        return climbs, note


class FeatureLadderCodec:
    """Frozen features -> K nested levels of codes 1..256. Subclasses implement `extract` (raw finest feature map)."""
    decodable = False

    def __init__(self, spec, device, mod, nd, mode):
        self.spec, self.device, self.mod, self.nd, self.mode = spec, device, mod, nd, mode
        self.lens = spec.lens(mod)
        self.trained, self.cb, self.protos, self.note = False, None, None, ""
        self.mean, self.std = 0.0, 1.0                     # the shard fingerprint reads these on the audio codec

    def extract(self, x):
        raise NotImplementedError

    @torch.no_grad()
    def features(self, x):
        """list K of raw pooled features [B, n_k, C]; level k = the finest map average-pooled to n_k slots."""
        f = self.extract(x)
        out = []
        for n in self.lens:
            p = F.adaptive_avg_pool2d(f, math.isqrt(n)).flatten(2) if self.nd == 2 else F.adaptive_avg_pool1d(f, n)
            out.append(p.transpose(1, 2).contiguous())
        return out

    @staticmethod
    def _nearest(xn, cb, chunk=65536):
        return torch.cat([(xn[i:i + chunk] @ cb.T).argmax(1) for i in range(0, len(xn), chunk)])

    @staticmethod
    def _protos(raw, a, n):
        s = torch.zeros(n, raw.shape[1], device=raw.device).index_add_(0, a, raw)
        c = torch.bincount(a, minlength=n).float()
        return s / c.clamp(min=1)[:, None], c

    @torch.no_grad()
    def fit(self, batches, sample=96):
        """batches: iterable of the codec's raw inputs (on the device). Keeps ~`sample` finest tokens per input (fp16)."""
        K, tree = self.spec.K, self.mode == "tree"
        pts = [[] for _ in range(K)]
        t0, nb = time.time(), len(batches)
        for bi, x in enumerate(batches):
            if bi == 0 or (bi + 1) % 25 == 0 or bi + 1 == nb:
                el = time.time() - t0
                print(f"      fit {self.mod}: batch {bi + 1}/{nb} | {el:.0f}s elapsed" + (f" | ~{el / bi * (nb - bi) / 60:.1f} min left" if bi else ""), flush=True)
            for k, f in enumerate(self.features(x)):
                if tree and k < K - 1:
                    continue
                f = f.reshape(-1, f.shape[-1])
                cap = sample * x.shape[0] if k == K - 1 else f.shape[0]
                if f.shape[0] > cap:
                    f = f[torch.randperm(f.shape[0], device=f.device)[:cap]]
                pts[k].append(f.half())
        unit = lambda a: F.normalize(a.float(), dim=-1)
        if tree:
            raw = torch.cat(pts[-1]).float()
            self.leaves = F.normalize(kmeans(unit(raw), 256, seed=0), dim=-1)
            self.tree = TreeHierarchy(self.leaves.cpu().numpy())
            climbs, self.note = self.tree.plan(K, 256)
            maps = [self.tree.level_map(c) for c in climbs]
            self.lut = [torch.from_numpy(m).long().to(self.device) for m, _ in maps]          # [V] -> 1..n_k
            a = self._nearest(unit(raw), self.leaves)
            leaf_mean, leaf_cnt = self._protos(raw, a, len(self.leaves))
            leaf_sum = leaf_mean * leaf_cnt[:, None]
            self.protos = []
            for k in range(K):
                idx, n = self.lut[k] - 1, maps[k][1]
                s = torch.zeros(n, raw.shape[1], device=raw.device).index_add_(0, idx, leaf_sum)
                c = torch.zeros(n, device=raw.device).index_add_(0, idx, leaf_cnt)
                self.protos.append(s / c.clamp(min=1)[:, None])
            self.cb = [F.normalize(p, dim=-1) for p in self.protos]
            self.note += f" | nodes per level {[int(m[1]) for m in maps]}"
        else:
            self.cbs, self.protos = [], []
            for k in range(K):
                raw = torch.cat(pts[k]).float()
                cb = F.normalize(kmeans(unit(raw), 256, seed=k), dim=-1)
                p, _ = self._protos(raw, self._nearest(unit(raw), cb), 256)
                self.cbs.append(cb); self.protos.append(p)
            self.cb = self.cbs
            self.note = "per-level k-means, 256 codes each"
        self.trained = True

    @torch.no_grad()
    def encode(self, x):
        out = []
        for k, f in enumerate(self.features(x)):
            B, n, C = f.shape
            xn = F.normalize(f.reshape(-1, C), dim=-1)
            if self.mode == "tree":
                out.append(self.lut[k][self._nearest(xn, self.leaves)].view(B, n))
            else:
                out.append((self._nearest(xn, self.cbs[k]) + 1).view(B, n))
        return out

    def reconstruct(self, codes, level):
        """prototype feature of every code of one level: [B, n_k, C] (what the codes keep of the features)."""
        return self.protos[level][(codes - 1).clamp(0, self.protos[level].shape[0] - 1)]     # the model may sample a code this level never uses

    def state(self):
        if not self.trained:
            return None
        d = dict(kind="feat", mode=self.mode, protos=[p.cpu() for p in self.protos], note=self.note)
        if self.mode == "tree":
            d.update(leaves=self.leaves.cpu(), lut=[l.cpu() for l in self.lut])
        else:
            d.update(cbs=[c.cpu() for c in self.cbs])
        return d

    def load(self, st):
        assert st.get("kind") == "feat" and st["mode"] == self.mode, \
            f"codec state is {st.get('kind')}/{st.get('mode')} but TOKENIZER='feat' FEAT_QUANT='{self.mode}': use a new CKPT_DIR"
        self.protos = [p.to(self.device) for p in st["protos"]]
        if self.mode == "tree":
            self.leaves, self.lut = st["leaves"].to(self.device), [l.to(self.device) for l in st["lut"]]
            self.cb = [F.normalize(p, dim=-1) for p in self.protos]
        else:
            self.cbs = [c.to(self.device) for c in st["cbs"]]
            self.cb = self.cbs
        self.note, self.trained = st.get("note", ""), True


class DinoImageCodec(FeatureLadderCodec):
    """DINOv2 patch tokens of the image resized to 224 px: a 16x16 grid, the same grid as the finest image level."""
    decodable = False

    def __init__(self, spec, device, mode):
        super().__init__(spec, device, "image", 2, mode)
        from transformers import AutoModel
        self.net = AutoModel.from_pretrained(DINO_MODEL).to(device).eval()
        self.n_reg = int(getattr(self.net.config, "num_register_tokens", 0))
        self.m = torch.tensor([0.485, 0.456, 0.406], device=device).view(1, 3, 1, 1)
        self.s = torch.tensor([0.229, 0.224, 0.225], device=device).view(1, 3, 1, 1)

    @torch.no_grad()
    def extract(self, img):
        hs = []
        for i in range(0, img.shape[0], 32):                                 # chunks bound the activation memory next to a running training step
            x = F.interpolate(img[i:i + 32].float() / 255.0, size=(224, 224), mode="bilinear", antialias=True, align_corners=False)
            with torch.autocast("cuda", dtype=torch.bfloat16, enabled=bool(IS_CUDA)):
                hs.append(self.net(pixel_values=(x - self.m) / self.s).last_hidden_state[:, 1 + self.n_reg:].float())   # drop CLS (+ register tokens)
        h = torch.cat(hs)
        g = math.isqrt(h.shape[1])
        assert g * g == h.shape[1], f"DINO returned {h.shape[1]} patch tokens: not a square grid"
        return h.transpose(1, 2).reshape(h.shape[0], -1, g, g)

    def decode(self, codes, level=None):
        raise NotImplementedError("DINO feature codes have no pixel decoder: image generation / PSNR are not available with this codec")


class DacAudioFeatureCodec(FeatureLadderCodec):
    """DAC (44.1 kHz) encoder latents, pooled to the notebook's frame count; decoded by the DAC decoder from prototype latents."""
    decodable = True
    SR = 16000 if DAC_MODEL == "16khz" else 44100

    def __init__(self, spec, device, mode):
        super().__init__(spec, device, "audio", 1, mode)
        import dac
        self.net = dac.DAC.load(dac.utils.download(model_type=DAC_MODEL)).to(device).eval()
        self.hop = int(getattr(self.net, "hop_length", 512))
        self.n_in = round(spec.wave_len * self.SR / spec.sr)
        self.n_in += (-self.n_in) % self.hop
        self.T = self.n_in // self.hop

    @torch.no_grad()
    def extract(self, wave):
        outs = []
        for i in range(0, wave.shape[0], FEAT_AUDIO_BATCH):                  # a whole micro-batch at 44.1 kHz in fp32 needs tens of GB: encode in small chunks
            x = wave[i:i + FEAT_AUDIO_BATCH].float()
            x = F.interpolate(x[:, None], size=round(x.shape[1] * self.SR / self.spec.sr), mode="linear", align_corners=False)
            z = self.net.encoder(F.pad(x, (0, self.n_in - x.shape[-1])))     # [b, 1024, T]
            outs.append(F.adaptive_avg_pool1d(z.float(), self.spec.audio))   # frames at the notebook's rate
        return torch.cat(outs)

    @torch.no_grad()
    def decode(self, codes, level=None, **_):
        level = self.spec.K - 1 if level is None else level
        if isinstance(codes, (list, tuple)):
            codes = codes[level]
        return self._wave(self.reconstruct(codes, level).transpose(1, 2))    # [B, C, n_k]

    @torch.no_grad()
    def _wave(self, z):
        """DAC latents [B, C, frames] -> waveform [B, wave_len] at the notebook's sample rate."""
        z = F.interpolate(z, size=self.T, mode="linear", align_corners=False)
        w = self.net.decoder(z)                                              # [B, 1, n_in] at 44.1 kHz
        w = F.interpolate(w, size=round(w.shape[-1] * self.spec.sr / self.SR), mode="linear", align_corners=False)[:, 0]
        return w[:, :self.spec.wave_len]


class AudioAdapterVQ(nn.Module):
    """Small conv encoder / decoder around a multi-scale residual quantiser (MSRVQ of section 4b): DAC latents -> d-dim -> 8-bit codes per level -> back."""

    def __init__(self, c_in, spec, d=64, ch=256):
        super().__init__()
        self.enc = nn.Sequential(nn.Conv1d(c_in, ch, 5, padding=2), _Res1d(ch, 1), _Res1d(ch, 2), _gn(ch), nn.SiLU(), nn.Conv1d(ch, d, 1))
        self.dec = nn.Sequential(nn.Conv1d(d, ch, 5, padding=2), _Res1d(ch, 1), _Res1d(ch, 2), _gn(ch), nn.SiLU(), nn.Conv1d(ch, c_in, 5, padding=2))
        self.rvq = MSRVQ(spec.lens("audio"), d, 1)


class DacResidualAudioCodec(DacAudioFeatureCodec):
    """AUDIO_RES="v2": DAC encoder latents (frozen, as in DacAudioFeatureCodec) quantised by a LEARNED residual quantiser with 8-bit codebooks.

    The ladder is the usual 5 levels plus `audio_extra` residual levels at the finest frame rate, each coding what the levels before it
    missed, so decoding level k needs the codes of all levels <= k. The frozen DAC decoder turns the rebuilt latents into audio."""
    decodable = True

    def __init__(self, spec, device, mode):
        super().__init__(spec, device, mode)
        self.K_aud = len(spec.lens("audio"))
        self.adapter = AudioAdapterVQ(int(getattr(self.net, "latent_dim", 1024)), spec).to(device).eval()
        self.mu = self.sd = None

    def fit(self, batches):                                  # not under no_grad: the quantiser is trained here
        with torch.no_grad():
            zs = [self.extract(w).half() for w in batches]
            z = torch.cat(zs); del zs
            ref = z[:256].float()
            self.mu, self.sd = ref.mean((0, 2), keepdim=True), ref.std((0, 2), keepdim=True).clamp(min=1e-4)
        n_hold = max(16, len(z) // 10)
        tr, hold = z[:-n_hold], z[-n_hold:]
        print(f"      audio residual quantiser: {len(tr)} train / {len(hold)} held-out clips, {AUDIO_VQ_STEPS} steps, ladder {self.lens}", flush=True)
        batch = lambda: (tr[torch.randint(len(tr), (16,), device=tr.device)].float() - self.mu) / self.sd
        train_vq(self.adapter, batch, _vq_loss(self.adapter, self.K_aud, lambda x: (x, x)), AUDIO_VQ_STEPS, 5e-4, "audio")
        self.cb, self.trained = [q.embed for q in self.adapter.rvq.q], True
        with torch.no_grad():
            x = (hold[:32].float() - self.mu) / self.sd
            with _amp_bf16():
                codes = self.adapter.rvq(self.adapter.enc(x).float())[1]
            r2 = []
            for k in range(self.K_aud):
                with _amp_bf16():
                    y = self.adapter.dec(self.adapter.rvq.from_codes(codes, k))
                r2.append(1 - float(((y.float() - x) ** 2).mean() / x.var()))
        self.note = f"latent R^2 by level {[round(r, 3) for r in r2]}"
        del z, tr, hold

    @torch.no_grad()
    def encode(self, wave):
        x = (self.extract(wave) - self.mu) / self.sd
        with _amp_bf16():
            z = self.adapter.enc(x)
        return self.adapter.rvq(z.float())[1]

    @torch.no_grad()
    def decode(self, codes, level=None, **_):
        level = self.K_aud - 1 if level is None else level
        with _amp_bf16():
            y = self.adapter.dec(self.adapter.rvq.from_codes(codes, level))
        return self._wave(y.float() * self.sd + self.mu)

    def state(self):
        if not self.trained:
            return None
        return dict(kind="feat_res", sd={k: v.detach().cpu() for k, v in self.adapter.state_dict().items()}, mu=self.mu.cpu(), std=self.sd.cpu(), note=self.note)

    def load(self, st):
        assert st.get("kind") == "feat_res", "this checkpoint holds another audio codec: use a new CKPT_DIR"
        self.adapter.load_state_dict(st["sd"]); self.adapter.eval()
        self.mu, self.sd = st["mu"].to(self.device), st["std"].to(self.device)
        self.cb, self.trained, self.note = [q.embed for q in self.adapter.rvq.q], True, st.get("note", "")


# ---- text ----
class TextKMCodec:
    """Data-fit ladder: byte embeddings from the SVD of the byte-PPMI, span features = their average, k-means per level.
    The finest level is the exact byte, so decoding is lossless."""
    D = 16
    decode = staticmethod(TextCodec.decode)

    def __init__(self, spec, device):
        self.spec, self.device, self.emb, self.cb, self.trained = spec, device, None, None, False

    @torch.no_grad()
    def _fit_emb(self, b):
        b = b.long()
        C = torch.zeros(256 * 256, device=b.device)
        for d in (1, 2, 3, 4):
            C += torch.bincount(b[:, :-d].reshape(-1) * 256 + b[:, d:].reshape(-1), minlength=256 * 256).float()
        C = C.view(256, 256) + C.view(256, 256).T
        tot = C.sum()
        pr = C.sum(1, keepdim=True) / tot
        U, S, _ = torch.linalg.svd(torch.log((C / tot + 1e-12) / (pr @ pr.T + 1e-12)).clamp(min=0))
        e = U[:, :self.D] * S[:self.D].sqrt()
        return e / e.std()

    def features(self, b):
        K, R = self.spec.K, self.spec.R_of("text")
        e = self.emb[b.long()].transpose(1, 2)
        return [F.avg_pool1d(e, R ** (K - 1 - j)).transpose(1, 2).contiguous() for j in range(K - 1)]

    @torch.no_grad()
    def fit(self, b):
        self.emb = self._fit_emb(b)
        self.cb = [kmeans(f.reshape(-1, self.D), 256, seed=j) for j, f in enumerate(self.features(b))]
        self.trained = True

    @torch.no_grad()
    def encode(self, b):
        codes = [vq(f, self.cb[j]) for j, f in enumerate(self.features(b))]
        codes.append(b.long() + 1)
        return codes

    def state(self):
        return None if self.cb is None else dict(kind="text_km", emb=self.emb.cpu(), cb=[c.cpu() for c in self.cb])

    def load(self, st):
        assert st.get("kind") == "text_km", "the checkpoint holds another text codec: use a new CKPT_DIR"
        self.emb, self.cb, self.trained = st["emb"].to(self.device), [c.to(self.device) for c in st["cb"]], True


class TextEntropyCodec:
    """omni's word-aligned span codec: coarse levels cut at word starts, spent first where the next 4-gram is novel, each span
    coded by a hash of its bytes. Deterministic and corpus-free. Slower than the others (a python loop per row)."""
    NGRAM = 4
    decode = staticmethod(TextCodec.decode)

    def __init__(self, spec, device, seed=1):
        self.spec, self.device, self.seed, self.trained = spec, device, int(seed), True

    def _repeat(self, raw):
        seen, out = {}, np.zeros(len(raw), dtype=np.int32)
        for i in range(len(raw)):
            w = bytes(raw[i:i + self.NGRAM])
            out[i] = seen.get(w, 0)
            seen[w] = out[i] + 1
        return out

    @staticmethod
    def _candidates(raw):
        n = len(raw)
        starts = [i for i in range(1, n) if raw[i] not in b" \t\n\r" and raw[i - 1] in b" \t\n\r"]
        return starts or list(range(1, n))

    def _edges(self, raw, l, cand, rep):
        n = len(raw)
        if l <= 1:
            return [0, n]
        if n <= 1:
            return [0] * (l + 1)
        m = l - 1
        if len(cand) > m:
            order = np.lexsort((np.asarray(cand), rep[cand]))
            cand = sorted(int(cand[j]) for j in order[:m])
        elif len(cand) < m:
            fill = np.linspace(1, n - 1, m).round().astype(int)
            cand = sorted(set(cand) | {int(p) for p in fill if 1 <= p < n})[:m]
        return [0] + list(cand) + [n]

    def _codes(self, raw, l, cand, rep):
        edges = self._edges(raw, l, cand, rep)
        if len(edges) - 1 != l:
            edges = [round(i * len(raw) / l) for i in range(l + 1)]
        out = np.empty(l, dtype=np.int64)
        for i in range(l):
            h = hashlib.blake2b(bytes(raw[edges[i]:edges[i + 1]]), digest_size=8, salt=self.seed.to_bytes(8, "little")).digest()
            out[i] = int.from_bytes(h, "little") % 256 + 1
        return out

    @torch.no_grad()
    def encode(self, b):
        rows, lens, K = b.cpu().numpy(), self.spec.lens("text"), self.spec.K
        out = [np.zeros((len(rows), n), dtype=np.int64) for n in lens]
        for i, r in enumerate(rows):
            raw = r.tobytes().rstrip(b"\x00")
            cand, rep = self._candidates(raw), self._repeat(raw)
            for k in range(K - 1):
                out[k][i] = self._codes(raw, lens[k], cand, rep)
        out[K - 1] = rows.astype(np.int64) + 1
        return [torch.from_numpy(o).to(self.device) for o in out]

    def state(self):
        return None

    def load(self, st):
        pass


class TextVQNet(nn.Module):
    """Bytes [B, L] -> L/R latent slots -> residual 8-bit codes for every level above the finest -> byte logits [B, 256, L]."""

    def __init__(self, spec, d=64, ch=256):
        super().__init__()
        r = spec.R_of("text")
        self.emb = nn.Embedding(256, 128)
        self.enc = nn.Sequential(nn.Conv1d(128, ch, 5, padding=2), _Res1d(ch, 1), nn.Conv1d(ch, ch, r, stride=r), _Res1d(ch, 1), _Res1d(ch, 2),
                                 _gn(ch), nn.SiLU(), nn.Conv1d(ch, d, 1))
        self.dec = nn.Sequential(nn.Conv1d(d, ch, 5, padding=2), _Res1d(ch, 1), _Res1d(ch, 2), nn.Upsample(scale_factor=r, mode="nearest"),
                                 nn.Conv1d(ch, ch, 5, padding=2), _Res1d(ch, 1), _gn(ch), nn.SiLU(), nn.Conv1d(ch, 256, 1))
        self.rvq = MSRVQ(spec.lens("text")[:-1], d, 1)             # every level except the finest, which is the exact byte

    def encode(self, b):
        return self.enc(self.emb(b.long()).transpose(1, 2))


class TextVQCodec:
    """TEXT_CODEC="vq": the NCM paper's idea for text (a multiscale residual VQ-VAE, trained with nested dropout), adapted to this ladder.

    Levels 0..K-2 are learned residual 8-bit codes, trained so a decoder can rebuild the bytes from the codes of the first k levels (a random
    prefix is trained as well as the full stack). The finest level stays the exact byte, so decoding is lossless."""
    decode = staticmethod(TextCodec.decode)

    def __init__(self, spec, device):
        self.spec, self.device, self.trained, self.note = spec, device, False, ""
        self.net = TextVQNet(spec).to(device).eval()

    def fit(self, b):
        b = b.long()
        n_hold = max(64, len(b) // 10)
        tr, hold = b[:-n_hold], b[-n_hold:]
        net = self.net

        def loss_fn(x):
            z = net.encode(x).float()
            outs, _, commit = net.rvq(z)
            ce = lambda f: F.cross_entropy(net.dec(f).float(), x)
            full = ce(outs[-1])
            k = random.randrange(len(outs) - 1) if len(outs) > 1 else 0
            return full + 0.5 * ce(outs[k]) + 0.25 * commit, dict(ce=float(full), commit=float(commit))

        print(f"      text residual quantiser: {len(tr)} train / {len(hold)} held-out rows, {TEXT_VQ_STEPS} steps, learned levels {self.spec.lens('text')[:-1]} + exact byte", flush=True)
        train_vq(net, lambda: tr[torch.randint(len(tr), (64,), device=tr.device)], loss_fn, TEXT_VQ_STEPS, 5e-4, "text")
        self.trained = True
        with torch.no_grad():
            x = hold[:256]
            with _amp_bf16():
                codes = net.rvq(net.encode(x).float())[1]
            acc, pad = [], x == 0
            for k in range(len(codes)):
                with _amp_bf16():
                    pred = net.dec(net.rvq.from_codes(codes, k)).argmax(1)
                acc.append(float(((pred == x) & ~pad).sum() / (~pad).sum().clamp(min=1)))
        self.note = f"byte accuracy rebuilt from the codes of levels 0..k: {[round(a, 3) for a in acc]}"

    @torch.no_grad()
    def encode(self, b):
        with _amp_bf16():
            z = self.net.encode(b)
        return list(self.net.rvq(z.float())[1]) + [b.long() + 1]

    def state(self):
        return dict(kind="text_vq", sd={k: v.detach().cpu() for k, v in self.net.state_dict().items()}, note=self.note) if self.trained else None

    def load(self, st):
        assert st.get("kind") == "text_vq", "the checkpoint holds another text codec: use a new CKPT_DIR"
        self.net.load_state_dict(st["sd"]); self.net.eval()
        self.trained, self.note = True, st.get("note", "")


def make_text_codec(spec, device):
    if TOKENIZER != "feat" or TEXT_CODEC == "hash":
        return TextCodec(spec, device)
    return {"km": TextKMCodec, "entropy": TextEntropyCodec, "vq": TextVQCodec}[TEXT_CODEC](spec, device)


def fit_image_vq(t0):
    """IMAGE_CODEC="vq": train the multi-scale residual image VQ-VAE (section 4b) on streamed images; the DAC audio and text codecs are fitted next to it."""
    path = os.path.join(CKPT_DIR, "image_vq.pt") if CKPT_DIR else ""          # saved right after training: a later failure (audio fit) must not cost 5-8 min again
    key = repr((repr(SPEC), VQ_IMAGES, VQ_STEPS))
    if path and os.path.exists(path):
        st = torch.load(path, map_location="cpu", weights_only=False)
        if st.get("key") == key:
            codecs.image.load(st["state"])
            print(f"  image VQ restored from {path} (trained earlier with the same settings: no retrain)")
            return
    imgs = _img_tensor(collect_rows(("image",), {"image": VQ_IMAGES}, 1, PARTS, WORKERS))
    n_hold = max(16, len(imgs) // 10)
    tr, hold = imgs[:-n_hold], imgs[-n_hold:]
    ci, K_img = codecs.image, LAYOUTS["image"].K
    print(f"  image VQ: {len(tr)} train / {len(hold)} held-out images, {VQ_STEPS} steps, ladder {LAYOUTS['image'].lens} ({time.time() - t0:.0f}s)", flush=True)
    train_vq(ci.net, lambda: tr[torch.randint(len(tr), (32,), device=tr.device)], _vq_loss(ci.net, K_img, _img_pair), VQ_STEPS, 5e-4, "image")
    ci.mark_trained()
    ps = eval_image_tok(ci, hold)
    ppl = code_perplexity(ci, lambda: (ci.encode(hold[i:i + 32]) for i in range(0, len(hold), 32)))
    print(f"  image PSNR by level (dB): {[round(p, 2) for p in ps]} | code perplexity per level (256 = all codes used): {[round(p) for p in ppl]}")
    if path:
        os.makedirs(os.path.dirname(path), exist_ok=True)
        torch.save(dict(key=key, state=ci.state()), path + ".tmp"); os.replace(path + ".tmp", path)
        print(f"  image VQ saved to {path}")
    del imgs, tr, hold


def fit_feature_codecs():
    """Stream a sample into RAM, fit the image / audio / text codecs on it, cache the result (no model compute yet)."""
    t0 = time.time()
    cache = _knob("TOK_CACHE", os.path.join(CKPT_DIR, "feat_codecs.pt") if CKPT_DIR else "")
    key = repr((repr(SPEC), TEXT_CODEC, FEAT_QUANT, FEAT_IMAGES, FEAT_AUDIO, FEAT_TEXT, DINO_MODEL, DAC_MODEL, IMAGE_CODEC, VQ_IMAGES, VQ_STEPS, AUDIO_RES, AUDIO_VQ_STEPS, TEXT_VQ_STEPS, TEXT_VQ_ROWS))
    if cache and os.path.exists(cache):
        st = torch.load(cache, map_location="cpu", weights_only=False)
        if st.get("key") == key:
            codecs.load(st["codecs"])
            print(f"feature codecs restored from {cache} (fitted earlier with the same settings: no refit)")
            return
        print(f"!! {cache} was fitted with other settings: refitting (and overwriting it)")
    print(f"feature codecs: {'training the image VQ on ' + str(VQ_IMAGES) if IMAGE_CODEC == 'vq' else 'streaming ' + str(FEAT_IMAGES)} images, {FEAT_AUDIO} clips, {FEAT_TEXT} text rows (RAM only)")
    if IMAGE_CODEC == "vq":
        fit_image_vq(t0)
    else:
        imgs = _img_tensor(collect_rows(("image",), {"image": FEAT_IMAGES}, 1, PARTS, WORKERS))
        codecs.image.fit([imgs[i:i + 64] for i in range(0, len(imgs), 64)])
        print(f"  image ({DINO_MODEL}, {FEAT_QUANT}): {codecs.image.note} ({time.time() - t0:.0f}s)")
        del imgs
    rows = collect_rows(("audio",), {"audio": FEAT_AUDIO}, 1, PARTS, WORKERS)
    wav = torch.from_numpy(np.stack([r["wav"] for r in rows])).to(dev); del rows
    codecs.audio.fit([wav[i:i + FEAT_AUDIO_BATCH].float() / 32768 for i in range(0, len(wav), FEAT_AUDIO_BATCH)])
    print(f"  audio (DAC {DAC_MODEL}, {FEAT_QUANT}): {codecs.audio.note} ({time.time() - t0:.0f}s)")
    del wav
    if hasattr(codecs.text, "fit"):
        n_text = TEXT_VQ_ROWS if TEXT_CODEC == "vq" else FEAT_TEXT
        rows = collect_rows(("text0",), {"text0": n_text}, 1, PARTS, WORKERS)
        codecs.text.fit(torch.from_numpy(np.stack([r["tgt"] for r in rows])).to(dev)); del rows
    print(f"  text ({type(codecs.text).__name__}) ready" + (f": {codecs.text.note}" if getattr(codecs.text, "note", "") else "") + f" ({time.time() - t0:.0f}s)")
    gc.collect(); torch.cuda.empty_cache() if IS_CUDA else None
    if cache:
        os.makedirs(os.path.dirname(cache) or ".", exist_ok=True)
        torch.save(dict(key=key, codecs=codecs.state()), cache + ".tmp")
        os.replace(cache + ".tmp", cache)
        print(f"  codecs saved to {cache}: later runs skip the fit")

## 5. Shards: encode once, keep exactly what the model reads (the shard maker is `encode_shards()` at the end of the next cell)

The model never sees raw rows, only the **codes**: one `uint8` per code (code - 1, so 1..256 fits in a byte),
stored per row as `[ prefix | level 0 | level 1 | ... | level K-1 ]`. That is ~300x smaller than the raw
images and audio (a 341-byte row instead of a ~100 KB JPEG), so 2B codes is ~2 GB on disk.

| `SHARD_MODE` | what happens |
|---|---|
| `encode` | **no model, no training**: stream the whole dataset (`SHARD_CODES_B` / `TOKENS` / the tier), encode it on the GPU and store it, one pass per row type, resumable. Run the notebook once like this, then switch to `read` |
| `record` (default) | train from the live stream as before, and **write every encoded micro-batch to `SHARD_DIR`** |
| `read` | train **only** from the stored shards (no Hub streaming for training; exact, seeded, resumable data order) |
| `off` | the previous behaviour: stream, store nothing |

Shards are only valid for the **codebooks they were encoded with**, so the codebooks are saved next to them
(`codecs.pt`) with a fingerprint. A later run loads those codebooks instead of refitting, and refuses to mix
shards with different ones. The same directory also keeps the **codebook-fit samples** (`fit/image.npy`, `fit/audio.npy`)
and the **held-out set** (`val/<row type>.npy`, from partition 0, so it never overlaps the shards), so a new session
in the same runtime/disk does not re-stream them.

In [ ]:
import hashlib
from glob import glob as _glob

SHARD_MB = _knob("SHARD_MB", 64)                                       # target size of one shard file
ROW_W = {r: LAYOUTS[r].cond + LAYOUTS[r].Tt for r in ROW_TYPES}        # stored bytes per row: prefix + every level
USE_SHARDS = bool(SHARD_DIR) and SHARD_MODE in ("record", "read", "encode")


def codec_fingerprint(cd):
    """Hash of everything the codes depend on: both codebooks, the mel statistics and the ladder spec."""
    h = hashlib.sha1()
    for cb in (cd.image.cb if cd.image.cb is not None else []) + (cd.audio.cb if cd.audio.cb is not None else []):
        h.update(cb.detach().float().cpu().numpy().tobytes())
    h.update(repr((round(cd.audio.mean, 6), round(cd.audio.std, 6))).encode())
    h.update(repr(SPEC).encode())
    return h.hexdigest()[:16]


def pack_rows(mb):
    """encoded micro-batch (long codes 1..256 on the device) -> uint8 numpy [B, ROW_W]."""
    parts = ([mb["cond"] - 1] if mb["cond"] is not None else []) + [c - 1 for c in mb["codes"]]
    return torch.cat(parts, 1).to(torch.uint8).cpu().numpy()


def unpack_rows(rtype, rows_u8, device):
    """uint8 numpy [B, ROW_W] -> (cond long [B, Lc] | None, codes: K long tensors), on `device`."""
    lay = LAYOUTS[rtype]
    t = torch.from_numpy(np.ascontiguousarray(rows_u8))
    if device.type == "cuda":
        t = t.pin_memory().to(device, non_blocking=True)
    cond = t[:, :lay.cond].long() + 1 if lay.cond else None
    codes, o = [], lay.cond
    for n in lay.lens:
        codes.append(t[:, o:o + n].long() + 1)
        o += n
    return cond, codes


def shard_prepare(root, cd):
    """Create/validate the shard directory against the codebooks that are about to be used."""
    os.makedirs(root, exist_ok=True)
    meta_p, fp = os.path.join(root, "meta.json"), codec_fingerprint(cd)
    if os.path.exists(meta_p):
        old = json.load(open(meta_p))["fingerprint"]
        if old != fp:
            raise RuntimeError(f"shards in {root} were encoded with other codebooks ({old} != {fp}). Use a new SHARD_DIR, "
                               f"delete this one, or resume from the checkpoint/shards that hold the matching codebooks.")
    elif SHARD_MODE in ("record", "encode"):
        json.dump(dict(fingerprint=fp, spec=repr(SPEC), preset=PRESET, row_w=ROW_W), open(meta_p, "w"))
        torch.save(cd.state(), os.path.join(root, "codecs.pt"))
    return fp


def shard_stats(root):
    n_files, n_bytes, rows = 0, 0, {}
    for r in ROW_TYPES:
        fs = _glob(os.path.join(root, r, "*.npy"))
        n_files += len(fs)
        n_bytes += sum(os.path.getsize(f) for f in fs)
        rows[r] = sum(np.load(f, mmap_mode="r").shape[0] for f in fs)
    return n_files, n_bytes, rows


class ShardWriter:
    """Appends encoded micro-batches to per-row-type shard files. Called from the prefetch thread."""

    def __init__(self, root):
        self.root, self.lock = root, threading.Lock()
        self.buf = {r: [] for r in ROW_TYPES}
        self.nbuf = {r: 0 for r in ROW_TYPES}
        self.written = {r: 0 for r in ROW_TYPES}
        self.idx = {}
        for r in ROW_TYPES:
            os.makedirs(os.path.join(root, r), exist_ok=True)
            have = [int(os.path.basename(f)[:-4]) for f in _glob(os.path.join(root, r, "*.npy"))]
            self.idx[r] = max(have) + 1 if have else 0

    def add(self, mb):
        r, a = mb["rtype"], pack_rows(mb)
        with self.lock:
            self.buf[r].append(a)
            self.nbuf[r] += len(a)
            if self.nbuf[r] * ROW_W[r] >= SHARD_MB * 2 ** 20:
                self._flush(r)

    def _flush(self, r):
        if not self.buf[r]:
            return
        arr = np.concatenate(self.buf[r])
        self.buf[r], self.nbuf[r] = [], 0
        path = os.path.join(self.root, r, f"{self.idx[r]:06d}.npy")
        self.idx[r] += 1
        with open(path + ".tmp", "wb") as f:
            np.save(f, arr)
        os.replace(path + ".tmp", path)                     # a crash never leaves a half-written shard
        self.written[r] += len(arr)

    def flush_all(self):
        with self.lock:
            for r in ROW_TYPES:
                self._flush(r)

    def close(self):
        self.flush_all()


class ShardFeeder(Prefetcher):
    """Same interface as `Prefetcher` (get / close), fed from stored shards instead of the stream.

    Micro-batch number m is drawn with a generator seeded by (SEED, m), so a resumed run (start at
    step * ACCUM) continues the exact same data order. Rows are sampled uniformly with replacement
    over all stored rows of the chosen row type, so a run longer than the data simply makes more epochs."""

    def __init__(self, root, start_micro=0, depth=PREFETCH, seed=0, micro_left=0):
        self.q = queue.Queue(maxsize=depth)
        self.stop, self.err, self.rows_in = threading.Event(), None, 0
        self.stream = torch.cuda.Stream() if IS_CUDA else None
        self.seed, self.m0 = seed, start_micro
        self.arr = {r: [np.load(f, mmap_mode="r") for f in sorted(_glob(os.path.join(root, r, "*.npy")))] for r in ROW_TYPES}
        self.cum = {r: np.cumsum([len(a) for a in self.arr[r]]) for r in ROW_TYPES}
        self.rows = {r: int(self.cum[r][-1]) if len(self.arr[r]) else 0 for r in ROW_TYPES}
        short = [r for r in ROW_TYPES if self.rows[r] < 1]
        if short:
            raise RuntimeError(f"no stored rows for {short} in {root}: {self.rows}. Run with SHARD_MODE='record' first.")
        p = np.array([TYPE_P[r] for r in ROW_TYPES])
        self.p = p / p.sum()
        epochs = {r: micro_left * self.p[i] * MICRO_B[r] / self.rows[r] for i, r in enumerate(ROW_TYPES)}
        print(f"reading shards: rows {self.rows} | this run draws about { {r: round(float(e), 2) for r, e in epochs.items()} } epochs of each", flush=True)
        if max(epochs.values()) > 4:
            print("  !! more than 4 epochs over some row type: record more data (longer live run) before reading", flush=True)
        self.t = threading.Thread(target=self._run, daemon=True)
        self.t.start()

    def _batch(self, m):
        rng = np.random.default_rng([self.seed, m])
        r = ROW_TYPES[int(rng.choice(len(ROW_TYPES), p=self.p))]
        idx = np.sort(rng.integers(0, self.rows[r], size=MICRO_B[r]))
        fi = np.searchsorted(self.cum[r], idx, side="right")
        base = np.concatenate([[0], self.cum[r]])[fi]
        rows = np.concatenate([self.arr[r][f][(idx - base)[fi == f]] for f in np.unique(fi)])
        return r, rows

    def _run(self):
        try:
            m = self.m0
            while not self.stop.is_set():
                r, rows = self._batch(m)
                m += 1
                if self.stream is not None:
                    with torch.cuda.stream(self.stream):
                        cond, codes = unpack_rows(r, rows, dev)
                    self.stream.synchronize()
                else:
                    cond, codes = unpack_rows(r, rows, dev)
                mb = dict(rtype=r, cond=cond, codes=codes, n_live=len(rows) * LAYOUTS[r].Tt, src=["shard"] * len(rows))
                while not self.stop.is_set():
                    try:
                        self.q.put(mb, timeout=1.0)
                        break
                    except queue.Full:
                        pass
        except Exception as e:
            self.err = e


# ---- encode-only: build the whole dataset as shards before any training ----
SHARD_CODES_B = _knob("SHARD_CODES_B", 0.0)            # billions of live codes to store; 0 = TOKENS, else the tier
ENCODE_SHARE = {"text0": CODE_SHARE["text"] * (1 - TEXT_PROMPT_FRAC), "text": CODE_SHARE["text"] * TEXT_PROMPT_FRAC,
                "image": CODE_SHARE["image"], "audio": CODE_SHARE["audio"]}      # share of live codes per row type


def encode_shards(kinds=ROW_TYPES):
    """Stream, encode and store the dataset, one row type at a time. No model step. Resumable: what is already
    on disk is counted, and only the missing codes are streamed."""
    total = SHARD_CODES_B or TOKENS or {"T0": 0.02, "T1": 2.0, "T2": 20.0, "T3": 50.0}[TIER]
    os.makedirs(SHARD_DIR, exist_ok=True)
    free = shutil.disk_usage(SHARD_DIR).free / 1e9
    need_gb = total * 1.2
    print(f"encoding {total:.3f}B live codes (~{need_gb:.1f} GB on disk, {free:.0f} GB free) into {SHARD_DIR}")
    print("  per row type:", {r: f"{total * ENCODE_SHARE[r]:.3f}B" for r in ROW_TYPES}, flush=True)
    if need_gb > 0.95 * free:
        raise RuntimeError(f"not enough disk for the shards: need ~{need_gb:.0f} GB, {free:.0f} GB free. Point SHARD_DIR at Drive or lower SHARD_CODES_B.")
    writer = ShardWriter(SHARD_DIR)
    _, _, have_rows = shard_stats(SHARD_DIR)
    for r in kinds:
        quota, have = total * 1e9 * ENCODE_SHARE[r], have_rows[r] * LAYOUTS[r].Tt
        if have >= quota:
            print(f"  {r:6s} already stored: {have / 1e9:.3f}B of {quota / 1e9:.3f}B codes, skipped", flush=True)
            continue
        stream = RowStream(MixDataset(SPEC, kinds=(r,), part0=1, nparts=PARTS, seed=SEED + 10007 + have_rows[r] % 1_000_003), WORKERS)
        pf = Prefetcher(stream, codecs, writer=writer)
        got, t0, last = 0, time.time(), time.time()
        try:
            while have + got < quota:
                got += pf.get()["n_live"]
                if time.time() - last > 30:
                    rate = got / (time.time() - t0)
                    print(f"  {r:6s} {(have + got) / 1e9:.3f} / {quota / 1e9:.3f}B codes | {rate / 1e3:.1f}k codes/s | "
                          f"eta {(quota - have - got) / max(rate, 1) / 3600:.1f} h", flush=True)
                    last = time.time()
                if SESSION_HOURS and time.time() - T_START > SESSION_HOURS * 3600:
                    print(f"  watchdog: {SESSION_HOURS} h reached. Shards written so far are kept: run this cell again to continue.")
                    return False
        finally:
            pf.close(); stream.close(); writer.flush_all()
        print(f"  {r:6s} done: {(have + got) / 1e9:.3f}B codes in {(time.time() - t0) / 60:.1f} min", flush=True)
    writer.close()
    n_files, n_bytes, rows = shard_stats(SHARD_DIR)
    print(f"shards complete: {n_files} files, {n_bytes / 2**30:.2f} GiB, rows {rows}")
    return True

## 6. Codec fit and held-out set (streamed, held in RAM only; stored under `SHARD_DIR` when shards are on)

This stage needs no model, so it sits right after the data and shard sections. With `SHARD_MODE="encode"` the notebook
goes from here straight to encoding the dataset (next cell); the kernel, model and build sections further down are
skipped, so no GPU time is spent on them.

In [ ]:
if not META_ONLY or SHARD_MODE == "encode":
    PARTS = max(WORKERS, 1) + 1                 # part 0 = validation; parts 1.. = training workers
    codecs = Codecs(SPEC, dev)

    assert not (TOKENIZER in ("vq", "feat") and USE_SHARDS), "shards store codes of one fixed codec: use TOKENIZER='kmeans' with SHARD_MODE record/read/encode, or SHARD_MODE='off'"

    def fit_codecs():
        if TOKENIZER == "feat":
            return fit_feature_codecs()                   # frozen DINOv2 / DAC features (section 3b): fit the codebooks, no training
        if TOKENIZER == "vq":
            return fit_tokenizers()                       # learned tokenizers (previous section): train, compare with k-means, gate
        t0 = time.time()
        fdir = os.path.join(SHARD_DIR, "fit") if USE_SHARDS else ""         # per-modality sample sets: raw enough to refit, tiny next to the raw data
        f_img, f_aud = os.path.join(fdir, "image.npy"), os.path.join(fdir, "audio.npy")
        feats, B = [], 64
        if fdir and os.path.exists(f_img):
            imgs_u8 = np.load(f_img, mmap_mode="r")
            print(f"fit set for images from {f_img} ({len(imgs_u8)} images, no streaming)")
            n_b = len(imgs_u8) // B + 1
            for i in range(0, len(imgs_u8), B):
                feats.append(sample_points(codecs.image.features(torch.from_numpy(np.ascontiguousarray(imgs_u8[i:i + B])).to(dev)), 200_000 // n_b))
        else:
            print(f"fitting frozen codebooks from {FIT_IMAGES} streamed images and {FIT_AUDIO} audio clips (RAM only)...")
            rows = collect_rows(("image",), {"image": FIT_IMAGES}, 1, PARTS, WORKERS)
            keep = []
            for i in range(0, len(rows), B):
                sub = rows[i:i + B]
                tmp = [None] * len(sub)
                jp = [j for j, r in enumerate(sub) if "jpeg" in r]
                if jp:
                    d = decode_jpeg_gpu([sub[j]["jpeg"] for j in jp], SPEC.side, dev)
                    for n, j in enumerate(jp):
                        tmp[j] = d[n]
                for j, r in enumerate(sub):
                    if tmp[j] is None:
                        tmp[j] = torch.from_numpy(r["img"]).to(dev)
                batch = torch.stack(tmp)
                if fdir:
                    keep.append(batch.cpu().numpy())
                feats.append(sample_points(codecs.image.features(batch), 200_000 // max(1, len(rows) // B + 1)))
            if fdir and keep:
                os.makedirs(fdir, exist_ok=True)
                np.save(f_img, np.concatenate(keep))
            del rows, keep
        codecs.image.fit(feats); del feats
        print(f"  image codebooks done ({time.time() - t0:.0f}s)")
        if fdir and os.path.exists(f_aud):
            wav_u8 = np.load(f_aud, mmap_mode="r")
            print(f"fit set for audio from {f_aud} ({len(wav_u8)} clips, no streaming)")
            waves = [torch.from_numpy(np.ascontiguousarray(wav_u8[i:i + 32])).to(dev).float() / 32768 for i in range(0, len(wav_u8), 32)]
        else:
            rows = collect_rows(("audio",), {"audio": FIT_AUDIO}, 1, PARTS, WORKERS)
            if fdir and rows:
                os.makedirs(fdir, exist_ok=True)
                np.save(f_aud, np.stack([r["wav"] for r in rows]))
            waves = [torch.from_numpy(np.stack([r["wav"] for r in rows[i:i + 32]])).to(dev).float() / 32768 for i in range(0, len(rows), 32)]
            del rows
        codecs.audio.set_stats(waves)
        codecs.audio.fit([sample_points(codecs.audio.features(w), 200_000 // max(1, len(waves))) for w in waves])
        print(f"  audio codebooks done ({time.time() - t0:.0f}s)  mel mean {codecs.audio.mean:.2f} std {codecs.audio.std:.2f}")

    CKPT_PATH = os.path.join(CKPT_DIR, "ckpt.pt") if CKPT_DIR else ""
    RESUME = None
    if CKPT_PATH and os.path.exists(CKPT_PATH):
        RESUME = torch.load(CKPT_PATH, map_location="cpu", weights_only=False)
        _ci_state = RESUME["codecs"]["image"]
        _kind = _ci_state["kind"] if isinstance(_ci_state, dict) and _ci_state.get("kind") in ("vq", "feat") else "kmeans"
        _want = "vq" if (TOKENIZER == "vq" or (TOKENIZER == "feat" and IMAGE_CODEC == "vq")) else TOKENIZER     # the image codec's state kind
        if _kind != _want:                                # e.g. the old 2B checkpoint in the same Drive folder: never mix codecs
            print(f"!! {CKPT_PATH} was trained with the '{_kind}' image codec but this run wants '{_want}' (TOKENIZER='{TOKENIZER}', IMAGE_CODEC='{IMAGE_CODEC}'): ignoring it (use a new CKPT_DIR to keep both)")
            RESUME = None
    if RESUME is not None:
        codecs.load(RESUME["codecs"])
        print(f"resuming from {CKPT_PATH}: step {RESUME['step']}, codebooks restored (no refit)")
    elif SHARD_DIR and SHARD_MODE in ("record", "read", "encode") and os.path.exists(os.path.join(SHARD_DIR, "codecs.pt")):
        codecs.load(torch.load(os.path.join(SHARD_DIR, "codecs.pt"), map_location="cpu", weights_only=False))
        print(f"codebooks restored from {SHARD_DIR} (the shards were encoded with them; no refit)")
    elif SHARD_MODE == "read":
        raise RuntimeError(f"SHARD_MODE='read' but {SHARD_DIR}/codecs.pt does not exist: run once with SHARD_MODE='record' or 'encode'.")
    else:
        fit_codecs()
    if USE_SHARDS:
        print(f"shards: {SHARD_MODE} | dir {SHARD_DIR} | codebook fingerprint {shard_prepare(SHARD_DIR, codecs)}")

    def build_val():
        t0 = time.time()
        vdir = os.path.join(SHARD_DIR, "val") if USE_SHARDS else ""         # encoded held-out rows, one file per row type
        chunk = {r: min(MICRO_B[r], 16) for r in ROW_TYPES}
        if vdir and all(os.path.exists(os.path.join(vdir, f"{r}.npy")) for r in ROW_TYPES):
            out = []
            for r in ROW_TYPES:
                arr = np.load(os.path.join(vdir, f"{r}.npy"))
                for i in range(0, len(arr), chunk[r]):
                    cond, codes = unpack_rows(r, arr[i:i + chunk[r]], dev)
                    n = len(arr[i:i + chunk[r]])
                    out.append(dict(rtype=r, cond=cond, codes=codes, n_live=n * LAYOUTS[r].Tt, src=["val"] * n))
            print(f"held-out set from {vdir}: { {r: sum(len(b['src']) for b in out if b['rtype'] == r) for r in ROW_TYPES} } rows, no streaming")
            return out
        quota = {r: VAL_ROWS for r in ROW_TYPES}
        rows = collect_rows(None, quota, 0, PARTS, 1, seed=12345, even=True)
        out = []
        for r in ROW_TYPES:
            rr = [x for x in rows if x["t"] == r]
            for i in range(0, len(rr), chunk[r]):
                out.append(encode_rows(r, rr[i:i + chunk[r]], codecs))
        if vdir and all(any(b["rtype"] == r for b in out) for r in ROW_TYPES):
            os.makedirs(vdir, exist_ok=True)
            for r in ROW_TYPES:
                np.save(os.path.join(vdir, f"{r}.npy"), np.concatenate([pack_rows(b) for b in out if b["rtype"] == r]))
        print(f"held-out set: { {r: sum(len(b['src']) for b in out if b['rtype'] == r) for r in ROW_TYPES} } rows ({time.time() - t0:.0f}s)")
        return out

    VAL = build_val()

## 7d. Codec checks (before the model is built: no GPU time is spent on a bad tokenizer)

Streams a small fresh sample of every modality through the **frozen codecs that were just fitted** and checks:

1. **Contract** (hard): every level has the ladder length, codes are in 1..256, no level is constant, no NaN, and the finest text
   level decodes to the exact bytes. A failure here stops the run (`CODEC_GATE="stop"`).
2. **What the codes keep** (soft): image PSNR (pixel codecs) or feature R^2 (DINO), audio log-mel R^2, per level.
3. **How the codes are used** (soft): entropy, codes used and perplexity per level (a level near 0 bits or with few codes is wasted).
4. **Learnability** (soft): (a) the bits the parent level tells about each level (a first-order estimate, bias-controlled with a
   shuffled control) and (b) a **probe**: a small conv net trained for a few hundred steps to predict level *k* from level *k-1*,
   scored on held-out rows against the unigram. The probe's gain is a rough ceiling on what the big model can win at that level
   from the previous one, so a level whose probe gain is ~0 will sit at the constant baseline however long the model trains.

The probe results are kept in `PROBE` and compared with the trained model in the final "did it learn" cells.

In [ ]:
CODEC_CHECK = _knob("CODEC_CHECK", True)
CODEC_GATE = _knob("CODEC_GATE", "stop")            # "stop" = raise on a hard failure | "warn" = print and continue
CHK_IMAGES, CHK_AUDIO, CHK_TEXT = _knob("CHK_IMAGES", 1200), _knob("CHK_AUDIO", 500), _knob("CHK_TEXT", 3000)
CHK_PROBE_STEPS = _knob("CHK_PROBE_STEPS", 300)
PROBE, CODEC_FLAGS, CODEC_REPORT = {}, [], {}


def _flag(sev, msg):
    CODEC_FLAGS.append((sev, msg))
    print(("  !! HARD: " if sev == "HARD" else "  ~ ") + msg, flush=True)


def _H(counts):
    p = counts.double(); p = p[p > 0]; p = p / p.sum()
    return float(-(p * p.log2()).sum())


def _cond_H(par, ch):                                        # H(child | context) in bits, both codes 1..256
    V = 257
    return _H(torch.bincount(par * V + ch, minlength=V * V)) - _H(torch.bincount(par, minlength=V))


def _parent_index(k, mod):                                   # parent slot (level k-1) of every slot at level k
    lens = SPEC.lens(mod)
    n, f = lens[k], math.isqrt(SPEC.R_of(mod))
    i = torch.arange(n, device=dev)
    if k and lens[k - 1] == n:                               # a residual level at the same resolution: the parent is the same slot
        return i
    if mod == "image":
        g = math.isqrt(n)
        return ((i // g) // f) * (g // f) + ((i % g) // f)
    return i // SPEC.R_of(mod)


def _upsample_parent(prev, n_k, mod):                        # [B, n_{k-1}] -> [B, n_k]: every slot sees its parent's code
    B, n_p = prev.shape
    r = n_k // n_p
    if mod == "image":
        g_p, f = math.isqrt(n_p), math.isqrt(r)
        return prev.view(B, g_p, g_p).repeat_interleave(f, 1).repeat_interleave(f, 2).reshape(B, n_k)
    return prev.repeat_interleave(r, 1)


class _Probe(nn.Module):
    """Small conv net: embed the parent's code at every slot (+ a learned position), 3 conv layers, predict the 257-way code."""

    def __init__(self, n, nd, d=64):
        super().__init__()
        self.nd = nd
        self.emb = nn.Embedding(257, d)
        self.pos = nn.Parameter(torch.zeros(1, d, *([math.isqrt(n)] * 2 if nd == 2 else [n])))
        conv, ks = (nn.Conv2d, 3) if nd == 2 else (nn.Conv1d, 5)
        self.net = nn.Sequential(conv(d, 128, ks, padding=ks // 2), nn.GELU(), conv(128, 128, ks, padding=ks // 2), nn.GELU(),
                                 conv(128, 128, ks, padding=ks // 2), nn.GELU(), conv(128, 257, 1))

    def forward(self, up):                                   # long [B, n] -> logits [B, 257, n]
        x = self.emb(up).transpose(1, 2)
        if self.nd == 2:
            g = math.isqrt(up.shape[1])
            x = x.reshape(up.shape[0], -1, g, g)
        return self.net(x + self.pos).flatten(2)


def _probe_level(codes, k, mod, steps):
    """(unigram bits, probe bits) per code on held-out rows for level k predicted from level k-1."""
    tgt = codes[k]
    N, n = tgt.shape
    n_tr = int(N * 0.8)
    up = _upsample_parent(codes[k - 1], n, mod)
    p = _Probe(n, 2 if mod == "image" else 1).to(dev)
    opt = torch.optim.AdamW(p.parameters(), lr=2e-3, weight_decay=0.01)
    with torch.enable_grad():                                # the caller runs under no_grad
        for _ in range(steps):
            idx = torch.randint(n_tr, (min(64, n_tr),), device=dev)
            loss = F.cross_entropy(p(up[idx]), tgt[idx])
            opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    with torch.no_grad():
        ce = F.cross_entropy(p(up[n_tr:]), tgt[n_tr:]).item()
        cnt = torch.bincount(tgt[:n_tr].flatten(), minlength=257).float() + 1
        uni = float(-(cnt / cnt.sum()).log()[tgt[n_tr:].flatten()].mean())
    return uni / math.log(2), ce / math.log(2)


def _analyse(name, mod, codes):
    """contract + usage + learnability for one modality; codes = K tensors [N, n_k] (long, 1..256)."""
    lens = SPEC.lens(mod)
    print(f"\n{name}: level | slots | codes used | entropy (bits, max 8) | perplexity | parent info (bits) | probe: unigram -> probe bits, gain")
    if [c.shape[1] for c in codes] != list(lens):
        _flag("HARD", f"{name}: level lengths {[c.shape[1] for c in codes]} != ladder {lens}")
    rows = []
    for k, c in enumerate(codes):
        cnt = torch.bincount(c.flatten(), minlength=257)
        h, used = _H(cnt), int((cnt[1:] > 0).sum())
        if int(c.min()) < 1 or int(c.max()) > 256:
            _flag("HARD", f"{name} L{k}: codes outside 1..256 (min {int(c.min())}, max {int(c.max())})")
        if used < 2:
            _flag("HARD", f"{name} L{k}: every code is identical (a constant level carries no information)")
        elif lens[k] >= 16 and used < 16:
            _flag("SOFT", f"{name} L{k}: only {used} codes used across {lens[k]} slots (wasted capacity)")
        line = f"   L{k}  {c.shape[1]:4d}   {used:3d}   {h:5.2f}   {2 ** h:6.1f}"
        if k == 0:
            print(line); rows.append(dict(level=k, entropy=h, used=used)); continue
        par, ch = codes[k - 1][:, _parent_index(k, mod)].flatten(), c.flatten()
        gain_par = _cond_H(par, ch[torch.randperm(len(ch), device=ch.device)]) - _cond_H(par, ch)
        uni, pb = _probe_level(codes, k, mod, CHK_PROBE_STEPS)
        PROBE[(mod, k)] = dict(uni_bits=uni, probe_bits=pb, gain_bits=uni - pb, parent_bits=gain_par, entropy=h, used=used)
        print(f"{line}   {gain_par:5.2f}   {uni:5.2f} -> {pb:5.2f}  ({uni - pb:+.2f})")
        rows.append(dict(level=k, entropy=h, used=used, parent=gain_par, probe_gain=uni - pb))
    if rows[-1]["entropy"] < 2.0:
        _flag("SOFT", f"{name}: the finest level has only {rows[-1]['entropy']:.2f} bits of entropy")
    if len(rows) > 1 and rows[-1].get("probe_gain", 1) < 0.05:
        _flag("SOFT", f"{name}: the finest level gains only {rows[-1]['probe_gain']:+.3f} bits from its parent level (little learnable structure)")
    return rows


def _fmt_levels(vals, fmt):
    return " ".join(f"L{k}=" + fmt.format(v) for k, v in enumerate(vals))


@torch.no_grad()
def run_codec_checks():
    K = SPEC.K
    print(f"codecs: image {type(codecs.image).__name__} | audio {type(codecs.audio).__name__} | text {type(codecs.text).__name__} "
          f"| TOKENIZER={TOKENIZER}" + (f" FEAT_QUANT={FEAT_QUANT} TEXT_CODEC={TEXT_CODEC}" if TOKENIZER == "feat" else ""))
    # ---------------- image ----------------
    try:
        imgs = _img_tensor(collect_rows(("image",), {"image": CHK_IMAGES}, 1, PARTS, WORKERS))
        Ki = LAYOUTS["image"].K                              # images may have one more level than SPEC.K (IMAGE_RES="v2")
        acc, num, den, kind = [[] for _ in range(Ki)], [0.0] * Ki, [0.0] * Ki, None
        for i in range(0, len(imgs), 64):
            x = imgs[i:i + 64]
            cl = codecs.image.encode(x)
            for k in range(Ki):
                acc[k].append(cl[k])
            if getattr(codecs.image, "decodable", True):
                kind = "psnr"
                for k in range(Ki):
                    num[k] += float(((codecs.image.decode(cl, level=k).float() - x.float()) ** 2).sum()); den[k] += x.numel()
            else:
                kind, feats = "r2", codecs.image.features(x)
                for k in range(Ki):
                    f = feats[k].reshape(-1, feats[k].shape[-1]); p = codecs.image.reconstruct(cl[k], k).reshape(f.shape)
                    num[k] += float(((f - p) ** 2).sum()); den[k] += float(((f - f.mean(0)) ** 2).sum())
        if kind == "psnr":
            rec = [10 * math.log10(255 ** 2 / max(num[k] / den[k], 1e-9)) for k in range(Ki)]
            print(f"IMAGE: {len(imgs)} images | PSNR from the codes of levels 0..k (dB): " + _fmt_levels(rec, "{:.1f}"))
        else:
            rec = [1 - num[k] / max(den[k], 1e-9) for k in range(Ki)]
            print(f"IMAGE: {len(imgs)} images | DINO-feature R^2 of the prototype features per level (1.0 = perfect): " + _fmt_levels(rec, "{:.2f}"))
            if rec[-1] < 0.3:
                _flag("SOFT", f"image finest-level feature R^2 {rec[-1]:.2f} is low: the 256 codes keep little of the features")
        if not all(math.isfinite(r) for r in rec):
            _flag("HARD", "image reconstruction metric is not finite")
        CODEC_REPORT["image"] = dict(kind=kind, recon=rec, rows=_analyse("IMAGE", "image", [torch.cat(a) for a in acc]))
        del imgs
    except Exception as e:
        _flag("HARD", f"image codec failed: {type(e).__name__}: {str(e)[:200]}")
    # ---------------- audio ----------------
    try:
        rows = collect_rows(("audio",), {"audio": CHK_AUDIO}, 1, PARTS, WORKERS)
        wav = torch.from_numpy(np.stack([r["wav"] for r in rows])).to(dev); del rows
        helper = AudioCodec(SPEC, dev)                       # only its STFT / mel filterbank: the same mel space for every audio codec
        Ka = LAYOUTS["audio"].K                              # audio may have residual levels beyond SPEC.K (AUDIO_RES="v2")
        acc, num, den = [[] for _ in range(Ka)], [0.0] * Ka, [0.0] * Ka
        AB = globals().get("FEAT_AUDIO_BATCH", 32)           # DAC at 44.1 kHz is heavy: keep the batch small on a small GPU
        for i in range(0, len(wav), AB):
            w = wav[i:i + AB].float() / 32768.0
            cl = codecs.audio.encode(w)
            for k in range(Ka):
                acc[k].append(cl[k])
            if hasattr(codecs.audio, "decode_mel") or getattr(codecs.audio, "decodable", False):
                ref = helper.logmel(w)
                for k in range(Ka):
                    z = codecs.audio.decode_mel(cl, level=k) if hasattr(codecs.audio, "decode_mel") else helper.logmel(codecs.audio.decode(cl, level=k).float())
                    num[k] += float(((z - ref) ** 2).sum()); den[k] += float(((ref - ref.mean()) ** 2).sum())
        rec = [1 - num[k] / max(den[k], 1e-9) for k in range(Ka)]
        print(f"AUDIO: {len(wav)} clips | log-mel R^2 of the decoded audio from the codes of levels 0..k (1.0 = perfect): " + _fmt_levels(rec, "{:.2f}"))
        if not all(math.isfinite(r) for r in rec):
            _flag("HARD", "audio reconstruction metric is not finite")
        elif rec[-1] < 0.5:
            _flag("SOFT", f"audio finest-level log-mel R^2 {rec[-1]:.2f} is low")
        CODEC_REPORT["audio"] = dict(kind="r2", recon=rec, rows=_analyse("AUDIO", "audio", [torch.cat(a) for a in acc]))
        del wav
    except Exception as e:
        _flag("HARD", f"audio codec failed: {type(e).__name__}: {str(e)[:200]}")
    # ---------------- text ----------------
    try:
        rows = collect_rows(("text0",), {"text0": CHK_TEXT}, 1, PARTS, WORKERS)
        b = torch.from_numpy(np.stack([r["tgt"] for r in rows])).to(dev); del rows
        cl = codecs.text.encode(b)
        exact = TextCodec.decode(cl[-1][:64]) == TextCodec.decode(b[:64].long() + 1)     # decode() subtracts 1: codes are byte + 1
        print(f"\nTEXT: {len(b)} rows | finest level decodes to the exact bytes: {exact}")
        if not exact:
            _flag("HARD", "text: the finest level does not decode to the input bytes")
        CODEC_REPORT["text"] = dict(exact=exact, rows=_analyse("TEXT", "text", cl))
    except Exception as e:
        _flag("HARD", f"text codec failed: {type(e).__name__}: {str(e)[:200]}")
    # ---------------- gate ----------------
    hard = [m for s, m in CODEC_FLAGS if s == "HARD"]
    soft = [m for s, m in CODEC_FLAGS if s == "SOFT"]
    gc.collect()
    if IS_CUDA:
        torch.cuda.empty_cache()                             # the probes and the check batches must not sit in the cache next to the model
    print(f"\nCODEC CHECKS: {len(hard)} hard failure(s), {len(soft)} warning(s)")
    if hard and CODEC_GATE == "stop":
        raise RuntimeError("codec checks failed: " + "; ".join(hard) + ". Fix the codec, or set CODEC_GATE='warn' to continue anyway.")


if CODEC_CHECK and "codecs" in globals():
    run_codec_checks()

## 7. Encode the dataset to shards (only when `SHARD_MODE="encode"`)

Streams the whole dataset for the chosen budget (`SHARD_CODES_B`, else `TOKENS`, else the tier: T3 = 50B live codes,
about 60 GB), encodes it with the frozen codebooks fitted above and stores it per row type under `SHARD_DIR`. No model
step runs, so it is limited by the Hub feed alone. It is resumable (run the cell again after an interruption). When it
finishes, the build, budget, correctness, training, verdict and generation cells skip themselves; set `SHARD_MODE="read"`
and run the notebook again to train from the stored shards.

In [ ]:
if SHARD_MODE == "encode":
    ENCODED = encode_shards()
    print("encode-only run finished: set SHARD_MODE='read' and run the notebook again to train from the shards."
          if ENCODED else "encode-only run stopped early: run this cell again to continue.")

### 7b. Bundle the shards into one archive (to copy to Drive yourself)

Runs after the encode cell when `SHARD_MODE="encode"` and prints how long it took.

In [ ]:
if SHARD_MODE == "encode" and os.path.isdir(SHARD_DIR):
    ARCHIVE = os.path.join(os.path.dirname(os.path.abspath(SHARD_DIR)), os.path.basename(SHARD_DIR.rstrip("/")) + "_" + TIER + ".tar.gz")
    t_tar = time.time()
    subprocess.run(["tar", "-czf", ARCHIVE, "-C", os.path.dirname(os.path.abspath(SHARD_DIR)), os.path.basename(SHARD_DIR.rstrip("/"))],
                   check=True, env={**os.environ, "GZIP": "-1"})
    print(f"tar.gz done in {(time.time() - t_tar) / 60:.1f} min ({time.time() - t_tar:.0f} s) | {os.path.getsize(ARCHIVE) / 2**30:.2f} GiB | {ARCHIVE}")

## 8. Kernels (training only; the shard path above never uses them)

| op | implementation |
|---|---|
| SwiGLU `silu(a) * b` on the fused gate+up projection | Triton: one read, one write, fused backward (registered as a `torch.library` custom op so `torch.compile` can see through it) |
| RMSNorm, residual adds, QK-norm | `F.rms_norm` + Inductor fusion |
| attention | flex-attention block mask (level-block-causal) -> SDPA with a bool mask -> CPU math |
| head matmul + cross-entropy | chunked and **checkpointed**: the `[tokens, V]` logits are never kept for backward |
| expert GEMMs | both experts of a group run as one `bmm` (capacity is uniform, so a grouped GEMM *is* a batched GEMM) |
| optimizer | `AdamW(fused=True)` |

In [ ]:
def swiglu_torch(h):
    a, b = h.chunk(2, dim=-1)
    return F.silu(a) * b

if HAS_TRITON:
    import triton
    import triton.language as tl

    @triton.jit
    def _swiglu_fwd_k(h_ptr, o_ptr, H, BLOCK: tl.constexpr):
        row = tl.program_id(0).to(tl.int64)
        cols = tl.program_id(1) * BLOCK + tl.arange(0, BLOCK)
        m = cols < H
        a = tl.load(h_ptr + row * 2 * H + cols, mask=m, other=0.0).to(tl.float32)
        b = tl.load(h_ptr + row * 2 * H + H + cols, mask=m, other=0.0).to(tl.float32)
        o = a * tl.sigmoid(a) * b
        tl.store(o_ptr + row * H + cols, o.to(o_ptr.dtype.element_ty), mask=m)

    @triton.jit
    def _swiglu_bwd_k(h_ptr, g_ptr, d_ptr, H, BLOCK: tl.constexpr):
        row = tl.program_id(0).to(tl.int64)
        cols = tl.program_id(1) * BLOCK + tl.arange(0, BLOCK)
        m = cols < H
        a = tl.load(h_ptr + row * 2 * H + cols, mask=m, other=0.0).to(tl.float32)
        b = tl.load(h_ptr + row * 2 * H + H + cols, mask=m, other=0.0).to(tl.float32)
        g = tl.load(g_ptr + row * H + cols, mask=m, other=0.0).to(tl.float32)
        s = tl.sigmoid(a)
        da = g * b * s * (1.0 + a * (1.0 - s))
        db = g * a * s
        tl.store(d_ptr + row * 2 * H + cols, da.to(d_ptr.dtype.element_ty), mask=m)
        tl.store(d_ptr + row * 2 * H + H + cols, db.to(d_ptr.dtype.element_ty), mask=m)

    @torch.library.custom_op("ncm::swiglu_fwd", mutates_args=())
    def _swiglu_fwd(h: torch.Tensor) -> torch.Tensor:
        h = h.contiguous()
        H = h.shape[-1] // 2
        o = torch.empty(*h.shape[:-1], H, device=h.device, dtype=h.dtype)
        n = h.numel() // (2 * H)
        BLOCK = min(1024, triton.next_power_of_2(H))
        _swiglu_fwd_k[(n, triton.cdiv(H, BLOCK))](h, o, H, BLOCK=BLOCK)
        return o

    @_swiglu_fwd.register_fake
    def _(h):
        return h.new_empty(*h.shape[:-1], h.shape[-1] // 2)

    @torch.library.custom_op("ncm::swiglu_bwd", mutates_args=())
    def _swiglu_bwd(h: torch.Tensor, g: torch.Tensor) -> torch.Tensor:
        h, g = h.contiguous(), g.contiguous()
        H = h.shape[-1] // 2
        d = torch.empty_like(h)
        n = h.numel() // (2 * H)
        BLOCK = min(1024, triton.next_power_of_2(H))
        _swiglu_bwd_k[(n, triton.cdiv(H, BLOCK))](h, g, d, H, BLOCK=BLOCK)
        return d

    @_swiglu_bwd.register_fake
    def _(h, g):
        return torch.empty_like(h)

    def _swiglu_setup(ctx, inputs, output):
        ctx.save_for_backward(inputs[0])

    def _swiglu_back(ctx, g):
        (h,) = ctx.saved_tensors
        return _swiglu_bwd(h, g)

    _swiglu_fwd.register_autograd(_swiglu_back, setup_context=_swiglu_setup)
    swiglu_triton = _swiglu_fwd
else:
    swiglu_triton = None


def _pick_swiglu():
    """Parity first, then a microbenchmark: Triton wins only if it is both correct and faster."""
    if SWIGLU == "torch" or swiglu_triton is None:
        return swiglu_torch, "torch"
    h = torch.randn(4096, 2 * P["ffn"], device=dev, dtype=AMP_DTYPE, requires_grad=True)
    ref = swiglu_torch(h.float())
    got = swiglu_triton(h)
    ok = torch.allclose(got.float(), ref, atol=3e-2, rtol=3e-2)
    g = torch.randn_like(got)
    g1, = torch.autograd.grad(got, h, g)
    g2, = torch.autograd.grad(swiglu_torch(h.float()), h, g.float())
    ok = ok and torch.allclose(g1.float(), g2.float(), atol=3e-2, rtol=3e-2)
    if not ok:
        print("!! triton swiglu failed parity: using torch")
        return swiglu_torch, "torch"
    if SWIGLU == "triton":
        return swiglu_triton, "triton"
    comp = torch.compile(swiglu_torch)
    def bench(f):
        for _ in range(3):
            f(h).sum().backward()
        torch.cuda.synchronize(); t = time.time()
        for _ in range(20):
            f(h).sum().backward()
        torch.cuda.synchronize()
        return (time.time() - t) / 20
    try:
        tt, tc = bench(swiglu_triton), bench(comp)
    except Exception as e:
        print("swiglu bench skipped:", type(e).__name__)
        return swiglu_triton, "triton"
    print(f"swiglu fwd+bwd: triton {tt*1e3:.2f} ms | torch.compile {tc*1e3:.2f} ms")
    return (swiglu_triton, "triton") if tt <= tc * 1.05 else (swiglu_torch, "torch")

if IS_CUDA and HAS_TRITON and SHARD_MODE != "encode":      # encode-only runs skip the GPU benchmark and compile
    swiglu, SWIGLU_IMPL = _pick_swiglu()
else:
    swiglu, SWIGLU_IMPL = swiglu_torch, "torch"
print("swiglu kernel:", SWIGLU_IMPL)


# ---- chunked, checkpointed head + cross-entropy ----
def _ce_chunk(h, W, b, tgt):
    logits = F.linear(h, W, b).float()
    lse = torch.logsumexp(logits, dim=-1)
    ce = lse - logits.gather(-1, tgt[:, None]).squeeze(-1)
    return torch.stack([ce, lse])


def fused_ce(h, W, b, tgt, chunk=16384, ckpt=True):
    """h [N, d], tgt [N] -> [2, N] = (cross-entropy, logsumexp). Logits live one chunk at a time."""
    from torch.utils.checkpoint import checkpoint
    outs = []
    for i in range(0, h.shape[0], chunk):
        args = (h[i:i + chunk], W, b, tgt[i:i + chunk])
        if ckpt and torch.is_grad_enabled():
            outs.append(checkpoint(_ce_chunk, *args, use_reentrant=False))
        else:
            outs.append(_ce_chunk(*args))
    return torch.cat(outs, dim=1)

## 9. The model (training only)

* **Embeddings**: one code table per modality (vocab 257) + learned absolute position per slot
  (layouts are fixed-length and the image is 2D, so learned positions replace RoPE).
* **Attention**: GQA 16/4, QK-RMSNorm, level-block-causal mask.
* **Experts**: each modality *group* has 2 SwiGLU experts. A token only ever reaches its own
  group (forced by modality tag, so no router collapse across modalities). Inside the group routing
  is **expert-choice per level**: expert *e* takes the `ceil(cf * l)` tokens of the level it likes
  best. Capacity is therefore exact, shapes are static, there is no load-balance loss, and a token
  gets 0, 1 or 2 experts (1 on average at `cf=0.5`). Choosing among the tokens of **one level**
  (never across levels) keeps routing from leaking information from finer levels.

In [ ]:
from torch.utils.checkpoint import checkpoint as _ckpt


class RMSNorm(nn.Module):
    def __init__(self, d, eps=1e-6):
        super().__init__()
        self.w, self.eps = nn.Parameter(torch.ones(d)), eps

    def forward(self, x):
        return F.rms_norm(x, (x.shape[-1],), self.w.to(x.dtype), self.eps)


_flex_fn = None
# Smaller tiles: the default 128x128 forward tile at head_dim 96 needs ~178 KB of shared memory, over the A100's 163 KB limit.
FLEX_OPTS = dict(BLOCK_M=64, BLOCK_N=64, BLOCK_M1=32, BLOCK_N1=64, BLOCK_M2=64, BLOCK_N2=32)
def _flex():
    global _flex_fn
    if _flex_fn is None:
        from torch.nn.attention.flex_attention import flex_attention
        _flex_fn = flex_attention if torch.compiler.is_compiling() else torch.compile(flex_attention)
    return _flex_fn


class Attn(nn.Module):
    def __init__(self, d, nh, nkv):
        super().__init__()
        self.nh, self.nkv, self.hd = nh, nkv, d // nh
        self.qkv = nn.Linear(d, (nh + 2 * nkv) * self.hd, bias=False)
        self.o = nn.Linear(d, d, bias=False)
        self.qn, self.kn = RMSNorm(self.hd), RMSNorm(self.hd)

    def forward(self, x, lay, use_flex):
        B, T, _ = x.shape
        q, k, v = self.qkv(x).split([self.nh * self.hd, self.nkv * self.hd, self.nkv * self.hd], dim=-1)
        q = self.qn(q.view(B, T, self.nh, self.hd)).transpose(1, 2)
        k = self.kn(k.view(B, T, self.nkv, self.hd)).transpose(1, 2)
        v = v.view(B, T, self.nkv, self.hd).transpose(1, 2)
        if ATTN == "levels":
            # block-causal over levels == for each level (and the prefix), plain unmasked attention over every key up to the end of that level
            r = self.nh // self.nkv
            segs = ([(0, lay.cond)] if lay.cond else []) + list(lay.lvl)
            y = torch.cat([F.scaled_dot_product_attention(q[:, :, a:b], k[:, :, :b].repeat_interleave(r, 1), v[:, :, :b].repeat_interleave(r, 1))
                           for a, b in segs], 2)
        elif use_flex:
            y = _flex()(q, k, v, block_mask=lay.flex_mask, enable_gqa=True, kernel_options=FLEX_OPTS)
        else:
            r = self.nh // self.nkv
            y = F.scaled_dot_product_attention(q, k.repeat_interleave(r, 1), v.repeat_interleave(r, 1),
                                               attn_mask=lay.mask)
        return self.o(y.transpose(1, 2).reshape(B, T, -1))


class GroupExperts(nn.Module):
    """E SwiGLU experts for one modality; expert-choice routing per segment (= per level)."""

    def __init__(self, d, h, E, cf):
        super().__init__()
        self.E, self.h, self.cf = E, h, cf
        self.router = nn.Linear(d, E, bias=False)
        self.w1 = nn.Parameter(torch.empty(E, d, 2 * h))      # gate | up, one GEMM
        self.w2 = nn.Parameter(torch.empty(E, h, d))
        self.record = False
        self.last_cov = None

    def forward(self, x, segs):
        B, T, d = x.shape
        E = self.E
        idxs, ws = [], []
        for a, b in segs:
            l = b - a
            c = max(1, math.ceil(self.cf * l))
            p = self.router(x[:, a:b]).float().softmax(-1)             # [B, l, E]
            w, i = p.transpose(1, 2).topk(c, dim=-1)                   # each expert's top-c tokens
            idxs.append(i + a); ws.append(w)
        idx, w = torch.cat(idxs, -1), torch.cat(ws, -1)               # [B, E, C]
        C = idx.shape[-1]
        if MOE_DISPATCH == "flat":
            # same maths, rows addressed in one flat table: one gather straight into the [E, B*C, d] layout and one index_add_ back
            tok = (idx + torch.arange(B, device=x.device).view(B, 1, 1) * T).permute(1, 0, 2).reshape(E * B * C)
            xg = x.reshape(B * T, d).index_select(0, tok).view(E, B * C, d)
            hh = swiglu(torch.bmm(xg, self.w1.to(xg.dtype)))          # [E, BC, h]
            y = torch.bmm(hh, self.w2.to(xg.dtype))                   # [E, BC, d]
            y = y * w.permute(1, 0, 2).reshape(E, B * C, 1).to(y.dtype)
            out = x.new_zeros(B * T, d).index_add_(0, tok, y.reshape(E * B * C, d).to(x.dtype)).view(B, T, d)
        else:
            xg = torch.gather(x.unsqueeze(1).expand(B, E, T, d), 2, idx.unsqueeze(-1).expand(B, E, C, d))
            xg = xg.permute(1, 0, 2, 3).reshape(E, B * C, d)
            hh = swiglu(torch.bmm(xg, self.w1.to(xg.dtype)))          # [E, BC, h]
            y = torch.bmm(hh, self.w2.to(xg.dtype)).view(E, B, C, d).permute(1, 0, 2, 3)
            y = y * w.unsqueeze(-1).to(y.dtype)
            out = x.new_zeros(B, T, d)
            out.scatter_add_(1, idx.reshape(B, E * C, 1).expand(B, E * C, d), y.reshape(B, E * C, d).to(out.dtype))
        if self.record:
            with torch.no_grad():
                cnt = torch.zeros(B, T, device=x.device).scatter_add_(1, idx.reshape(B, E * C), torch.ones(B, E * C, device=x.device))
                live = torch.zeros(T, dtype=torch.bool, device=x.device)
                for a, b in segs:
                    live[a:b] = True
                cl = cnt[:, live]
                self.last_cov = torch.stack([(cl == j).float().mean() for j in range(E + 1)])
        return out


class Block(nn.Module):
    def __init__(self, d, nh, nkv, h, E, cf):
        super().__init__()
        self.ln1, self.ln2 = RMSNorm(d), RMSNorm(d)
        self.attn = Attn(d, nh, nkv)
        self.experts = nn.ModuleDict({m: GroupExperts(d, h, E, cf) for m in ("text", "image", "audio")})

    def forward(self, x, lay, use_flex):
        x = x + self.attn(self.ln1(x), lay, use_flex)
        y = self.ln2(x)
        moe = None
        for g, segs in lay.segs.items():                 # only the groups this row type touches
            o = self.experts[g](y, segs)
            moe = o if moe is None else moe + o
        return x + moe


def up_codes(codes, m, spec, n_out=None):
    """Level k-1 codes -> level k grid (nearest). Image is 2D (2x2), text/audio are 1D. Same length (a residual level) = identity."""
    r = spec.R_of(m)                     # text pools x2, image 2x2, audio x4 - per modality
    if n_out is not None and codes.shape[1] == n_out:
        return codes
    if m == "image":
        B, n = codes.shape
        g, f = math.isqrt(n), math.isqrt(r)
        return codes.view(B, g, g).repeat_interleave(f, 1).repeat_interleave(f, 2).reshape(B, -1)
    return codes.repeat_interleave(r, 1)


class Model(nn.Module):
    def __init__(self, spec, d, layers, heads, kv_heads, ffn, cf, act_ckpt, E=2):
        super().__init__()
        self.spec, self.d, self.act_ckpt = spec, d, act_ckpt
        mods = ("text", "image", "audio")
        self.emb = nn.ModuleDict({m: nn.Embedding(spec.V, d) for m in mods})
        self.pos = nn.ParameterDict({m: nn.Parameter(torch.zeros(sum(spec.lens(m)), d)) for m in mods})
        self.pos_cond = nn.Parameter(torch.zeros(max(spec.cond, 1), d))
        self.blocks = nn.ModuleList([Block(d, heads, kv_heads, ffn, E, cf) for _ in range(layers)])
        self.norm = RMSNorm(d)
        self.heads = nn.ModuleDict({m: nn.Linear(d, spec.V) for m in mods})
        self.use_flex = False
        self.lay = None
        self.apply(self._init)
        resid = 0.02 / math.sqrt(2 * layers)
        for blk in self.blocks:
            nn.init.normal_(blk.attn.o.weight, std=resid)
            for g in blk.experts.values():
                nn.init.normal_(g.w1, std=0.02)
                nn.init.normal_(g.w2, std=resid)
                nn.init.normal_(g.router.weight, std=0.02)
        for p_ in list(self.pos.values()) + [self.pos_cond]:
            nn.init.normal_(p_, std=0.02)
        for hd in self.heads.values():                  # zero head: step-0 prediction is uniform, loss = ln(257) at any width
            nn.init.zeros_(hd.weight)

    @staticmethod
    def _init(m):
        if isinstance(m, (nn.Linear, nn.Embedding)):
            nn.init.normal_(m.weight, std=0.02)
            if getattr(m, "bias", None) is not None:
                nn.init.zeros_(m.bias)

    def hidden(self, rtype, cond, codes):
        """cond [B, Lc] long | None, codes: K tensors [B, l_k] long -> [B, T, d].
        Level k's input is the upsampled codes of level k-1; level 0 reads the NULL code."""
        lay, m, spec = self.lay[rtype], ROW_MOD[rtype], self.spec
        B = codes[0].shape[0]
        parts = []
        if lay.cond:
            parts.append(self.emb["text"](cond) + self.pos_cond[:lay.cond])
        E, o = self.emb[m], 0
        for k, n in enumerate(lay.lens):
            ids = codes[0].new_zeros(B, n) if k == 0 else up_codes(codes[k - 1], m, spec, n)
            parts.append(E(ids) + self.pos[m][o:o + n])
            o += n
        x = torch.cat(parts, 1)
        if RESID == "bf16" and torch.is_autocast_enabled():          # fp32 residual = a cast + an fp32 norm around every matmul
            x = x.to(AMP_DTYPE)
        for blk in self.blocks:
            if self.act_ckpt and self.training and torch.is_grad_enabled():
                x = _ckpt(blk, x, lay, self.use_flex, use_reentrant=False)
            else:
                x = blk(x, lay, self.use_flex)
        return self.norm(x)

    def loss(self, rtype, cond, codes):
        lay, m = self.lay[rtype], ROW_MOD[rtype]
        h = self.hidden(rtype, cond, codes)[:, lay.cond:]
        B, Tt, d = h.shape
        tgt = torch.cat(codes, 1)
        head = self.heads[m]
        out = fused_ce(h.reshape(-1, d), head.weight, head.bias, tgt.reshape(-1))
        ce, lse = out[0].view(B, Tt), out[1].view(B, Tt)
        tok = ce + 1e-4 * lse * lse                                # light z-loss keeps logits bounded
        per_level = torch.stack([ce[:, a - lay.cond:b - lay.cond].mean() for a, b in lay.lvl]).detach()
        if LEVEL_W == "slots":
            loss = tok.mean()                                      # every position counts the same: a level's weight is its slot count
        else:                                                      # average inside each level first, then weight the levels (coarse levels are not drowned out)
            lvl = torch.stack([tok[:, a - lay.cond:b - lay.cond].mean() for a, b in lay.lvl])
            w = torch.tensor([float(b - a) ** (0.5 if LEVEL_W == "sqrt" else 0.0) for a, b in lay.lvl], device=lvl.device)
            loss = (lvl * w).sum() / w.sum()
        return loss, per_level

    def level_logits(self, rtype, h, k):
        lay, m = self.lay[rtype], ROW_MOD[rtype]
        a, b = lay.lvl[k]
        return self.heads[m](h[:, a:b]).float()


def build_model(meta=False):
    ctx = torch.device("meta") if meta else torch.device("cpu")
    with ctx:
        net = Model(SPEC, P["d"], P["layers"], P["heads"], P["kv_heads"], P["ffn"], MOE_CF, P["act_ckpt"] if ACT_CKPT < 0 else bool(ACT_CKPT))
    return net


def param_report(net):
    """total, trunk, per-expert, and active-per-token for each modality."""
    tot = sum(p.numel() for p in net.parameters())
    ex = {g: 0 for g in ("text", "image", "audio")}
    for blk in net.blocks:
        for g, mod in blk.experts.items():
            ex[g] += sum(p.numel() for n, p in mod.named_parameters() if n != "router.weight")
    experts_total = sum(ex.values())
    E = net.blocks[0].experts["text"].E
    per_expert = ex["text"] / E
    trunk = tot - experts_total
    act = {}
    for m in ex:
        own_emb = net.emb[m].weight.numel() + net.pos[m].numel() + sum(p.numel() for p in net.heads[m].parameters())
        other = sum(net.emb[o].weight.numel() + net.pos[o].numel() + sum(p.numel() for p in net.heads[o].parameters())
                    for o in ex if o != m)
        act[m] = trunk - other + ex[m] * (MOE_CF * E) / E
    return dict(total=tot, trunk=trunk, experts_total=experts_total, per_expert=per_expert,
                active=act, active_max=max(act.values()))

## 10. Build and size check (training only; skipped when `SHARD_MODE="encode"`)

The parameter report comes from a **meta-device** build (no memory), so it prints even on a machine that
cannot hold the model. The real build then checks that fp32 weights + AdamW (16 bytes/param) fit.

In [ ]:
net_meta = build_model(meta=True)
REPORT = param_report(net_meta)
del net_meta
M = 1e6
print(f"total {REPORT['total']/M:,.1f}M | trunk {REPORT['trunk']/M:.1f}M | experts {REPORT['experts_total']/M:,.1f}M "
      f"= 6 x {REPORT['per_expert']/M:.1f}M")
print("active per token:", {m: f"{v/M:.1f}M" for m, v in REPORT["active"].items()},
      f"| capacity {MOE_CF} x 2 experts = {2*MOE_CF:.1f} active expert(s)/token")
if PRESET == "full":
    assert abs(REPORT["total"] - 1.68e9) / 1.68e9 < 0.05, "full preset drifted from the 1.2B+ / ~360M-active target"
    assert 330e6 < REPORT["active_max"] < 390e6, f"active params {REPORT['active_max']/M:.0f}M"
need_gb = REPORT["total"] * 16 / 2 ** 30
print(f"fp32 weights + grads + AdamW ~ {need_gb:.1f} GiB before activations | device has {VRAM_GB:.0f} GiB")
META_ONLY = _knob("META_ONLY", False) or SHARD_MODE == "encode"
if IS_CUDA and not META_ONLY and VRAM_GB < need_gb * 1.1:
    raise SystemExit(f"PRESET='{PRESET}' needs ~{need_gb:.0f} GiB of VRAM; this GPU has {VRAM_GB:.0f}. "
                     f"Use PRESET='tiny' or an A100/H100 (see the budget cell).")

In [ ]:
if not META_ONLY:
    net = build_model().to(dev)
    net.lay = {r: LAYOUTS[r].to(dev) for r in ROW_TYPES}
    # EVAL_ONLY must restore the WEIGHTS, not just the codebooks: `train()` is the only other place
    # that loads RESUME["net"], and EVAL_ONLY skips it - so without this an "evaluate the"
    # "checkpoint" run scores a freshly initialised model and prints numbers that look real.
    if EVAL_ONLY:
        if RESUME is not None:
            net.load_state_dict(RESUME["net"])
            print(f"EVAL_ONLY: weights restored from {CKPT_PATH} (step {RESUME['step']}, "
                  f"codes seen {sum(RESUME['seen'].values())/1e9:.4f}B)")
        else:
            print("!! EVAL_ONLY=1 but no checkpoint at CKPT_PATH: everything below runs on a RANDOM "
                  "model. Set NCM_CKPT_DIR to the folder that holds ckpt.pt.")
    torch._dynamo.config.cache_size_limit = 64

    def _try_flex():
        """Use flex-attention only if it compiles, matches SDPA, and is not slower."""
        from torch.nn.attention.flex_attention import create_block_mask
        for l in net.lay.values():
            bid = l.block_id
            l.flex_mask = create_block_mask(lambda b, h, q, kv, bid=bid: bid[kv] <= bid[q], None, None, l.T, l.T, device=dev)
        blk = net.blocks[0].attn
        r = "audio"
        x = torch.randn(2, net.lay[r].T, P["d"], device=dev, dtype=AMP_DTYPE)
        with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE):
            a = blk(x, net.lay[r], False).float()
            b = blk(x, net.lay[r], True).float()
        ok = torch.allclose(a, b, atol=5e-2, rtol=5e-2)
        for rr in ROW_TYPES:                                   # every packed geometry, forward AND backward: each compiles its own kernel
            xg = torch.randn(2, net.lay[rr].T, P["d"], device=dev, dtype=AMP_DTYPE, requires_grad=True)
            with torch.enable_grad(), torch.autocast("cuda", dtype=AMP_DTYPE):
                blk(xg, net.lay[rr], True).float().sum().backward()
        net.zero_grad(set_to_none=True)
        def bench(use):
            for _ in range(3):
                blk(x, net.lay[r], use)
            torch.cuda.synchronize(); t = time.time()
            for _ in range(10):
                blk(x, net.lay[r], use)
            torch.cuda.synchronize()
            return (time.time() - t) / 10
        with torch.no_grad(), torch.autocast("cuda", dtype=AMP_DTYPE):
            ts, tf = bench(False), bench(True)
        print(f"attention: sdpa {ts*1e3:.2f} ms | flex {tf*1e3:.2f} ms | parity {ok}")
        return ok and (ATTN == "flex" or tf <= ts * 1.05)

    net.use_flex = False
    _smem_kb = getattr(torch.cuda.get_device_properties(0), "shared_memory_per_block_optin", 0) // 1024 if IS_CUDA else 0   # missing on ROCm
    if IS_CUDA and HAS_FLEX and ATTN == "auto" and _smem_kb < 160:
        print(f"flex_attention skipped: {_smem_kb} KB shared memory per block (A100-class has 163); using sdpa. ATTN=flex forces it.")
    elif IS_CUDA and HAS_FLEX and ATTN in ("auto", "flex"):
        try:
            net.use_flex = _try_flex()
        except Exception as e:
            print("flex_attention unavailable:", type(e).__name__, str(e)[:80])
    print("attention backend:", "levels (one unmasked flash call per level)" if ATTN == "levels" else ("flex (block mask)" if net.use_flex else "sdpa (bool mask)"),
          "| expert dispatch:", MOE_DISPATCH)

    COMPILED = bool(COMPILE and IS_CUDA)
    loss_fn = torch.compile(net.loss, dynamic=False, mode=COMPILE_MODE) if COMPILED else net.loss
    print(f"model on {dev} | {sum(p.numel() for p in net.parameters())/M:,.1f}M params | torch.compile {COMPILED}")

## 11. Budget: tiers, steps, throughput, ETA (printed before anything expensive runs)

Compute-optimal is ~20 tokens per active parameter (~7B here); capable small models are trained 10-100x
past that. A "live code" is one predicted (non-prefix) position.

| tier | live codes | text | image | audio | tokens / active param | what it buys |
|---|---|---|---|---|---|---|
| T0 smoke | 0.02B | 40% | 35% | 25% | 0.06 | pipeline works, loss moves |
| T1 pilot | 2B | | | | 5.5 | per-modality CE separates, coarse levels sensible |
| **T2 foundation-lite (default)** | **20B** | 8B | 7B | 5B | 55 | readable text, recognisable image structure, intelligible TTS |
| T3 foundation | 50B | 20B | 18B | 12.5B | 140 | still far below the 1-4T tokens published ~360M text LMs saw |

Wall-clock is `6 * N_active * D / sustained FLOPS`: ~3.6 days on an A100 at 45% MFU, ~1.3 days on an
H100 for T2. The measured numbers below replace these estimates.

In [ ]:
TIERS = {"T0": 0.02, "T1": 2.0, "T2": 20.0, "T3": 50.0}
TOK_B = TOKENS or TIERS[TIER]
if TOKENS and abs(float(TOKENS) - TIERS[TIER]) > 1e-9:
    # The two knobs contradict each other: a truthy TOKENS wins and TIER is only a label.
    # That is how a "TIER=T1" run quietly became a 0.3B-code run (297 steps instead of 1,953).
    print(f"  !! TOKENS={TOKENS}B overrides TIER={TIER} ({TIERS[TIER]}B): this run is {TOK_B:.3f}B live "
          f"codes. Set NCM_TOKENS=0 to take the budget from the tier.")
AVG_LIVE_MICRO = sum(TYPE_P[r] * MICRO_B[r] * LAYOUTS[r].Tt for r in ROW_TYPES)
ACCUM = max(1, round(BATCH_CODES / AVG_LIVE_MICRO))
CODES_PER_STEP = ACCUM * AVG_LIVE_MICRO
STEPS_TOTAL = STEPS or max(1, round(TOK_B * 1e9 / CODES_PER_STEP))
TOK_B = STEPS_TOTAL * CODES_PER_STEP / 1e9

rows_per_code = {r: 1.0 / LAYOUTS[r].Tt for r in ROW_TYPES}
tgt = {"text": TOK_B * CODE_SHARE["text"], "image": TOK_B * CODE_SHARE["image"], "audio": TOK_B * CODE_SHARE["audio"]}
n_text = tgt["text"] * 1e9 / SPEC.text / (1)                       # approx rows of text (live codes per row = ladder total)
print(f"tier {TIER}: {TOK_B:.3f}B live codes = {STEPS_TOTAL:,} optimizer steps x {CODES_PER_STEP:,.0f} codes "
      f"({ACCUM} micro-batches of ~{MICRO_TOKENS} packed positions)")
print(f"tokens per active parameter: {TOK_B*1e9/REPORT['active_max']:.1f} (compute-optimal ~20)")
for m in ("text", "image", "audio"):
    live = LAYOUTS["text0" if m == "text" else m].Tt
    rows = tgt[m] * 1e9 / live
    raw = {"text": f"~{rows * SPEC.text / 1e9:.1f} GB of text", "image": f"~{rows/1e6:.2f}M images",
           "audio": f"<= {rows * SPEC.wave_len / SPEC.sr / 3600:,.0f} h of audio (incl. silence padding)"}[m]
    print(f"  {m:5s} {tgt[m]:7.3f}B live codes = {rows/1e6:8.2f}M rows  ({raw})")
print("row-type mix (rows):", {r: f"{TYPE_P[r]:.3f}" for r in ROW_TYPES})

FLOPS_TOTAL = 6 * REPORT["active_max"] * TOK_B * 1e9 * (sum(TYPE_P[r] * LAYOUTS[r].T for r in ROW_TYPES)
                                                         / sum(TYPE_P[r] * LAYOUTS[r].Tt for r in ROW_TYPES))
print(f"training FLOPs (6 N_active x packed tokens): {FLOPS_TOTAL:.2e}")
for nm, tf in (("A100 @45% MFU", 140), ("H100 bf16 @40%", 400), ("RTX 3070 (measured below)", None)):
    if tf:
        print(f"  {nm:26s} ~{FLOPS_TOTAL / (tf * 1e12) / 3600:7.1f} h")

In [ ]:
def synth(rtype, B):
    lay = LAYOUTS[rtype]
    codes = [torch.randint(1, 257, (B, n), device=dev) for n in lay.lens]
    cond = torch.randint(1, 257, (B, lay.cond), device=dev) if lay.cond else None
    return cond, codes


def _amp():
    return torch.autocast("cuda", dtype=AMP_DTYPE) if (IS_CUDA and BF16) else torch.autocast("cpu", enabled=False)


def bench_compute(iters=3):
    """Measured fwd+bwd time per micro-batch of each row type -> blended live codes/s, packed tokens/s, MFU."""
    net.train()
    t_of = {}
    for r in ROW_TYPES:
        B = MICRO_B[r]
        cond, codes = synth(r, B)
        for i in range(iters + 2):
            if i == 2:
                if IS_CUDA: torch.cuda.synchronize()
                t0 = time.time()
            with _amp():
                loss, _ = loss_fn(r, cond, codes)
            loss.backward()
        if IS_CUDA: torch.cuda.synchronize()
        t_of[r] = (time.time() - t0) / iters
        net.zero_grad(set_to_none=True)
    t_avg = sum(TYPE_P[r] * t_of[r] for r in ROW_TYPES)
    live_s = AVG_LIVE_MICRO / t_avg
    tok_s = sum(TYPE_P[r] * MICRO_B[r] * LAYOUTS[r].T for r in ROW_TYPES) / t_avg
    tflops = 6 * REPORT["active_max"] * tok_s / 1e12
    return t_of, live_s, tok_s, tflops


if not META_ONLY and not EVAL_ONLY:                 # no benchmark in the eval notebook
    t_of, LIVE_PER_S, TOK_PER_S, TFLOPS = bench_compute()
    mfu = f" | MFU {100 * TFLOPS / PEAK_TFLOPS:.0f}% of ~{PEAK_TFLOPS} peak" if PEAK_TFLOPS else ""
    print("seconds per micro-batch:", {r: round(t, 3) for r, t in t_of.items()})
    print(f"compute: {LIVE_PER_S:,.0f} live codes/s | {TOK_PER_S:,.0f} packed tokens/s | {TFLOPS:.1f} TFLOPS{mfu}")
    if IS_CUDA:
        print(f"peak memory {torch.cuda.max_memory_allocated() / 2**30:.1f} GiB of {VRAM_GB:.0f}")
    eta_h = TOK_B * 1e9 / LIVE_PER_S / 3600
    need_rows = LIVE_PER_S / (sum(TYPE_P[r] * LAYOUTS[r].Tt for r in ROW_TYPES))
    print(f"==> {TOK_B:.3f}B codes would take ~{eta_h:,.1f} h on this device if the feed keeps up; "
          f"the feed must deliver ~{need_rows:,.0f} rows/s ({WORKERS} workers -> ~{need_rows / max(WORKERS, 1):.1f} rows/s each)")

## 12. Correctness: fails loudly and cheaply, before any streamed token is spent

1. **No level leakage.** Perturbing the codes of level *j* leaves the hidden state of every slot at
   levels <= *j* unchanged and moves level *j+1* (so the model cannot copy its own targets).
2. **The prefix is a real condition.** Perturbing the prompt/caption/transcript moves level 0, and
   perturbing targets never moves the prefix.
3. **Modality isolation.** A text-only batch sends exactly zero gradient into the image and audio experts.
4. **Expert capacity.** Coverage stats: every token gets 0, 1 or 2 experts, about 1 on average.
5. **Kernel parity.** Triton SwiGLU vs reference, chunked CE vs `F.cross_entropy`, flex vs SDPA (printed above).
6. **Codec round trip** and an initial loss at the uniform-prediction value `ln(257)`.

In [ ]:
def _fresh_codes(rtype, B, seed):
    g = torch.Generator(device=dev).manual_seed(seed)
    lay = LAYOUTS[rtype]
    codes = [torch.randint(1, 257, (B, n), device=dev, generator=g) for n in lay.lens]
    cond = torch.randint(1, 257, (B, lay.cond), device=dev, generator=g) if lay.cond else None
    return cond, codes


@torch.no_grad()
def check_no_leak(net, rtype):
    lay = LAYOUTS[rtype]
    cond, codes = _fresh_codes(rtype, 2, 1)
    h0 = net.hidden(rtype, cond, codes)
    worst = {}
    for j in range(lay.K):
        codes2 = [c.clone() for c in codes]
        codes2[j] = (codes2[j] % 256) + 1                                  # every code at level j changes
        h1 = net.hidden(rtype, cond, codes2)
        upto = lay.lvl[j][1]
        same = (h0[:, :upto] - h1[:, :upto]).abs().max().item()            # prefix + levels <= j
        assert same < 1e-5, f"{rtype}: perturbing level {j} moved a slot at level <= {j} by {same:.2e} (LEAK)"
        if j + 1 < lay.K:
            a, b = lay.lvl[j + 1]
            moved = (h0[:, a:b] - h1[:, a:b]).abs().max().item()
            assert moved > 1e-4, f"{rtype}: level {j+1} ignores level {j} ({moved:.2e})"
            worst[j] = round(moved, 3)
    if lay.cond:
        cond2 = (cond % 256) + 1
        h1 = net.hidden(rtype, cond2, codes)
        a, b = lay.lvl[0]
        moved = (h0[:, a:b] - h1[:, a:b]).abs().max().item()
        assert moved > 1e-4, f"{rtype}: level 0 ignores the prefix"
    return worst


if not META_ONLY:
    net.eval()
    for r in ROW_TYPES:
        w = check_no_leak(net, r)
        print(f"  no-leak ok: {r:6s} level->level influence { w }")

    # (6) initial loss: heads start at zero, so the first prediction is uniform and the loss is ln(257)
    net.eval()
    with torch.no_grad():
        l0, _ = net.loss("text0", *_fresh_codes("text0", 2, 2))
    assert abs(l0.item() - math.log(257)) < 0.05, f"init loss {l0.item():.3f} != ln(257)={math.log(257):.3f}"
    print(f"  init loss {l0.item():.3f} ~ ln(257) = {math.log(257):.3f}")

    # (3) modality isolation. A zero head sends exactly zero gradient into the trunk on step 0, so the test
    # swaps in a small random head, checks the gradient, and puts the zero head back.
    saved = {k: v.clone() for k, v in net.heads.state_dict().items()}
    for hd in net.heads.values():
        nn.init.normal_(hd.weight, std=0.02)
    net.train(); net.zero_grad(set_to_none=True)
    cond, codes = _fresh_codes("text0", 2, 2)
    loss, _ = net.loss("text0", cond, codes)
    loss.backward()
    g = {m: sum(float(p.grad.abs().sum()) if p.grad is not None else 0.0
                for blk in net.blocks for p in blk.experts[m].parameters()) for m in ("text", "image", "audio")}
    assert g["text"] > 0 and g["image"] == 0 and g["audio"] == 0, f"modality isolation broken: {g}"
    print("  isolation ok: text0 batch ->", {m: ("grad" if v > 0 else "zero grad") for m, v in g.items()})
    net.zero_grad(set_to_none=True)
    net.heads.load_state_dict(saved)

    # (4) coverage
    net.eval()
    for blk in net.blocks:
        for e in blk.experts.values():
            e.record = True
    with torch.no_grad():
        net.hidden("image", *_fresh_codes("image", 4, 3)[:1], _fresh_codes("image", 4, 3)[1])
    cov = {m: net.blocks[-1].experts[m].last_cov for m in ("text", "image")}
    for blk in net.blocks:
        for e in blk.experts.values():
            e.record = False
    for m, c in cov.items():
        if c is not None:
            mean = float(c[1] + 2 * c[2])
            print(f"  coverage (last block, {m} group): 0 experts {c[0]:.2f} | 1 expert {c[1]:.2f} | 2 experts {c[2]:.2f} | mean {mean:.2f}")
            assert 0.9 < mean < 1.3

    # (5) kernels
    h = torch.randn(64, 2 * 48, device=dev, dtype=torch.float32, requires_grad=True)
    ref = swiglu_torch(h)
    assert torch.allclose(swiglu(h), ref, atol=1e-4, rtol=1e-4)
    W, b_ = torch.randn(257, 32, device=dev), torch.randn(257, device=dev)
    hh, tt = torch.randn(1000, 32, device=dev), torch.randint(1, 257, (1000,), device=dev)
    _tf32 = torch.backends.cuda.matmul.allow_tf32      # TF32 rounds the two matmuls differently (~1e-2), which hides real bugs
    torch.backends.cuda.matmul.allow_tf32 = False
    try:
        ce = fused_ce(hh, W, b_, tt, chunk=300)[0]
        ref_ce = F.cross_entropy(F.linear(hh.double(), W.double(), b_.double()), tt, reduction="none")
        err = (ce.double() - ref_ce).abs().max().item()
        assert err < 1e-3, f"chunked CE differs from reference by {err:.2e}"
    finally:
        torch.backends.cuda.matmul.allow_tf32 = _tf32
    print(f"  kernels ok: swiglu[{SWIGLU_IMPL}] == reference, chunked CE == F.cross_entropy")

    # (6) codecs
    txt = torch.randint(0, 256, (2, SPEC.text), dtype=torch.uint8, device=dev)
    assert TextCodec.decode(codecs.text.encode(txt)[-1])[0] == bytes(txt[0].tolist()).rstrip(b"\x00").decode("utf-8", "replace")
    print("  codec round trip ok (text exact); sizes:", {r: LAYOUTS[r].Tt for r in ROW_TYPES})
    # image / audio tokenizers: right shapes, codes in 1..256, decode gives the right size at every level (works for k-means and learned)
    _img = torch.randint(0, 256, (2, 3, SPEC.side, SPEC.side), dtype=torch.uint8, device=dev)
    _ci = codecs.image.encode(_img)
    assert [c.shape[1] for c in _ci] == LAYOUTS["image"].lens and all(1 <= int(c.min()) and int(c.max()) <= 256 for c in _ci), "image codes: wrong shape or range"
    if getattr(codecs.image, "decodable", True):                      # the DINO feature codec has no pixel decoder
        assert all(codecs.image.decode(_ci, level=k).shape == _img.shape for k in range(LAYOUTS["image"].K)), "image decode: wrong size"
    _wv = torch.randn(2, SPEC.wave_len, device=dev) * 0.1
    _ca = codecs.audio.encode(_wv)
    assert [c.shape[1] for c in _ca] == LAYOUTS["audio"].lens and all(1 <= int(c.min()) and int(c.max()) <= 256 for c in _ca), "audio codes: wrong shape or range"
    if hasattr(codecs.audio, "decode_mel"):
        assert all(codecs.audio.decode_mel(_ca, level=k).shape == (2, SPEC.audio, SPEC.n_mels) for k in range(LAYOUTS["audio"].K)), "audio decode_mel: wrong shape"
    else:                                                            # the DAC codec decodes straight to a waveform
        assert all(codecs.audio.decode(_ca, level=k).shape == (2, SPEC.wave_len) for k in range(LAYOUTS["audio"].K)), "audio decode: wrong size"
    print(f"  tokenizer round trip ok ({TOKENIZER}): image {[tuple(c.shape[1:]) for c in _ci]}, audio {[tuple(c.shape[1:]) for c in _ca]}")

    # (7) the optional speed paths must equal the reference: level-wise attention vs masked SDPA, flat expert dispatch vs gather/scatter
    _a0, _m0 = ATTN, MOE_DISPATCH
    try:
        net.eval()
        for r in ROW_TYPES:
            lay = net.lay[r]
            x = torch.randn(2, lay.T, P["d"], device=dev, dtype=AMP_DTYPE)
            with torch.no_grad(), _amp():
                ATTN = "sdpa";   a_ref = net.blocks[0].attn(x, lay, False).float()
                ATTN = "levels"; a_lv = net.blocks[0].attn(x, lay, False).float()
                ATTN = _a0
                MOE_DISPATCH = "gather"; m_ref = sum(net.blocks[0].experts[g](x, s) for g, s in lay.segs.items()).float()
                MOE_DISPATCH = "flat";   m_fl = sum(net.blocks[0].experts[g](x, s) for g, s in lay.segs.items()).float()
            ea, em = (a_ref - a_lv).abs().max().item(), (m_ref - m_fl).abs().max().item()
            assert ea < 5e-2, f"{r}: level-wise attention differs from masked SDPA by {ea:.2e}"
            assert em < 5e-2, f"{r}: flat expert dispatch differs from gather/scatter by {em:.2e}"
        print(f"  speed paths ok: ATTN='levels' == masked SDPA, MOE_DISPATCH='flat' == gather/scatter (last max abs err {ea:.1e} / {em:.1e}); active: ATTN={_a0}, MOE_DISPATCH={_m0}")
    finally:
        ATTN, MOE_DISPATCH = _a0, _m0
        net.train()
    print("\nALL CORRECTNESS CHECKS PASSED")
    net.train()

## 12b. Profile and A/B the speed levers (optional: `PROFILE=1`)

Where does a training step spend its GPU time, and do the optional paths (`ATTN="levels"`, `MOE_DISPATCH="flat"`) help on THIS GPU?
It profiles forward+backward per row type (kernel time grouped into attention / matmul / swiglu / dispatch+copies / optimizer / other),
then times the attention variants and the two dispatch variants on the real micro-batch shapes and prints the knobs to set.
Nothing here changes the model; run it once on a new GPU before a long run.

In [ ]:
if not META_ONLY and PROFILE and IS_CUDA:
    import contextlib
    from torch.profiler import profile, ProfilerActivity

    def _cat(name):
        n = name.lower()
        if any(s in n for s in ("flash", "fmha", "attention", "sdpa", "flex", "efficient", "mha")):
            return "attention"
        if any(s in n for s in ("swiglu", "silu")):
            return "swiglu"
        if any(s in n for s in ("gemm", "cutlass", "cublas", "xmma", "nvjet", "ampere_", "hopper_", "sm80_", "sm90_", "sm100", "bmm")):
            return "matmul"
        if any(s in n for s in ("index", "gather", "scatter", "topk", "sort", "cat", "copy", "permute", "transpose", "memcpy")):
            return "dispatch / copies"
        if any(s in n for s in ("adam", "foreach", "multi_tensor")):
            return "optimizer"
        return "other (norms, elementwise, CE, ...)"

    def _kernel_us(e):
        return getattr(e, "self_device_time_total", None) or getattr(e, "self_cuda_time_total", 0) or 0

    def _profile_row(rtype, steps=3):
        B = MICRO_B[rtype]
        cond, codes = synth(rtype, B)
        net.train()
        for _ in range(2):                                           # warm-up (compile, autotune)
            with _amp(): loss, _ = loss_fn(rtype, cond, codes)          # bf16 autocast, exactly as the training step runs it
            loss.backward()
        net.zero_grad(set_to_none=True); torch.cuda.synchronize()
        with profile(activities=[ProfilerActivity.CUDA]) as prof:
            for _ in range(steps):
                with _amp(): loss, _ = loss_fn(rtype, cond, codes)
                loss.backward()
            torch.cuda.synchronize()
        net.zero_grad(set_to_none=True)
        by, top = {}, []
        for e in prof.key_averages():
            t = _kernel_us(e)
            if t:
                by[_cat(e.key)] = by.get(_cat(e.key), 0) + t
                top.append((t, e.key))
        tot = sum(by.values()) or 1
        print(f"  {rtype:6s} GPU kernel time per micro-batch {tot / steps / 1e3:7.1f} ms | " +
              " | ".join(f"{k} {100 * v / tot:.0f}%" for k, v in sorted(by.items(), key=lambda kv: -kv[1])))
        return sorted(top, reverse=True)[:60]

    print("=== where the time goes (forward + backward, one micro-batch per row type) ===")
    tops = {r: _profile_row(r) for r in ROW_TYPES}
    for t, k in tops["image"][:6]:
        print(f"    image top kernel: {t / 3e3:7.2f} ms  {k[:90]}")
    for r_ in ("image", "audio"):                                     # what is hiding inside the copy bucket?
        print(f"    {r_}: biggest kernels in the dispatch / copies bucket:")
        for t, k in [x for x in tops[r_] if _cat(x[1]) == "dispatch / copies"][:8]:
            print(f"      {t / 3e3:7.2f} ms  {k[:110]}")

    def _time(fn, n=5):
        for _ in range(2):
            fn()
        torch.cuda.synchronize(); t0 = time.time()
        for _ in range(n):
            fn()
        torch.cuda.synchronize()
        return (time.time() - t0) / n * 1e3

    _a0, _m0 = ATTN, MOE_DISPATCH
    print("\n=== attention variants, one layer fwd+bwd (ms) ===")
    best_attn = {}
    for r in ROW_TYPES:
        lay, B = net.lay[r], MICRO_B[r]
        x0 = torch.randn(B, lay.T, P["d"], device=dev, dtype=AMP_DTYPE)
        res = {}
        from torch.nn.attention import sdpa_kernel, SDPBackend
        for name, use_flex in (("sdpa", False), ("flex", True), ("levels", False), ("levels+cudnn", False), ("levels+flash", False)):
            if name == "flex" and not getattr(net, "use_flex", False):
                continue
            ATTN = "levels" if name.startswith("levels") else "sdpa"
            _be = {"levels+cudnn": SDPBackend.CUDNN_ATTENTION, "levels+flash": SDPBackend.FLASH_ATTENTION}.get(name)
            def f(use_flex=use_flex, _be=_be):
                xx = x0.detach().requires_grad_(True)
                with _amp(), (sdpa_kernel(_be) if _be else contextlib.nullcontext()):
                    y = net.blocks[0].attn(xx, lay, use_flex)
                y.float().sum().backward()
            try:
                res[name] = _time(f)
            except Exception as e:
                print(f"    {r} {name}: failed ({type(e).__name__})")
        ATTN = _a0
        best_attn[r] = min(res, key=res.get)
        print(f"  {r:6s} " + " | ".join(f"{k} {v:.2f}" for k, v in res.items()) + f"   -> best {best_attn[r]}")

    print("\n=== expert dispatch, one MoE layer fwd+bwd (ms) ===")
    flat_wins = 0
    for r in ROW_TYPES:
        lay, B, m = net.lay[r], MICRO_B[r], ROW_MOD[r]
        x0 = torch.randn(B, lay.T, P["d"], device=dev, dtype=AMP_DTYPE)
        res = {}
        for name in ("gather", "flat"):
            MOE_DISPATCH = name
            def f():
                xx = x0.detach().requires_grad_(True)
                with _amp():
                    y = net.blocks[0].experts[m](xx, lay.segs[m])
                y.float().sum().backward()
            res[name] = _time(f)
        MOE_DISPATCH = _m0
        flat_wins += res["flat"] < res["gather"]
        print(f"  {r:6s} gather {res['gather']:.2f} | flat {res['flat']:.2f}")
    net.zero_grad(set_to_none=True)
    ATTN, MOE_DISPATCH = _a0, _m0
    pick_attn = max(set(best_attn.values()), key=list(best_attn.values()).count)
    print(f"\nper-layer timing favours: ATTN={pick_attn!r}, MOE_DISPATCH={'flat' if flat_wins >= 3 else 'gather'!r}  "
          f"(current: ATTN={ATTN!r}, MOE_DISPATCH={MOE_DISPATCH!r}). Differences under ~5% are noise.\n"
          f"  NOTE: this times ONE layer. On an RTX PRO 6000 it picked ATTN='sdpa' but a real run was 13% slower with it "
          f"(62k vs 72k codes/s): confirm any attention change with a short training run before adopting it.")

## 13. Train

* One optimizer step = `ACCUM` micro-batches (each a single row type, a fixed token budget) -> ~`BATCH_CODES` live codes.
* AdamW (fused), beta (0.9, 0.95), wd on matrices only, linear warmup then cosine to 10%, grad-clip, bf16 autocast.
* Logged every `LOG_EVERY` steps: loss, per-(modality, level) CE, tokens/s, TFLOPS/MFU, and the **share of the step spent
  waiting for data**, which tells a feed-bound run from a compute-bound one.
* Held-out evaluation every `EVAL_EVERY_B` billion codes against the **constant-predictor baseline**: the cross-entropy of the
  held-out codes under the *training* unigram of that (modality, level). A score under it is not a result.
* Checkpoints (weights, optimizer, step, codes seen, codebooks) go to `CKPT_DIR` every `CKPT_MIN` minutes and a watchdog saves
  and stops before `SESSION_HOURS`. Data is never saved; a resume re-streams with a fresh shuffle seed ("approximately resume").
* Early abort: if the loss has not fallen 5% by step 200, stop instead of burning the budget.
* **Caption dropout** (`CAP_DROP`): a fraction of image and audio rows train with an empty caption/transcript, so at sampling time
  the model can be pushed away from "no caption" toward the caption (classifier-free guidance, `cfg` in the generate cell).

In [ ]:
HIST = []
UNI = {}          # (modality, level) -> running histogram of TRAINING target codes, the honest constant predictor


def make_optimizer():
    dec, nodec = [], []
    for n, p_ in net.named_parameters():
        (dec if p_.ndim >= 2 and not n.startswith(("emb", "pos")) and "router" not in n else nodec).append(p_)
    return torch.optim.AdamW([dict(params=dec, weight_decay=WD), dict(params=nodec, weight_decay=0.0)],
                             lr=LR, betas=(0.9, 0.95), eps=1e-8, fused=IS_CUDA)


WARM = min(WARMUP, max(1, STEPS_TOTAL // 5))
def lr_at(step):
    if step < WARM:
        return LR * (step + 1) / WARM
    prog = min(1.0, (step - WARM) / max(1, STEPS_TOTAL - WARM))
    return LR * (0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * prog)))


@torch.no_grad()
def evaluate():
    """Teacher-forced held-out CE / top-1 per (modality, level) + the constant-predictor baseline."""
    net.eval()
    acc = {}
    for mb in VAL:
        r = mb["rtype"]; m = ROW_MOD[r]
        with _amp():
            h = net.hidden(r, mb["cond"], mb["codes"])
        for k in range(LAYOUTS[r].K):
            lg = net.level_logits(r, h, k)
            tg = mb["codes"][k]
            ce = F.cross_entropy(lg.reshape(-1, SPEC.V), tg.reshape(-1), reduction="none")
            a = acc.setdefault((m, k), dict(ce=0.0, ok=0.0, n=0, cnt=torch.zeros(SPEC.V, device=dev)))
            a["ce"] += float(ce.sum()); a["ok"] += float((lg.argmax(-1) == tg).sum()); a["n"] += tg.numel()
            a["cnt"] += torch.bincount(tg.reshape(-1), minlength=SPEC.V).float()
    out = {}
    for (m, k), a in acc.items():
        fv = a["cnt"] / a["cnt"].sum()                              # held-out code frequencies
        u = UNI.get((m, k))
        if u is not None and float(u.sum()) > 0:                    # constant predictor = training unigram, add-one smoothed
            lp = ((u + 1) / (u.sum() + SPEC.V)).log()
            base = float(-(fv * lp).sum())
        else:                                                       # nothing trained yet: held-out entropy (optimistic)
            base = float(-(fv[fv > 0] * fv[fv > 0].log()).sum())
        out[f"{m}_L{k}"] = dict(ce=a["ce"] / a["n"], acc=a["ok"] / a["n"], base=base, n=a["n"])
    net.train()
    return out


BEST = {"score": float("inf")}      # best harness-wide held-out CE so far (set in the eval block)


def save_ckpt(step, opt, seen, path=None):
    if not CKPT_DIR:
        return
    os.makedirs(CKPT_DIR, exist_ok=True)
    path = path or CKPT_PATH
    tmp = path + ".tmp"
    torch.save(dict(net=net.state_dict(), opt=opt.state_dict(), step=step, seen=seen, hist=HIST[-2000:],
                    codecs=codecs.state(), preset=PRESET), tmp)
    os.replace(tmp, path)
    print(f"  checkpoint -> {path} (step {step})", flush=True)


def fmt_val(v):
    mods = {}
    for key, d in v.items():
        mods.setdefault(key.split("_")[0], []).append(d)
    return " | ".join(f"{m} ce {np.mean([d['ce'] for d in ds]):.2f} (base {np.mean([d['base'] for d in ds]):.2f}, "
                      f"acc {100*np.mean([d['acc'] for d in ds]):.0f}%)" for m, ds in mods.items())


def train():
    global HIST
    net.train()
    opt = make_optimizer()
    seen, step0 = dict(text=0, image=0, audio=0), 0
    if RESUME is not None:
        net.load_state_dict(RESUME["net"]); opt.load_state_dict(RESUME["opt"])
        step0, seen, HIST = RESUME["step"], RESUME["seen"], RESUME["hist"]
        print(f"resumed at step {step0}, codes seen {seen}")
    eval_every = max(1, min(round(EVAL_EVERY_B * 1e9 / CODES_PER_STEP), max(1, STEPS_TOTAL // 3)))
    writer, stream = None, None
    if SHARD_MODE == "read" and SHARD_DIR:
        pf = ShardFeeder(SHARD_DIR, start_micro=step0 * ACCUM, seed=SEED, micro_left=(STEPS_TOTAL - step0) * ACCUM)
    else:
        if SHARD_MODE == "record" and SHARD_DIR:
            os.makedirs(SHARD_DIR, exist_ok=True)
            need, free = (STEPS_TOTAL - step0) * CODES_PER_STEP * 1.4 / 1e9, shutil.disk_usage(SHARD_DIR).free / 1e9
            if need > 0.9 * free:
                print(f"  !! shards would need ~{need:.0f} GB but only {free:.0f} GB are free: not recording this run")
            else:
                writer = ShardWriter(SHARD_DIR)
                print(f"recording shards to {SHARD_DIR} (~{need:.1f} GB for this run, {free:.0f} GB free)")
        ds = MixDataset(SPEC, part0=1, nparts=PARTS, seed=SEED + 1 + step0)
        stream = RowStream(ds, WORKERS)
        pf = Prefetcher(stream, codecs, writer=writer)
        print(f"streaming with {WORKERS} workers; first batch can take a minute while sources connect...", flush=True)
    t_run, t_ckpt, t_log, wait, steps_in_log = time.time(), time.time(), time.time(), 0.0, 0
    lvl_sum, lvl_n, loss_sum = {}, {}, torch.zeros((), device=dev)
    live_in_log, packed_in_log = 0, 0
    abort_at = 200 if STEPS_TOTAL >= 400 else None
    try:
        for step in range(step0, STEPS_TOTAL):
            for g in opt.param_groups:
                g["lr"] = lr_at(step)
            opt.zero_grad(set_to_none=True)
            for _ in range(ACCUM):
                t0 = time.time()
                mb = pf.get()
                wait += time.time() - t0
                cond = mb["cond"]
                if CAP_DROP > 0 and cond is not None and mb["rtype"] in ("image", "audio"):
                    blank = torch.rand(cond.shape[0], 1, device=cond.device) < CAP_DROP       # blank the caption/transcript in a fraction of rows,
                    cond = torch.where(blank, torch.ones_like(cond), cond)                      # so the model also learns "no caption" (= all padding, code 1)
                with _amp():
                    loss, per_level = loss_fn(mb["rtype"], cond, mb["codes"])
                (loss / ACCUM).backward()
                m = ROW_MOD[mb["rtype"]]
                loss_sum += loss.detach() / ACCUM
                lvl_sum[(m, mb["rtype"])] = lvl_sum.get((m, mb["rtype"]), 0) + per_level
                lvl_n[(m, mb["rtype"])] = lvl_n.get((m, mb["rtype"]), 0) + 1
                seen[m] += mb["n_live"]; live_in_log += mb["n_live"]
                for k_, c_ in enumerate(mb["codes"]):
                    UNI[(m, k_)] = UNI.get((m, k_), 0) + torch.bincount(c_.reshape(-1), minlength=SPEC.V).float()
                packed_in_log += len(mb["src"]) * LAYOUTS[mb["rtype"]].T
            gn = torch.nn.utils.clip_grad_norm_(net.parameters(), CLIP, foreach=True)
            opt.step()
            steps_in_log += 1
            done = step + 1
            if done % LOG_EVERY == 0 or done == STEPS_TOTAL or done == 1:
                if IS_CUDA: torch.cuda.synchronize()
                dt = time.time() - t_log
                loss_v = float(loss_sum) / steps_in_log
                ce = {}
                for (m, r), v in lvl_sum.items():
                    for k, x in enumerate((v / lvl_n[(m, r)]).tolist()):
                        ce.setdefault(f"{m}_L{k}", []).append(x)
                ce = {k: round(float(np.mean(v)), 4) for k, v in ce.items()}
                tok_s = packed_in_log / dt
                tfl = 6 * REPORT["active_max"] * tok_s / 1e12
                rec = dict(step=done, loss=loss_v, lr=lr_at(step), gnorm=float(gn), codes_s=live_in_log / dt,
                           tflops=tfl, wait=wait / dt, seen=dict(seen), ce=ce)
                HIST.append(rec)
                eta = (STEPS_TOTAL - done) * dt / steps_in_log / 3600
                mfu = f" ({100*tfl/PEAK_TFLOPS:.0f}% MFU)" if PEAK_TFLOPS else ""
                print(f"step {done}/{STEPS_TOTAL} | loss {loss_v:.3f} | lr {rec['lr']:.1e} | gn {rec['gnorm']:.2f} | "
                      f"{rec['codes_s']/1e3:.1f}k codes/s | {tfl:.1f} TFLOPS{mfu} | data-wait {100*rec['wait']:.0f}% | "
                      f"seen {sum(seen.values())/1e9:.4f}B | eta {eta:.1f}h", flush=True)
                if rec["wait"] > 0.5 and done > 3 and (done // LOG_EVERY) % 10 == 1:
                    print(f"  !! feed-bound: the GPU waits {100*rec['wait']:.0f}% of each step for data. "
                          f"Raise WORKERS (now {WORKERS}) or run where more CPU cores / bandwidth are available.", flush=True)
                lvl_sum, lvl_n, loss_sum = {}, {}, torch.zeros((), device=dev)
                live_in_log = packed_in_log = steps_in_log = 0
                wait, t_log = 0.0, time.time()
                if not math.isfinite(loss_v):
                    raise FloatingPointError("non-finite loss: stop and lower LR")
            if abort_at and done == abort_at:
                tr = [h["loss"] for h in HIST if "loss" in h and h["step"] <= done]   # val records have no "loss"
                if len(tr) >= 8 and np.mean(tr[-3:]) > 0.95 * np.mean(tr[:3]):
                    print("NOT LEARNING: loss fell <5% in 200 steps. Stop and check the CE table before spending the budget.")
                    print(HIST[-1]["ce"])
                    return False
            if done % eval_every == 0 or done == STEPS_TOTAL:
                v = evaluate()
                HIST.append(dict(step=done, val=v))
                print(f"  eval @ {done}: {fmt_val(v)}", flush=True)
                # Keep the best held-out model in its own file. The rolling checkpoint is
                # overwritten every CKPT_MIN minutes, which is how audio's best model (held-out
                # 1.52 at 0.73B codes) was replaced by a late overfit 1.78 as the only copy.
                sc = float(np.mean([d["ce"] for d in v.values()]))
                if sc < BEST["score"]:
                    BEST["score"] = sc
                    _best = CKPT_PATH.replace(".pt", "_best.pt")
                    save_ckpt(done, opt, seen, path=_best)
                    print(f"  best held-out so far: mean CE {sc:.3f} -> {_best}", flush=True)
            if CKPT_DIR and time.time() - t_ckpt > CKPT_MIN * 60:
                if writer is not None:
                    writer.flush_all()
                save_ckpt(done, opt, seen); t_ckpt = time.time()
            if SESSION_HOURS and time.time() - t_run > SESSION_HOURS * 3600:
                save_ckpt(done, opt, seen)
                print(f"watchdog: {SESSION_HOURS} h reached at step {done}. Checkpoint saved: re-run this notebook with the same "
                      f"CKPT_DIR to resume.")
                return True
        save_ckpt(STEPS_TOTAL, opt, seen)
    finally:
        pf.close()
        if stream is not None:
            stream.close()
        if writer is not None:
            writer.close()
    return True


if not META_ONLY and not EVAL_ONLY:                 # EVAL_ONLY: the eval notebook defines evaluate() and the helpers above but never trains
    TRAINED = train()

## 14. Did it learn? (verdict from the run's own records)

In [ ]:
def verdict():
    tr = [h for h in HIST if "loss" in h]
    va = [h for h in HIST if "val" in h]
    fails = []
    def need(ok, why):
        if not ok: fails.append(why)
    need(len(tr) >= 1, "no logged steps")
    need(all(math.isfinite(h["loss"]) and math.isfinite(h["gnorm"]) for h in tr), "non-finite loss or grad norm")
    short = len(tr) < 6
    if not short:
        n = max(1, len(tr) // 5)
        early, late = np.median([h["loss"] for h in tr[:n]]), np.median([h["loss"] for h in tr[-n:]])
        drop = 100 * (1 - late / early)
        print(f"loss  median {early:.3f} -> {late:.3f} ({drop:+.1f}%; ln(257) at init = {math.log(257):.3f})")
        need(late <= 0.83 * early, f"loss moved only {drop:.1f}%, want >= 17%")
        per = {}
        for h in tr:
            for k, v in h["ce"].items():
                per.setdefault(k.split("_")[0], {}).setdefault(k, []).append(v)
        moved = 0
        for m, ks in per.items():
            first = np.mean([v[0] for v in ks.values()]); last = np.mean([v[-1] for v in ks.values()])
            print(f"  {m:5s} train CE {first:.3f} -> {last:.3f} ({100*(1-last/first):+.1f}%)")
            moved += (last <= 0.9 * first)
        need(moved >= 2, f"only {moved}/3 modalities' CE fell >= 10%")
    if va:
        v = va[-1]["val"]
        beat = {}
        for key, d in v.items():
            beat.setdefault(key.split("_")[0], []).append(d["base"] - d["ce"])
        gain = {m: float(np.mean(x)) for m, x in beat.items()}
        print("held-out gain over the constant-predictor baseline (nats, mean over levels):", {m: round(g, 3) for m, g in gain.items()})
        if not short:
            need(sum(g > 0.05 for g in gain.values()) >= 2, f"held-out CE does not beat the constant baseline for 2+ modalities: {gain}")
    print()
    if short:
        print(f"VERDICT: TOO SHORT to judge ({len(tr)} logged steps) | pipeline ran end to end, loss finite" if not fails else f"VERDICT: BROKEN {fails}")
    elif fails:
        print("VERDICT:", "NOT LEARNING" if late > 0.95 * early else "PARTIAL")
        for f in fails: print("  -", f)
    else:
        print(f"VERDICT: PASS | loss {drop:.1f}% down | seen {sum(tr[-1]['seen'].values())/1e9:.3f}B live codes")
    try:
        import matplotlib.pyplot as plt
        fig, ax = plt.subplots(1, 2, figsize=(11, 3.2))
        ax[0].plot([h["step"] for h in tr], [h["loss"] for h in tr]); ax[0].set_title("train loss"); ax[0].set_xlabel("step")
        for m in ("text", "image", "audio"):
            ys = [np.mean([v for k, v in h["ce"].items() if k.startswith(m)]) for h in tr]
            ax[1].plot([h["step"] for h in tr], ys, label=m)
        ax[1].legend(); ax[1].set_title("train CE per modality"); ax[1].set_xlabel("step"); plt.tight_layout(); plt.show()
    except Exception:
        pass

if not META_ONLY:
    verdict()

## 14b. Did it learn? (the evidence the verdict cell cannot give)

The verdict cell asks whether the loss fell. A falling loss can still be memorisation, a prefix the model ignores, or a codec whose
codes are unpredictable. These checks use the held-out set (never trained on) and the codec checks from section 7d:

1. **Held-out gain per modality and level.** CE against the constant predictor (the training unigram of that level), plus top-1
   accuracy against the unigram's. A level counts as *learned* when its CE gain is above 0.02 nats (accuracy is shown, not used: a skewed
   unigram such as silence makes top-1 a poor test); a modality needs a positive finest-level gain and a positive slot-weighted gain. Next to it, the **probe ceiling** from the codec
   check: how much a small net could win at that level from the previous level alone. A model gain far below a positive probe gain
   points at the model / data / training length; a probe gain of ~0 points at the codec.
2. **Does it use the prefix?** CE with the true caption / prompt / transcript against a shuffled one and a blank one. If the three
   are equal the conditioning is ignored.
3. **Does each level use the level before it?** CE with the real previous-level codes against random ones.
4. **Train against held-out CE.** A held-out CE far above the training CE is memorisation.

The final line per modality is `LEARNED`, `PARTIAL` or `NOT LEARNING`, with the reason.

In [ ]:
LEARNED = {}


@torch.no_grad()
def _ce_levels(mb, cond, codes_in):
    """held-out CE of every level of one micro-batch, with `cond` / `codes_in` as the (possibly corrupted) inputs."""
    r = mb["rtype"]
    with _amp():
        h = net.hidden(r, cond, codes_in)
    return [float(F.cross_entropy(net.level_logits(r, h, k).reshape(-1, SPEC.V).float(), mb["codes"][k].reshape(-1)))
            for k in range(LAYOUTS[r].K)]


def _wavg(items):
    tot = sum(n for _, n in items)
    return [sum(c[k] * n for c, n in items) / tot for k in range(len(items[0][0]))]


def learned_report():
    net.eval()
    v = evaluate()
    mods = {}
    for key, d in v.items():
        m, k = key.split("_L")
        mods.setdefault(m, {})[int(k)] = d
    # ---- 1. held-out gain over the constant predictor
    print("1. held-out CE vs the constant-predictor baseline (nats), top-1 accuracy vs the baseline's, and the codec probe ceiling")
    summary = {}
    for m, ks in mods.items():
        print(f"  {m}: level |   CE  | baseline |  gain  |  acc  | base acc | probe ceiling")
        gains, ws, n_ok = [], [], 0
        for k in sorted(ks):
            d = ks[k]
            gain = d["base"] - d["ce"]
            u = UNI.get((m, k))
            bacc = float(u.max() / u.sum()) if u is not None and float(u.sum()) > 0 else float("nan")
            pr = PROBE.get((m, k)) if "PROBE" in globals() else None
            ceil = f"{pr['gain_bits'] * math.log(2):+.3f}" if pr else "  n/a"
            ok = gain > 0.02                                                  # CE is the stable metric; top-1 is printed but a skewed unigram (silence) makes it a poor test
            n_ok += ok; gains.append(gain); ws.append(SPEC.lens(m)[k])
            print(f"      L{k}   {d['ce']:6.3f}  {d['base']:7.3f}   {gain:+.3f}  {d['acc']:.3f}   {bacc:.3f}    {ceil}   {'learned' if ok else ''}")
        summary[m] = dict(levels=len(ks), n_ok=n_ok, gain_finest=gains[-1], gain_mean=float(np.mean(gains)),
                          gain_w=float(np.average(gains, weights=ws)),         # weighted by slots: the finest level carries most of the live codes
                          ce=float(np.mean([ks[k]["ce"] for k in ks])))
    steps_done = max([h["step"] for h in HIST if "loss" in h] or [0])
    if steps_done < 200:
        print(f"\nNOTE: only {steps_done} optimizer steps. The model starts at CE ~ ln(257) = 5.55, above the low-entropy baselines of the coarse levels,\n"
              "and needs hundreds of steps to cross them: read a run this short as a pipeline check, not as a verdict.")
    # ---- 2. the prefix
    print("\n2. does it use the prefix? held-out CE (mean over levels) with the true prefix, a shuffled one and a blank one")
    pref, uses_prefix = {}, {}
    # 0.01 nats printed "uses the prefix" for image and text in the 0.3B-code run when the true
    # reading of the same numbers was that they ignored it. 0.05 is the honest bar.
    PREFIX_MIN = 0.05
    for mb in VAL:
        if mb["cond"] is None:
            continue
        n = len(mb["src"])
        t, s, b = (_ce_levels(mb, c, mb["codes"]) for c in (mb["cond"], mb["cond"].roll(1, 0), torch.ones_like(mb["cond"])))
        pref.setdefault(mb["rtype"], []).append(((t, s, b), n))
    for r, items in pref.items():
        T, S, B_ = (_wavg([(x[i], n) for x, n in items]) for i in range(3))
        delta = [s - t for s, t in zip(S, T)]
        uses_prefix[r] = max(delta) > PREFIX_MIN
        print(f"  {r:6s} true {np.mean(T):.3f} | shuffled {np.mean(S):.3f} | blank {np.mean(B_):.3f} | shuffled - true per level {[round(x, 3) for x in delta]}"
              f"  -> {'uses the prefix' if uses_prefix[r] else 'IGNORES the prefix'}")
    # ---- 3. the previous level
    print("\n3. does each level use the level before it? held-out CE with the real previous-level codes vs random ones (per level k >= 1)")
    g = torch.Generator(device=dev).manual_seed(0)
    ctx, uses_ctx = {}, {}
    for mb in VAL:
        r, n = mb["rtype"], len(mb["src"])
        real = _ce_levels(mb, mb["cond"], mb["codes"])
        rand = []
        for k in range(1, LAYOUTS[r].K):
            c2 = list(mb["codes"])
            c2[k - 1] = torch.randint(1, 257, c2[k - 1].shape, device=dev, generator=g)
            rand.append(_ce_levels(mb, mb["cond"], c2)[k])
        ctx.setdefault(r, []).append(((real[1:], rand), n))
    for r, items in ctx.items():
        Rl, Rn = (_wavg([(x[i], n) for x, n in items]) for i in range(2))
        delta = [b - a for a, b in zip(Rl, Rn)]
        uses_ctx[r] = max(delta) > 0.02
        print(f"  {r:6s} random - real per level 1..{len(delta)}: {[round(x, 3) for x in delta]}  -> {'uses the coarser levels' if uses_ctx[r] else 'IGNORES the coarser levels'}")
    # ---- 4. train vs held-out
    print("\n4. training CE vs held-out CE (mean over levels; a large gap is memorisation)")
    tr = [h for h in HIST if "ce" in h][-5:]
    gap = {}
    for m, s in summary.items():
        t_ce = float(np.mean([np.mean([x for k, x in h["ce"].items() if k.startswith(m + "_")]) for h in tr])) if tr else float("nan")
        gap[m] = s["ce"] - t_ce
        print(f"  {m:6s} train {t_ce:.3f} | held-out {s['ce']:.3f} | gap {gap[m]:+.3f}" + ("  !! overfitting" if gap[m] > 0.5 else ""))
    # ---- verdict per modality
    print()
    for m, s in summary.items():
        rts = [r for r in ROW_TYPES if ROW_MOD[r] == m]
        ctx_ok = any(uses_ctx.get(r, False) for r in rts)
        pre_ok = all(uses_prefix.get(r, True) for r in rts)
        learned = s["gain_finest"] > 0.02 and s["gain_w"] > 0.02
        reasons = []
        if not learned:
            reasons.append(f"finest-level gain {s['gain_finest']:+.3f} nats, slot-weighted gain {s['gain_w']:+.3f}, {s['n_ok']}/{s['levels']} levels beat the baseline")
            pr = [PROBE[(m, k)]["gain_bits"] for k in range(1, len(SPEC.lens(m))) if (m, k) in PROBE] if "PROBE" in globals() else []
            if pr and max(pr) < 0.05:
                reasons.append("the codec probe also finds ~no structure: suspect the codec")
            elif pr:
                reasons.append("the codec probe finds structure the model has not yet used: suspect training length / data mix")
        if not ctx_ok:
            reasons.append("ignores the coarser levels")
        if not pre_ok:
            reasons.append("ignores the prefix")
        if gap.get(m, 0) > 0.5:
            reasons.append("overfitting")
        LEARNED[m] = "LEARNED" if learned and ctx_ok and pre_ok and gap.get(m, 0) <= 0.5 else ("PARTIAL" if learned or s["n_ok"] else "NOT LEARNING")
        print(f"{m:6s} {LEARNED[m]:13s} {'; '.join(reasons) if reasons else 'beats the baseline at the finest level, uses the prefix and the coarser levels'}")
    n_ok = sum(x == "LEARNED" for x in LEARNED.values())
    print(f"\nOVERALL: {n_ok}/{len(LEARNED)} modalities LEARNED  {LEARNED}")
    net.train()


if not META_ONLY:
    learned_report()

## 15. Generate: K forwards per sample, coarse to fine

Level 0 is sampled first (from NULL input, given only the prefix), then each next level is sampled in
one forward from the previous level's sampled codes. The prefix makes the same weights do
**text -> text** (prompt), **text -> image** (caption) and **text -> speech** (transcript = basic TTS).
The vocoder is Griffin-Lim on the decoded log-mel (positive NNLS spectrum inversion): intelligible at best, robotic by construction.
A short run will produce babble; what these cells prove is that the whole conditional path works.

**Guidance (`cfg`).** With caption dropout in training the model also knows the empty caption. At sampling time each level's
log-probabilities are `uncond + cfg * (cond - uncond)`: `cfg=1` is plain sampling, larger values push the sample toward the caption.
It costs two forwards per level. Decoding a level needs the codes of ALL levels up to it (the learned tokenizers are residual).

In [ ]:
@torch.no_grad()
def generate(rtype, prompt=None, n=1, temperature=0.9, top_k=64, seed=0, cfg=1.0, diag=None):
    """`diag`: optional list; each level appends {k, distinct, p} so a caller can tell a collapsed
    hierarchy (1 code, p~1.0) from a healthy one (many codes, middling p) without its own loop."""
    net.eval()
    lay = LAYOUTS[rtype]
    g = torch.Generator(device=dev).manual_seed(seed)
    cond, blank = None, None
    if lay.cond:
        arr, _ = _bytes(prompt or "", SPEC.cond)
        cond = torch.from_numpy(arr)[None].repeat(n, 1).to(dev).long() + 1
        blank = torch.ones_like(cond)                                 # empty caption = all padding, exactly what caption dropout trains
    guided = cfg != 1.0 and cond is not None
    codes = [torch.zeros(n, l, dtype=torch.long, device=dev) for l in lay.lens]
    for k in range(lay.K):
        with _amp():
            h = net.hidden(rtype, cond, codes)
            hu = net.hidden(rtype, blank, codes) if guided else None
        lg = net.level_logits(rtype, h, k)
        if guided:
            lc, lu = lg.log_softmax(-1), net.level_logits(rtype, hu, k).log_softmax(-1)
            lg = lu + cfg * (lc - lu)
        lg = lg / temperature
        lg[..., 0] = float("-inf")                                    # NULL is never a sample
        if top_k:
            kth = lg.topk(top_k, dim=-1).values[..., -1:]
            lg = lg.masked_fill(lg < kth, float("-inf"))
        codes[k] = torch.multinomial(lg.softmax(-1).reshape(-1, SPEC.V), 1, generator=g).view(n, -1)
        if diag is not None:
            _p = lg.softmax(-1)
            diag.append(dict(k=k, distinct=int(codes[k].unique().numel()),
                             p=float(_p.max(-1).values.mean())))
    net.train()
    return codes


try:
    from IPython.display import display, Audio
except Exception:
    display = print; Audio = None


def show_text(prompt, **kw):
    kw.pop("cfg", None)                                               # guidance is for image / speech
    out = TextCodec.decode(generate("text", prompt, **kw)[-1])[0]
    print(f"[text->text]  {prompt!r}\n  -> {out!r}")


def show_image(caption, n=4, **kw):
    from PIL import Image as PILImage
    if not getattr(codecs.image, "decodable", True):
        print(f"[text->image] {caption!r}: skipped, the {type(codecs.image).__name__} codes have no pixel decoder "
              "(judge image learning from the held-out CE in the 'did it learn' cells)")
        return
    codes = generate("image", caption, n=n, **kw)
    rows = []
    for k in range(LAYOUTS["image"].K):                               # the coarse-to-fine progression, one row per level
        im = codecs.image.decode(codes, level=k).cpu().numpy()
        rows.append(np.concatenate([x.transpose(1, 2, 0) for x in im], 1))
    display(PILImage.fromarray(np.concatenate(rows, 0)))
    print(f"[text->image] {caption!r}: rows = level 0..{LAYOUTS['image'].K - 1} (blurry -> detailed), {n} samples")


def tts(text, **kw):
    codes = generate("audio", text, n=1, **kw)
    wav = codecs.audio.decode(codes, gl_iters=60)[0].float().cpu().numpy()
    wav = wav / max(1e-6, abs(wav).max()) * 0.9
    print(f"[text->speech] {text!r}: {len(wav)/SPEC.sr:.1f} s, " + ("Griffin-Lim vocoder" if hasattr(codecs.audio, "decode_mel") else "DAC decoder"))
    if Audio is not None:
        display(Audio(wav, rate=SPEC.sr))
    return wav


if not META_ONLY:
    gen_kw = dict(temperature=0.9, top_k=64)
    gen_g = dict(gen_kw, cfg=CFG_SCALE if CAP_DROP > 0 else 1.0)       # guided sampling for image / speech

    print("TEXT: one prompt at three temperatures. `Lk(n_codes,p)` = distinct codes sampled at that level")
    print("      and the model's mean top-1 probability there. A collapsed hierarchy shows 1 code at p~1.00;")
    print("      a healthy one samples many codes at a middling p. Low temperature exposes the dominant mode.")
    for _T in (0.2, 0.7, 1.0):
        _d = []
        _s = TextCodec.decode(generate("text", "what is the capital of france?",
                                       temperature=_T, top_k=64, diag=_d)[-1])[0]
        print("  T=%-4s %s" % (_T, " ".join("L%d(%dc,p%.2f)" % (x["k"], x["distinct"], x["p"]) for x in _d)))
        print("         %r" % _s[:130])

    print("\nTEXT: three prompts at T=0.7. Near-identical output means the prompt is doing nothing.")
    for _p in ("what is the capital of france?", "1 + 1 =", "the cat sat on the"):
        print("  %-34r -> %r" % (_p, TextCodec.decode(
            generate("text", _p, temperature=0.7, top_k=64)[-1])[0][:110]))

    print("\nIMAGE:")
    show_image("a photo of a dog on the grass", **gen_g)
    if not getattr(codecs.image, "decodable", True):
        print("       To actually LOOK at a generated image you need a pixel decoder: NCM_IMAGE_CODEC=vq")
        print("       (the learned multi-scale residual VQ-VAE, ~21.7 dB). Its codes differ from DINO's, so")
        print("       that needs a fresh codec fit AND a fresh training run - a DINO-trained checkpoint")
        print("       cannot be rendered by any decoder, and this is not a bug in the generation path.")

    print("\nSPEECH:")
    for _t in ("the quick brown fox jumps over the lazy dog", "hello world", "one two three four five"):
        _ = tts(_t, **gen_g)

## 15b. Does it generate anything? (sanity checks, not a benchmark)

**Text.** A character-bigram model fitted on real text scores four byte sets in bits per byte (lower = more like real text): real held-out
text, the same bytes shuffled, bytes sampled from the training unigram, and model samples. Model samples below the shuffled / unigram
controls have learned local structure; samples near the real text have learned a lot. A few samples are printed.

**Image.** The DINO codec has no pixel decoder, so generated images cannot be shown. Instead each generated code map (level 2, a 4x4 layout)
is turned back into DINO prototype features and compared with a gallery of real images: the mean best cosine similarity for (a) real
images against the other real images, (b) model samples, (c) random codes drawn from the training unigram. Model samples near (a) and
well above (c) are plausible image layouts; near (c) they are noise. The nearest real images of the first samples are displayed.

Both use unconditional / generic prompts: caption and prompt following are checked separately in section 14b.

In [ ]:
import math
from PIL import Image as PILImage


def _bigram(rows):
    a, b = rows[:, :-1].reshape(-1).long(), rows[:, 1:].reshape(-1).long()
    C = torch.bincount(a * 256 + b, minlength=65536).float().view(256, 256) + 0.1
    return (C / C.sum(1, keepdim=True)).log2()


def _bits(M, x):
    return float(-M[x[:, :-1].long(), x[:, 1:].long()].mean())


if not META_ONLY:
    # ---- text
    _rows = collect_rows(("text0",), {"text0": 2000}, 1, PARTS, WORKERS)
    M = _bigram(torch.from_numpy(np.stack([r["tgt"] for r in _rows])).to(dev))
    del _rows
    real = (torch.cat([mb["codes"][-1] for mb in VAL if mb["rtype"] == "text0"]) - 1).clamp(min=0)
    u = UNI[("text", 4)].float()
    uni = (torch.multinomial(u / u.sum(), real.numel(), replacement=True) - 1).clamp(min=0).view(real.shape)
    shuf = torch.gather(real, 1, torch.rand(real.shape, device=dev).argsort(1))
    gen = (generate("text0", None, n=64, temperature=0.9, top_k=64)[-1] - 1).clamp(min=0)
    print("TEXT: bigram bits per byte (lower = more like real text) and printable-ASCII share")
    for name, x in (("real held-out", real), ("shuffled real", shuf), ("unigram samples", uni), ("model samples", gen)):
        print(f"  {name:16s} {_bits(M, x):.2f} bits | printable {((x >= 32) & (x < 127)).float().mean():.2f}")
    for s in TextCodec.decode(gen[:3] + 1):
        print("  sample:", repr(s[:160]))

    # ---- does the prompt change the output? A prompt the model follows must move the bytes.
    # Two unrelated prompts yielding the same text is the cheapest possible proof that the
    # conditioning is inert - stronger than a CE gap, because it is what a user would see.
    try:
        _P = ["what is the capital of france?", "1 + 1 =",
              "the quick brown fox jumps over the lazy dog"]
        _outs = [TextCodec.decode((generate("text", p, n=1, temperature=0.9, top_k=64)[-1] - 1)
                                  .clamp(min=0) + 1)[0][:200] for p in _P]

        def _grams(s, k=4):
            return {s[i:i + k] for i in range(max(1, len(s) - k + 1))}

        _sim = [len(_grams(_outs[i]) & _grams(_outs[j])) /
                max(1, len(_grams(_outs[i]) | _grams(_outs[j])))
                for i in range(len(_outs)) for j in range(i + 1, len(_outs))]
        print(f"TEXT: prompt sensitivity - 4-gram Jaccard between the outputs of {len(_P)} different "
              f"prompts: mean {float(np.mean(_sim)):.2f}  (0.0 = the prompt changes everything, "
              f"~1.0 = the prompt does nothing)")
        if not any(mb["rtype"] == "text" for mb in VAL):
            print("  !! this run had NO prompt->response rows (KEEP_INSTRUCT=0 and a prompt-free "
                  "text source), so the prompt never had a gradient: a high number here is the "
                  "configuration talking, not a model defect. Set NCM_KEEP_INSTRUCT=1 to test it.")
        for p, o in zip(_P, _outs):
            print(f"  {p!r:44s} -> {o[:90]!r}")
    except Exception as _e:                                   # noqa: BLE001
        print(f"  prompt sensitivity skipped: {type(_e).__name__}: {_e}")

    # ---- image (feature space: the DINO codec has no pixel decoder)
    if not getattr(codecs.image, "decodable", True):
        rows = collect_rows(("image",), {"image": 600}, 1, PARTS, WORKERS)
        gal = _img_tensor(rows)
        del rows

        def _norm(f):
            return F.normalize(F.normalize(f, dim=-1).flatten(1), dim=-1)

        with torch.no_grad():
            gf = torch.cat([_norm(codecs.image.features(gal[i:i + 32])[2]) for i in range(0, len(gal), 32)])
            q = _norm(codecs.image.reconstruct(codecs.image.encode(gal[:100])[2], 2))
            s = q @ gf.T
            s[torch.arange(100), torch.arange(100)] = -1
            real_top1 = float(s.max(1).values.mean())
            gcodes = generate("image", "a photo", n=32, temperature=0.9, top_k=64)[2]
            n2 = codecs.image.protos[2].shape[0]                            # level 2 has only n2 distinct codes, but the model can sample any of 256
            bad = float((gcodes > n2).float().mean())
            gcodes = gcodes.clamp(1, n2)
            qg = _norm(codecs.image.reconstruct(gcodes, 2))
            gen_top1 = float((qg @ gf.T).max(1).values.mean())
            ui = UNI[("image", 2)].float()
            rc = torch.multinomial(ui / ui.sum(), 32 * 16, replacement=True).view(32, 16)
            rand_top1 = float((_norm(codecs.image.reconstruct(rc, 2)) @ gf.T).max(1).values.mean())
        print(f"IMAGE: mean best cosine to a real image (level-2 DINO layout): real vs other real {real_top1:.3f} | model samples {gen_top1:.3f} | random codes {rand_top1:.3f}")
        print(f"  {100 * bad:.1f}% of the sampled level-2 codes were codes that never occur at that level ({n2} are valid); they were clamped")
        top = (qg[:6] @ gf.T).topk(4, dim=1).indices
        canvas = PILImage.new("RGB", (96 * 4, 96 * 6))
        for i in range(6):
            for j in range(4):
                canvas.paste(PILImage.fromarray(gal[top[i, j]].permute(1, 2, 0).cpu().numpy()).resize((96, 96)), (j * 96, i * 96))
        display(canvas)
        print("rows = the 4 nearest real images of the first 6 model samples")

## 16. What was stored

The HF cache lives in RAM and the only files this run may have written are checkpoints (`CKPT_DIR`) and the encoded
code shards (`SHARD_DIR`, ~1 byte per code, never raw images or audio). This cell lists any other new file larger than
100 KB on disk, checks that the in-RAM Hub cache holds only metadata (< 50 MB), and reports the shard directory.

In [ ]:
def _new_files():
    out = []
    roots = [os.getcwd()]                      # disk only: the RAM cache (/dev/shm) is checked by size below
    for root in roots:
        for dp, dn, fn in os.walk(root):
            dn[:] = [d for d in dn if d not in (".git", "__pycache__", ".ipynb_checkpoints", ".venv")]
            if any(d_ and os.path.abspath(dp).startswith(os.path.abspath(d_)) for d_ in (CKPT_DIR, SHARD_DIR)):
                continue
            for f in fn:
                p_ = os.path.join(dp, f)
                try:
                    if os.path.getmtime(p_) >= T_START and os.path.getsize(p_) > 100_000 and not f.endswith((".ipynb", ".py", ".md")):
                        out.append((os.path.getsize(p_), p_))
                except OSError:
                    pass
    return sorted(out, reverse=True)

_nf = _new_files()
ram_mb = sum(os.path.getsize(os.path.join(dp, f)) for dp, _, fs in os.walk(_RAM_HF) for f in fs) / 2 ** 20 if _RAM_HF else 0
print(f"HF cache in RAM (Hub metadata only): {ram_mb:.1f} MB | new files > 100 KB on disk: {len(_nf)}")
for sz, p_ in _nf[:10]:
    print(f"  {sz/2**20:8.2f} MB  {p_}")
assert not _nf, "this run wrote files other than checkpoints and code shards"
assert ram_mb < 50, f"HF cache in RAM is {ram_mb:.0f} MB: more than metadata was cached"
if USE_SHARDS:
    _nfiles, _nbytes, _rows = shard_stats(SHARD_DIR)
    print(f"shards in {SHARD_DIR}: {_nfiles} files, {_nbytes / 2**30:.2f} GiB, rows {_rows}")
print("OK: no raw data and no dataset cache stored" + (" (only encoded code shards)" if USE_SHARDS else ""))